## Cell 1. Setup + Materials Project API + Zintl-like rules

In [1]:
from pathlib import Path
import os
import sys
import time
import typing
from getpass import getpass

import numpy as np
import pandas as pd
from IPython.display import display
from pymatgen.core import Composition

try:
    from typing_extensions import NotRequired, Required
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "typing_extensions"])
    from typing_extensions import NotRequired, Required

typing.NotRequired = NotRequired
typing.Required = Required

for name in list(sys.modules.keys()):
    if name.startswith("mp_api") or name.startswith("emmet"):
        del sys.modules[name]

from mp_api.client import MPRester

print("MPRester imported successfully.")

if not os.environ.get("MP_API_KEY"):
    os.environ["MP_API_KEY"] = getpass("Paste Materials Project API key: ")

MP_API_KEY = os.environ["MP_API_KEY"]
print("MP_API_KEY set:", MP_API_KEY[:6] + "...")

PROJECT_ROOT = Path(r"D:\TB3")
PROCESSED_DIR = PROJECT_ROOT / "processed"
ROOT = PROCESSED_DIR / "paired_training"

OUT_DIR = ROOT / "paper_outputs" / "external_pools"
OUT_DIR.mkdir(parents=True, exist_ok=True)

POOL_CSV = OUT_DIR / "pool_zintl_like_no_element_count_filter.csv"
FAILED_CSV = OUT_DIR / "pool_zintl_like_no_element_count_filter_failed_pairs.csv"

print("Output folder:", OUT_DIR)

DONORS = {
    "Li", "Na", "K", "Rb", "Cs",
    "Mg", "Ca", "Sr", "Ba",
    "Eu", "Yb",
}

FRAMEWORK = {
    "B", "Al", "Ga", "In", "Tl",
    "Si", "Ge", "Sn", "Pb",
    "P", "As", "Sb", "Bi",
}

EXCLUDE = {
    "H", "C", "N", "O",
    "F", "Cl", "Br", "I",
    "He", "Ne", "Ar", "Kr", "Xe", "Rn",
}

def get_symbol(el):
    return el.symbol if hasattr(el, "symbol") else str(el)


def safe_float(x, default=np.nan):
    try:
        x = float(x)
        return x if np.isfinite(x) else default
    except Exception:
        return default


def is_zintl_klemm_like_formula(formula):
    """
    Formula-level Zintl-like screening rule.
    No 2-5 element-count condition.
    """
    try:
        comp = Composition(formula).reduced_composition
        symbols = {str(el) for el in comp.elements}

        if symbols & EXCLUDE:
            return False

        if not (symbols & DONORS):
            return False

        if not (symbols & FRAMEWORK):
            return False

        guesses = comp.oxi_state_guesses()
        if len(guesses) == 0:
            return False

        for oxi in guesses:
            has_positive_donor = False
            has_negative_framework = False

            for el, val in oxi.items():
                sym = get_symbol(el)

                if sym in DONORS and val > 0:
                    has_positive_donor = True

                if sym in FRAMEWORK and val < 0:
                    has_negative_framework = True

            if has_positive_donor and has_negative_framework:
                return True

        return False

    except Exception:
        return False


print("Donor elements:", sorted(DONORS))
print("Framework elements:", sorted(FRAMEWORK))
print("Excluded elements:", sorted(EXCLUDE))

MPRester imported successfully.
MP_API_KEY set: TeFXWF...
Output folder: D:\TB3\processed\paired_training\paper_outputs\external_pools
Donor elements: ['Ba', 'Ca', 'Cs', 'Eu', 'K', 'Li', 'Mg', 'Na', 'Rb', 'Sr', 'Yb']
Framework elements: ['Al', 'As', 'B', 'Bi', 'Ga', 'Ge', 'In', 'P', 'Pb', 'Sb', 'Si', 'Sn', 'Tl']
Excluded elements: ['Ar', 'Br', 'C', 'Cl', 'F', 'H', 'He', 'I', 'Kr', 'N', 'Ne', 'O', 'Rn', 'Xe']


# Cell 2. Query Materials Project and build Zintl-like candidate pool
Database 1 = Zintl-like Materials Project pool

Failed donor-framework pairs: 0

Duplicated material_id: 0

Duplicated formula_pretty: 178

DFT-ready theo điều kiện band_gap + ehull: 1100 vật liệu

Stable trong DFT-ready: 830 vật liệu

In [2]:
# Cell 2. Build Zintl-like pool without 2-5 element-count condition
# Purpose:
#   Query Materials Project using donor-framework element pairs.
#   Then apply the Zintl-Klemm-like chemical rule defined in Cell 1.
#   This step only builds the candidate metadata pool, not structures.

FIELDS = [
    "material_id",
    "formula_pretty",
    "band_gap",
    "energy_above_hull",
    "is_stable",
]

rows = []
seen_ids = set()
failed_pairs = []

with MPRester(MP_API_KEY) as mpr:
    total_pairs = len(DONORS) * len(FRAMEWORK)
    pair_i = 0

    for donor in sorted(DONORS):
        for fw in sorted(FRAMEWORK):
            pair_i += 1
            print(f"[{pair_i}/{total_pairs}] Query {donor}-{fw} ...", end=" ")

            try:
                try:
                    docs = mpr.materials.summary.search(
                        elements=[donor, fw],
                        exclude_elements=sorted(EXCLUDE),
                        band_gap=(0.001, None),
                        fields=FIELDS,
                        all_fields=False,
                        chunk_size=200,
                        include_gnome=False,
                    )
                except TypeError:
                    docs = mpr.materials.summary.search(
                        elements=[donor, fw],
                        exclude_elements=sorted(EXCLUDE),
                        band_gap=(0.001, None),
                        fields=FIELDS,
                        all_fields=False,
                        chunk_size=200,
                    )

                print("docs:", len(docs))

                for doc in docs:
                    d = doc.model_dump() if hasattr(doc, "model_dump") else dict(doc)

                    mid = str(d.get("material_id")).strip()
                    if mid in seen_ids:
                        continue

                    formula = d.get("formula_pretty")
                    if formula is None:
                        continue

                    if not is_zintl_klemm_like_formula(formula):
                        continue

                    seen_ids.add(mid)

                    rows.append({
                        "material_id": mid,
                        "formula_pretty": formula,
                        "band_gap": safe_float(d.get("band_gap")),
                        "energy_above_hull": safe_float(d.get("energy_above_hull")),
                        "is_stable": d.get("is_stable", None),
                        "zintl_like_rule": True,
                        "rule_basis": "donor-framework + Zintl-Klemm oxidation-state proxy",
                    })

                time.sleep(0.15)

            except Exception as exc:
                print("FAILED")
                failed_pairs.append({
                    "donor": donor,
                    "framework": fw,
                    "error": str(exc),
                })
                time.sleep(1.0)

zintl_pool_df = pd.DataFrame(rows)

if len(zintl_pool_df) > 0:
    zintl_pool_df = (
        zintl_pool_df
        .drop_duplicates("material_id")
        .sort_values(["energy_above_hull", "band_gap"], ascending=[True, True])
        .reset_index(drop=True)
    )

failed_df = pd.DataFrame(failed_pairs)

zintl_pool_df.to_csv(POOL_CSV, index=False, encoding="utf-8-sig")
failed_df.to_csv(FAILED_CSV, index=False, encoding="utf-8-sig")

print("\nDONE")
print("Zintl-like pool without 2-5 element-count filter:", len(zintl_pool_df))
print("Failed donor-framework pairs:", len(failed_pairs))
print("Saved pool:", POOL_CSV)
print("Saved failed pairs:", FAILED_CSV)

display(zintl_pool_df.head(30))

if len(failed_df) > 0:
    display(failed_df.head(20))

[1/143] Query Ba-Al ... 

Retrieving SummaryDoc documents:   0%|          | 0/15 [00:00<?, ?it/s]

docs: 15
[2/143] Query Ba-As ... 

Retrieving SummaryDoc documents:   0%|          | 0/34 [00:00<?, ?it/s]

docs: 34
[3/143] Query Ba-B ... 

...

Retrieving SummaryDoc documents: 0it [00:00, ?it/s]

docs: 0
[143/143] Query Yb-Tl ... 

Retrieving SummaryDoc documents: 0it [00:00, ?it/s]

docs: 0

DONE
Zintl-like pool without 2-5 element-count filter: 1361
Failed donor-framework pairs: 0
Saved pool: D:\TB3\processed\paired_training\paper_outputs\external_pools\pool_zintl_like_no_element_count_filter.csv
Saved failed pairs: D:\TB3\processed\paired_training\paper_outputs\external_pools\pool_zintl_like_no_element_count_filter_failed_pairs.csv


,material_id,formula_pretty,band_gap,energy_above_hull,is_stable,zintl_like_rule,rule_basis
0,mp-soz,SrCuAs,0.003800,0.0,True,True,donor-framework + Zintl-Klemm oxidation-state ...
1,mp-nwt,Sr(SnAs)2,0.004700,0.0,True,True,donor-framework + Zintl-Klemm oxidation-state ...
2,mp-nmw,CaP3,0.004900,0.0,True,True,donor-framework + Zintl-Klemm oxidation-state ...
3,mp-cprao,Eu3(AlAs2)2,0.005900,0.0,True,True,donor-framework + Zintl-Klemm oxidation-state ...
4,mp-sef,CaAgP,0.007000,0.0,True,True,donor-framework + Zintl-Klemm oxidation-state ...
5,mp-cqhyc,Eu3(GeAs2)2,0.012200,0.0,True,True,donor-framework + Zintl-Klemm oxidation-state ...
6,mp-cqpvg,Sr(InAs)2,0.012200,0.0,True,True,donor-framework + Zintl-Klemm oxidation-state ...
7,mp-crxoo,K38Nb7As24,0.013100,0.0,True,True,donor-framework + Zintl-Klemm oxidation-state ...
8,mp-crsuc,Eu3DySb3,0.013100,0.0,True,True,donor-framework + Zintl-Klemm oxidation-state ...
9,mp-bghya,Sr14MnSb11,0.015600,0.0,True,True,donor-framework + Zintl-Klemm oxidation-state ...


In [3]:
# Cell 3. Filter DFT-ready Zintl-like pool
# Conditions:
#   0.30 <= band_gap <= 4.50 eV
#   energy_above_hull <= 0.10 eV/atom

DFT_CSV = OUT_DIR / "pool_zintl_like_no_element_count_filter_DFT_ready.csv"

df = pd.read_csv(POOL_CSV)

df["band_gap"] = pd.to_numeric(df["band_gap"], errors="coerce")
df["energy_above_hull"] = pd.to_numeric(df["energy_above_hull"], errors="coerce")

df["is_stable_bool"] = (
    df["is_stable"]
    .astype(str)
    .str.lower()
    .isin(["true", "1", "yes"])
)

dft_pool = df[
    (df["band_gap"] >= 0.30) &
    (df["band_gap"] <= 4.50) &
    (df["energy_above_hull"] <= 0.10)
].copy()

dft_pool = dft_pool.sort_values(
    ["is_stable_bool", "energy_above_hull", "band_gap"],
    ascending=[False, True, True],
).reset_index(drop=True)

dft_pool.to_csv(DFT_CSV, index=False, encoding="utf-8-sig")

print("Original Zintl-like pool:", len(df))
print("DFT-ready Zintl-like pool:", len(dft_pool))
print("Stable candidates in DFT-ready pool:", int(dft_pool["is_stable_bool"].sum()))
print("Low-Ehull candidates:", int((dft_pool["energy_above_hull"] <= 0.10).sum()))
print("Saved:", DFT_CSV)

display(dft_pool.head(30))

Original Zintl-like pool: 1361
DFT-ready Zintl-like pool: 1100
Stable candidates in DFT-ready pool: 830
Low-Ehull candidates: 1100
Saved: D:\TB3\processed\paired_training\paper_outputs\external_pools\pool_zintl_like_no_element_count_filter_DFT_ready.csv


,material_id,formula_pretty,band_gap,energy_above_hull,is_stable,zintl_like_rule,rule_basis,is_stable_bool
0,mp-cqwjb,Na3Sr3GaSb4,0.306600,0.0,True,True,donor-framework + Zintl-Klemm oxidation-state ...,True
1,mp-bpvf,Na7Al2Sb5,0.309368,0.0,True,True,donor-framework + Zintl-Klemm oxidation-state ...,True
2,mp-bxdtm,K2SrCdSb2,0.315200,0.0,True,True,donor-framework + Zintl-Klemm oxidation-state ...,True
3,mp-bjidg,NaSr2AlP3,0.316300,0.0,True,True,donor-framework + Zintl-Klemm oxidation-state ...,True
4,mp-bgkcx,Ca2CdSb2,0.319000,0.0,True,True,donor-framework + Zintl-Klemm oxidation-state ...,True
5,mp-bbor,K8SnSb4,0.319000,0.0,True,True,donor-framework + Zintl-Klemm oxidation-state ...,True
6,mp-cqfpp,Cs9(NbAs3)2,0.319500,0.0,True,True,donor-framework + Zintl-Klemm oxidation-state ...,True
7,mp-crroq,K5U(PS4)3,0.319800,0.0,True,True,donor-framework + Zintl-Klemm oxidation-state ...,True
8,mp-cruqh,Cs3Sb5Se9,0.322300,0.0,True,True,donor-framework + Zintl-Klemm oxidation-state ...,True
9,mp-ltn,LiAs,0.328996,0.0,True,True,donor-framework + Zintl-Klemm oxidation-state ...,True


In [4]:
# Cell 4. Summary table for teacher/seminar

SUMMARY_CSV = OUT_DIR / "zintl_like_no_element_filter_summary.csv"

pool_df = pd.read_csv(POOL_CSV)
dft_df = pd.read_csv(DFT_CSV)

for tmp_df in [pool_df, dft_df]:
    tmp_df["band_gap"] = pd.to_numeric(tmp_df["band_gap"], errors="coerce")
    tmp_df["energy_above_hull"] = pd.to_numeric(tmp_df["energy_above_hull"], errors="coerce")
    tmp_df["is_stable_bool"] = (
        tmp_df["is_stable"]
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes"])
    )

summary = pd.DataFrame([
    {
        "stage": "Zintl-like pool without 2-5 element filter",
        "n_materials": len(pool_df),
        "n_stable": int(pool_df["is_stable_bool"].sum()),
        "n_ehull_le_0.10": int((pool_df["energy_above_hull"] <= 0.10).sum()),
        "n_bg_0.30_4.50": int(((pool_df["band_gap"] >= 0.30) & (pool_df["band_gap"] <= 4.50)).sum()),
        "condition": "donor-framework + Zintl-Klemm oxidation-state proxy; no element-count filter",
        "output_file": str(POOL_CSV),
    },
    {
        "stage": "DFT-ready Zintl-like pool",
        "n_materials": len(dft_df),
        "n_stable": int(dft_df["is_stable_bool"].sum()),
        "n_ehull_le_0.10": int((dft_df["energy_above_hull"] <= 0.10).sum()),
        "n_bg_0.30_4.50": int(((dft_df["band_gap"] >= 0.30) & (dft_df["band_gap"] <= 4.50)).sum()),
        "condition": "0.30 <= band_gap <= 4.50 eV; energy_above_hull <= 0.10 eV/atom",
        "output_file": str(DFT_CSV),
    },
])

summary.to_csv(SUMMARY_CSV, index=False, encoding="utf-8-sig")

display(summary)

print("Saved summary:", SUMMARY_CSV)
print("Saved pool:", POOL_CSV)
print("Saved DFT-ready pool:", DFT_CSV)
print("Saved failed-pair log:", FAILED_CSV)

,stage,n_materials,n_stable,n_ehull_le_0.10,n_bg_0.30_4.50,condition,output_file
0,Zintl-like pool without 2-5 element filter,1361,939,1258,1165,donor-framework + Zintl-Klemm oxidation-state ...,D:\TB3\processed\paired_training\paper_outputs...
1,DFT-ready Zintl-like pool,1100,830,1100,1100,0.30 <= band_gap <= 4.50 eV; energy_above_hull...,D:\TB3\processed\paired_training\paper_outputs...


Saved summary: D:\TB3\processed\paired_training\paper_outputs\external_pools\zintl_like_no_element_filter_summary.csv
Saved pool: D:\TB3\processed\paired_training\paper_outputs\external_pools\pool_zintl_like_no_element_count_filter.csv
Saved DFT-ready pool: D:\TB3\processed\paired_training\paper_outputs\external_pools\pool_zintl_like_no_element_count_filter_DFT_ready.csv
Saved failed-pair log: D:\TB3\processed\paired_training\paper_outputs\external_pools\pool_zintl_like_no_element_count_filter_failed_pairs.csv


In [5]:
# Cell 5. Import utils_tb and prepare final Zintl SLME screening

from pathlib import Path
import os
import sys
import getpass
import pandas as pd

PROJECT_ROOT = Path(r"D:\TB3")
ROOT = PROJECT_ROOT / "processed" / "paired_training"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import utils_tb as TB3

POOL_CSV = (
    ROOT
    / "paper_outputs"
    / "external_pools"
    / "pool_zintl_like_no_element_count_filter_DFT_ready.csv"
)

SCREEN_DIR = (
    ROOT
    / "paper_outputs"
    / "screening_SLME_direct_alpha_500nm"
)

SCREEN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MP_API_KEY = os.environ.get(
    "MP_API_KEY",
    "",
).strip()

if not MP_API_KEY:
    MP_API_KEY = getpass.getpass(
        "Enter Materials Project API key: "
    ).strip()

    os.environ["MP_API_KEY"] = MP_API_KEY

pool_df = pd.read_csv(
    POOL_CSV
)

print("Imported package:", TB3)
print("Pool CSV:", POOL_CSV)
print("Pool shape:", pool_df.shape)

print("Screening target:")
print("  Direct-alpha -> SLME")
print("  L = 500 nm")
print("  T = 300 K")
print("  fr = 1")
print("  Primary target: eta_SLME > 30%")
print("Output folder:", SCREEN_DIR)

Imported package: <module 'utils_tb' from 'D:\\TB3\\utils_tb\\__init__.py'>
Pool CSV: D:\TB3\processed\paired_training\paper_outputs\external_pools\pool_zintl_like_no_element_count_filter_DFT_ready.csv
Pool shape: (1100, 8)
Screening target:
  Direct-alpha -> SLME
  L = 500 nm
  T = 300 K
  fr = 1
  Primary target: eta_SLME > 30%
Output folder: D:\TB3\processed\paired_training\paper_outputs\screening_SLME_direct_alpha_500nm


In [6]:
# Cell 8A. Find original database metadata file for overlap checking

from pathlib import Path
import pandas as pd

SEARCH_ROOTS = [
    Path(r"D:\TB3"),
    Path(r"D:\TB3\processed"),
    Path(r"D:\TB3\processed\paired_training"),
]

patterns = [
    "*dataset_audit*.csv",
    "*fixed_split*.csv",
    "*paired*.csv",
    "*summary*.csv",
]

candidates = []

for root in SEARCH_ROOTS:
    if not root.exists():
        continue

    for pattern in patterns:
        for p in root.rglob(pattern):
            try:
                head = pd.read_csv(p, nrows=5)
                cols = set(head.columns)

                has_id = any(c in cols for c in ["mat_id", "material_id", "mp_id"])
                has_formula = any(c in cols for c in ["formula", "formula_pretty", "pretty_formula"])
                has_split = "split" in cols

                if has_id or has_formula:
                    candidates.append({
                        "path": str(p),
                        "n_preview_rows": len(head),
                        "has_id": has_id,
                        "has_formula": has_formula,
                        "has_split": has_split,
                        "columns": ", ".join(list(head.columns)[:12]),
                    })

            except Exception:
                pass

candidates_df = pd.DataFrame(candidates).drop_duplicates("path")

print("Candidate metadata files found:", len(candidates_df))

if len(candidates_df) > 0:
    display(candidates_df)
else:
    print("No metadata CSV found. We may need to rebuild the original database audit from the JSON zip.")

Candidate metadata files found: 15


,path,n_preview_rows,has_id,has_formula,has_split,columns
0,D:\TB3\processed\dataset_audit.csv,5,True,True,False,"file_name, status, reason, mat_id, formula, sp..."
1,D:\TB3\processed\dataset_audit_canonical_v1.csv,5,True,True,False,"file_name, mat_id, formula, spg, nsites, num_e..."
2,D:\TB3\processed\dataset_audit_revised.csv,5,True,True,False,"file_name, status, reason, mat_id, formula, sp..."
3,D:\TB3\processed\dataset_audit_revised_v2.csv,5,True,True,False,"file_name, status, reason, mat_id, formula, sp..."
4,D:\TB3\processed\dataset_audit\dataset_audit_i...,5,True,True,False,"file_name, status, reason, mat_id, formula, sp..."
5,D:\TB3\processed\dataset_audit_v2\dataset_audi...,5,True,True,False,"file_name, status, reason, mat_id, formula, sp..."
6,D:\TB3\processed\dataset_bootstrap_cache\datas...,5,True,True,False,"file_name, status, reason, mat_id, formula, sp..."
7,D:\TB3\processed\optical_bootstrap_cache\datas...,5,True,True,False,"file_name, status, reason, mat_id, formula, sp..."
8,D:\TB3\processed\fixed_splits\fixed_split_pair...,5,True,True,True,"split, view_idx, base_idx, mat_id, formula, fi..."
9,D:\TB3\processed\paired_training\fixed_splits\...,5,True,True,True,"split, view_idx, base_idx, mat_id, formula, fi..."


In [7]:
# Cell 8B. Show full fixed-split metadata path

display(candidates_df[candidates_df["has_split"] == True]["path"].tolist())

['D:\\TB3\\processed\\fixed_splits\\fixed_split_paired_epsR_epsI_seed2025_v1.csv',
 'D:\\TB3\\processed\\paired_training\\fixed_splits\\fixed_split_paired_epsR_epsI_seed2025_v1.csv']

In [8]:
from pathlib import Path
import sys
import importlib

PROJECT_ROOT = Path(r"D:\TB3")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import utils_tb.external_slme_db1 as DB1

DB1 = importlib.reload(DB1)

db1_result = DB1.run_final_db1_slme_screening(
    project_root=PROJECT_ROOT,
    show=True,
)

FINAL EXTERNAL DIRECT-ALPHA -> SLME SCREENING
Pool CSV          : D:\TB3\processed\paired_training\paper_outputs\external_pools\pool_zintl_like_no_element_count_filter_DFT_ready.csv
Structure cache   : D:\TB3\processed\paired_training\paper_outputs\screening_srp_200nm\external_pool_cache\screening_zintl_like_external_fixed_1100_with_structures.pkl
Direct-alpha dir  : D:\TB3\processed\paired_training\epsI_alpha_log10_1p_direct_alpha_ssl_init_enhanced_19_24_d005_wd5e5_3seed
Output directory  : D:\TB3\processed\paired_training\paper_outputs\screening_SLME_direct_alpha_500nm

Protocol:
  model route     : structure -> Direct-alpha GINE
  graph dims      : 19 / 24
  model seeds     : (42, 123, 2025)
  thickness       : 500.0 nm
  temperature     : 300.0 K
  fr              : 1.0
  solar spectrum  : ASTM G173 AM1.5G
  primary target  : SLME > 30%

Source pool         : 1100
Validated structures: 1100
ID alignment        : PASS

DIRECT-ALPHA CHECKPOINTS
seed   42 | epoch  97 | target=epsI_alp

D:\TB3\utils_tb\features.py:932: UserWarning: No data available for atomic_radius_calculated for U
  atomic_radius_calculated, _ = clip_scale(getattr(element, "atomic_radius_calculated", np.nan), 3.5)
D:\TB3\utils_tb\features.py:932: UserWarning: No data available for atomic_radius_calculated for La
  atomic_radius_calculated, _ = clip_scale(getattr(element, "atomic_radius_calculated", np.nan), 3.5)
D:\TB3\utils_tb\features.py:932: UserWarning: No data available for atomic_radius_calculated for Ce
  atomic_radius_calculated, _ = clip_scale(getattr(element, "atomic_radius_calculated", np.nan), 3.5)
D:\TB3\utils_tb\features.py:932: UserWarning: No data available for atomic_radius_calculated for Th
  atomic_radius_calculated, _ = clip_scale(getattr(element, "atomic_radius_calculated", np.nan), 3.5)
D:\TB3\utils_tb\features.py:932: UserWarning: No data available for atomic_radius_calculated for Ho
  atomic_radius_calculated, _ = clip_scale(getattr(element, "atomic_radius_calculated", np.na

Valid graphs for epsI_alpha_log10_1p: 1100 / 1100
Failed structures for epsI_alpha_log10_1p: 0
Graphs built             : 1100
Graph feature dimensions: 19/24
Graph-dimension check   : PASS

DIRECT-ALPHA ENSEMBLE INFERENCE
Device: cuda
Predicting with: D:\TB3\processed\paired_training\epsI_alpha_log10_1p_direct_alpha_ssl_init_enhanced_19_24_d005_wd5e5_3seed\best_optical_epsI_alpha_log10_1p_direct_alpha_ssl_init_enhanced_19_24_d005_wd5e5_3seed_seed42.pt
Predicting with: D:\TB3\processed\paired_training\epsI_alpha_log10_1p_direct_alpha_ssl_init_enhanced_19_24_d005_wd5e5_3seed\best_optical_epsI_alpha_log10_1p_direct_alpha_ssl_init_enhanced_19_24_d005_wd5e5_3seed_seed123.pt
Predicting with: D:\TB3\processed\paired_training\epsI_alpha_log10_1p_direct_alpha_ssl_init_enhanced_19_24_d005_wd5e5_3seed\best_optical_epsI_alpha_log10_1p_direct_alpha_ssl_init_enhanced_19_24_d005_wd5e5_3seed_seed2025.pt

PREDICTED ALPHA
alpha shape : (1100, 2001)
alpha max   : 1.935e+06 cm^-1

AM1.5G SANITY
Psolar = 

,SLME_rank,material_id,formula_pretty,band_gap,energy_above_hull,is_stable,predicted_SLME_percent,predicted_Voc_V,predicted_Jsc_A_m2,SLME_gap_cutoff_eV
0,1,mp-btzq,NaP5,1.354200,0.009960,False,33.753417,1.095923,343.968025,1.354200
1,2,mp-bevqy,Rb2Ag(AsSe2)3,1.349700,0.000000,True,33.727537,1.086707,346.898434,1.349700
2,3,mp-cqyqi,KIn(PSe3)2,1.351200,0.000479,False,33.690766,1.096221,343.227247,1.351200
3,4,mp-bsnx,Ba2B4Se13,1.333000,0.000000,True,33.674518,1.077692,349.529872,1.333000
4,5,mp-cqcrk,KAgPSe3,1.354700,0.000000,True,33.668234,1.096319,342.964157,1.354700
5,6,mp-cqpjs,Rb2As2Pd,1.145847,0.000000,True,33.667991,0.900150,426.255610,1.145847
6,7,mp-crclc,Ca11GaSb9,1.170400,0.000000,True,33.666295,0.927751,412.186532,1.170400
7,8,mp-bkski,CsAs2Se3,1.389100,0.000000,True,33.661544,1.124006,333.665188,1.389100
8,9,mp-ekf,CsSbSe2,1.134800,0.000000,True,33.656419,0.890929,431.013586,1.134800
9,10,mp-ue,Ba3As14,1.176300,0.000000,True,33.652101,0.927864,411.957186,1.176300



Primary discovery pool: predicted SLME > 30%


,SLME_rank,material_id,formula_pretty,band_gap,energy_above_hull,is_stable,predicted_SLME_percent,predicted_Voc_V,predicted_Jsc_A_m2,SLME_gap_cutoff_eV
0,1,mp-btzq,NaP5,1.354200,0.009960,False,33.753417,1.095923,343.968025,1.354200
1,2,mp-bevqy,Rb2Ag(AsSe2)3,1.349700,0.000000,True,33.727537,1.086707,346.898434,1.349700
2,3,mp-cqyqi,KIn(PSe3)2,1.351200,0.000479,False,33.690766,1.096221,343.227247,1.351200
3,4,mp-bsnx,Ba2B4Se13,1.333000,0.000000,True,33.674518,1.077692,349.529872,1.333000
4,5,mp-cqcrk,KAgPSe3,1.354700,0.000000,True,33.668234,1.096319,342.964157,1.354700
5,6,mp-cqpjs,Rb2As2Pd,1.145847,0.000000,True,33.667991,0.900150,426.255610,1.145847
6,7,mp-crclc,Ca11GaSb9,1.170400,0.000000,True,33.666295,0.927751,412.186532,1.170400
7,8,mp-bkski,CsAs2Se3,1.389100,0.000000,True,33.661544,1.124006,333.665188,1.389100
8,9,mp-ekf,CsSbSe2,1.134800,0.000000,True,33.656419,0.890929,431.013586,1.134800
9,10,mp-ue,Ba3As14,1.176300,0.000000,True,33.652101,0.927864,411.957186,1.176300


In [9]:
from pathlib import Path
import sys
import importlib

PROJECT_ROOT = Path(r"D:\TB3")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import utils_tb.gap_dominance_audit as GDA

GDA = importlib.reload(GDA)

gap_audit = GDA.run_mp_gap_dominance_audit(
    project_root=PROJECT_ROOT,
    show=True,
)

MP-GAP DOMINANCE AUDIT
Pearson  ML-SLME vs gap-only SQ : 0.9952
Spearman ML-SLME vs gap-only SQ : 0.9832

Top-k overlap with gap-only SQ ranking:


,k,overlap_count,overlap_percent
0,10,2,20.0
1,20,7,35.0
2,50,23,46.0
3,100,62,62.0


Mean eta_ML / eta_SQ for eta_ML > 30%: 0.9852
Median eta_ML / eta_SQ for eta_ML > 30%: 0.9920
Mean |eta_ML - eta_SQ| for eta_ML > 30%: 0.4850 percentage points

Top-20 FINAL candidates with SQ comparison:


,SLME_rank,material_id,formula_pretty,band_gap,predicted_SLME_percent,SQ_efficiency_from_MP_gap_percent,SLME_minus_SQ_gap_pp,SLME_to_SQ_gap_ratio,energy_above_hull,is_stable
0,1,mp-btzq,NaP5,1.354200,33.753417,33.762712,-0.009295,0.999725,0.009960,False
1,2,mp-bevqy,Rb2Ag(AsSe2)3,1.349700,33.727537,33.765432,-0.037895,0.998878,0.000000,True
2,3,mp-cqyqi,KIn(PSe3)2,1.351200,33.690766,33.762712,-0.071945,0.997869,0.000479,False
3,4,mp-bsnx,Ba2B4Se13,1.333000,33.674518,33.798857,-0.124340,0.996321,0.000000,True
4,5,mp-cqcrk,KAgPSe3,1.354700,33.668234,33.762712,-0.094478,0.997202,0.000000,True
5,6,mp-cqpjs,Rb2As2Pd,1.145847,33.667991,33.694521,-0.026530,0.999213,0.000000,True
6,7,mp-crclc,Ca11GaSb9,1.170400,33.666295,33.666862,-0.000568,0.999983,0.000000,True
7,8,mp-bkski,CsAs2Se3,1.389100,33.661544,33.668276,-0.006733,0.999800,0.000000,True
8,9,mp-ekf,CsSbSe2,1.134800,33.656419,33.689630,-0.033211,0.999014,0.000000,True
9,10,mp-ue,Ba3As14,1.176300,33.652101,33.666862,-0.014761,0.999562,0.000000,True



Saved:
D:\TB3\processed\paired_training\paper_outputs\screening_SLME_direct_alpha_500nm\gap_dominance_audit\zintl_SLME_vs_MP_gap_only_SQ.csv
D:\TB3\processed\paired_training\paper_outputs\screening_SLME_direct_alpha_500nm\gap_dominance_audit\zintl_SLME_vs_MP_gap_only_SQ_topk_overlap.csv
D:\TB3\processed\paired_training\paper_outputs\screening_SLME_direct_alpha_500nm\gap_dominance_audit\zintl_SLME_vs_MP_gap_only_SQ_summary.csv


In [10]:
# CELL 8 FINAL
# Structure-level novelty / leakage audit for Zintl candidates
#
# FINAL solar route:
# Direct-alpha -> SLME -> eta > 30% -> StructureMatcher audit
#
# StructureMatcher protocol:
#   ltol = 0.20
#   stol = 0.30
#   angle_tol = 5 deg
#   primitive_cell = True
#   scale = True
#   attempt_supercell = True
#
# Important:
# - External mp-* IDs are NOT compared with internal agm* IDs.
# - Formula is ONLY used as a prefilter.
# - Final overlap is decided by StructureMatcher.
# - Only eta_SLME > 30% candidates are subjected to the
#   expensive structure-level discovery audit.
# ============================================================

from pathlib import Path
import sys
import json
import importlib
import time

import numpy as np
import pandas as pd

from IPython.display import display
from pymatgen.core import Structure
from pymatgen.analysis.structure_matcher import StructureMatcher


# ============================================================
# 0. Paths
# ============================================================

PROJECT_ROOT = Path(r"D:\TB3")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ROOT = (
    PROJECT_ROOT
    / "processed"
    / "paired_training"
)

SLME_DIR = (
    ROOT
    / "paper_outputs"
    / "screening_SLME_direct_alpha_500nm"
)

SLME_ALL_CSV = (
    SLME_DIR
    / "zintl_external_1100_direct_alpha_SLME_all.csv"
)

EXTERNAL_STRUCTURE_CACHE = (
    ROOT
    / "paper_outputs"
    / "screening_srp_200nm"
    / "external_pool_cache"
    / "screening_zintl_like_external_fixed_1100_with_structures.pkl"
)

ORIGINAL_ZIP = (
    PROJECT_ROOT
    / "database_300.zip"
)

FIXED_SPLIT_CSV = (
    PROJECT_ROOT
    / "processed"
    / "fixed_splits"
    / "fixed_split_paired_epsR_epsI_seed2025_v1.csv"
)

OUT_DIR = (
    ROOT
    / "paper_outputs"
    / "dft_validation_shortlist"
    / "discovery_first_zintl_SLME"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ORIGINAL_STRUCTURE_CACHE = (
    OUT_DIR
    / "original_database_10189_structures.pkl"
)

ORIGINAL_STRUCTURE_FAILURES_CSV = (
    OUT_DIR
    / "original_database_structure_cache_failures.csv"
)


# ============================================================
# 1. Configuration
# ============================================================

SLME_THRESHOLD = 30.0
TOP_K = 50

MATCHER = StructureMatcher(
    ltol=0.20,
    stol=0.30,
    angle_tol=5.0,
    primitive_cell=True,
    scale=True,
    attempt_supercell=True,
)

print("=" * 110)
print("FINAL ZINTL STRUCTURE-NOVELTY AUDIT")
print("=" * 110)

print("Primary target       : predicted SLME > 30%")
print("Novelty reference    : full original database")
print("Leakage reference    : original training split")

print()
print("StructureMatcher:")
print("  ltol              : 0.20")
print("  stol              : 0.30")
print("  angle_tol         : 5 deg")
print("  primitive_cell    : True")
print("  scale             : True")
print("  attempt_supercell : True")


# ============================================================
# 2. Validate required inputs
# ============================================================

for path in [
    SLME_ALL_CSV,
    EXTERNAL_STRUCTURE_CACHE,
    ORIGINAL_ZIP,
    FIXED_SPLIT_CSV,
]:
    if not path.exists():
        raise FileNotFoundError(
            f"Required input not found:\n{path}"
        )


# ============================================================
# 3. Load FINAL Direct-alpha -> SLME ranking
# ============================================================

ranked = pd.read_csv(
    SLME_ALL_CSV
)

required_cols = [
    "material_id",
    "formula_pretty",
    "band_gap",
    "energy_above_hull",
    "is_stable",
    "predicted_SLME_percent",
]

missing = [
    c
    for c in required_cols
    if c not in ranked.columns
]

if missing:
    raise KeyError(
        f"FINAL SLME CSV missing columns: {missing}"
    )


ranked["material_id"] = (
    ranked["material_id"]
    .astype(str)
    .str.strip()
)

ranked["predicted_SLME_percent"] = pd.to_numeric(
    ranked["predicted_SLME_percent"],
    errors="coerce",
)

ranked["band_gap"] = pd.to_numeric(
    ranked["band_gap"],
    errors="coerce",
)

ranked["energy_above_hull"] = pd.to_numeric(
    ranked["energy_above_hull"],
    errors="coerce",
)


ranked = (
    ranked
    .dropna(
        subset=[
            "predicted_SLME_percent",
            "material_id",
        ]
    )
    .sort_values(
        "predicted_SLME_percent",
        ascending=False,
    )
    .drop_duplicates(
        "material_id",
        keep="first",
    )
    .reset_index(
        drop=True
    )
)


if "SLME_rank" not in ranked.columns:
    ranked["SLME_rank"] = np.arange(
        1,
        len(ranked) + 1,
    )


if len(ranked) != 1100:
    raise RuntimeError(
        f"Expected 1100 ranked candidates, found {len(ranked)}."
    )


primary_candidates = (
    ranked[
        ranked["predicted_SLME_percent"]
        > SLME_THRESHOLD
    ]
    .copy()
    .reset_index(drop=True)
)


print()
print("=" * 110)
print("SLME DISCOVERY POOL")
print("=" * 110)

print("All screened candidates :", len(ranked))
print(
    f"SLME > {SLME_THRESHOLD:.0f}% candidates :",
    len(primary_candidates),
)

print(
    "SLME > 31% candidates  :",
    int(
        (
            ranked["predicted_SLME_percent"]
            > 31.0
        ).sum()
    ),
)

print(
    "SLME > 32% candidates  :",
    int(
        (
            ranked["predicted_SLME_percent"]
            > 32.0
        ).sum()
    ),
)


# ============================================================
# 4. Structure conversion helper
# ============================================================

def to_pmg_structure(obj):

    if isinstance(obj, Structure):
        return obj

    if isinstance(obj, dict):
        return Structure.from_dict(obj)

    if isinstance(obj, str):
        return Structure.from_dict(
            json.loads(obj)
        )

    raise TypeError(
        f"Unsupported structure type: {type(obj)}"
    )


# ============================================================
# 5. Load validated external structures
# ============================================================

external_struct_df = pd.read_pickle(
    EXTERNAL_STRUCTURE_CACHE
)

if not isinstance(
    external_struct_df,
    pd.DataFrame,
):
    raise TypeError(
        "External structure cache is not a DataFrame."
    )


for col in [
    "material_id",
    "structure",
]:
    if col not in external_struct_df.columns:
        raise KeyError(
            f"External cache missing column: {col}"
        )


external_struct_df["material_id"] = (
    external_struct_df["material_id"]
    .astype(str)
    .str.strip()
)


if external_struct_df["material_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate material_id found in external structure cache."
    )


external_structure_map = dict(
    zip(
        external_struct_df["material_id"],
        external_struct_df["structure"],
    )
)


if set(ranked["material_id"]) != set(
    external_structure_map.keys()
):
    raise RuntimeError(
        "FINAL SLME IDs and external structure-cache IDs do not match."
    )


print()
print(
    "External validated structures:",
    len(external_structure_map),
)

print("External ID alignment       : PASS")


# ============================================================
# 6. Load authoritative paired split
# ============================================================

fixed_split = pd.read_csv(
    FIXED_SPLIT_CSV
)

required_split_cols = [
    "base_idx",
    "split",
]

missing = [
    c
    for c in required_split_cols
    if c not in fixed_split.columns
]

if missing:
    raise KeyError(
        f"Fixed split missing columns: {missing}"
    )


fixed_split["base_idx"] = pd.to_numeric(
    fixed_split["base_idx"],
    errors="raise",
).astype(int)


if fixed_split["base_idx"].duplicated().any():
    raise RuntimeError(
        "Duplicate base_idx found in fixed split."
    )


split_counts = (
    fixed_split["split"]
    .value_counts()
    .to_dict()
)


print()
print("=" * 110)
print("FIXED-SPLIT PROVENANCE")
print("=" * 110)

print("Fixed split rows :", len(fixed_split))
print("Split counts     :", split_counts)


expected_counts = {
    "train": 7931,
    "val": 992,
    "test": 992,
}

for split_name, expected_n in expected_counts.items():

    actual_n = int(
        split_counts.get(
            split_name,
            0,
        )
    )

    if actual_n != expected_n:
        raise RuntimeError(
            f"{split_name} count mismatch: "
            f"{actual_n} vs expected {expected_n}."
        )


split_by_base_idx = dict(
    zip(
        fixed_split["base_idx"],
        fixed_split["split"],
    )
)


# ============================================================
# 7. Build/load ALL 10,189 original structures
#
# Uses the project's actual CrystalOpticalDataset parser,
# rather than manually guessing JSON layout.
# ============================================================

import utils_tb.data as DATA

DATA = importlib.reload(DATA)


original_ds = DATA.CrystalOpticalDataset(
    zip_path=ORIGINAL_ZIP,
    require_any_target=True,
    require_full_spectrum=False,
    verbose=False,
)


N_ORIGINAL = len(
    original_ds
)


print()
print("=" * 110)
print("FULL ORIGINAL DATABASE")
print("=" * 110)

print(
    "Valid original structures:",
    N_ORIGINAL,
)


if N_ORIGINAL != 10189:
    raise RuntimeError(
        f"Expected 10189 original structures, "
        f"found {N_ORIGINAL}."
    )


# ============================================================
# Helper to extract a structure from get_full_item()
# ============================================================

def extract_structure_from_item(item):

    if isinstance(item, dict):

        for key in [
            "structure",
            "final_structure",
            "crystal_structure",
        ]:

            if key in item:

                try:
                    return to_pmg_structure(
                        item[key]
                    )
                except Exception:
                    pass


        # Recursive fallback
        for value in item.values():

            if isinstance(
                value,
                Structure,
            ):
                return value

            if isinstance(
                value,
                dict,
            ):

                if (
                    "lattice" in value
                    and "sites" in value
                ):

                    try:
                        return Structure.from_dict(
                            value
                        )
                    except Exception:
                        pass


    if isinstance(
        item,
        (list, tuple),
    ):

        for value in item:

            if isinstance(
                value,
                Structure,
            ):
                return value

            if isinstance(
                value,
                dict,
            ):

                try:
                    structure = extract_structure_from_item(
                        value
                    )

                    if structure is not None:
                        return structure

                except Exception:
                    pass


    return None


# ============================================================
# Build cache if absent or stale
# ============================================================

rebuild_original_cache = True

if ORIGINAL_STRUCTURE_CACHE.exists():

    try:

        original_struct_df = pd.read_pickle(
            ORIGINAL_STRUCTURE_CACHE
        )

        if (
            isinstance(
                original_struct_df,
                pd.DataFrame,
            )
            and len(
                original_struct_df
            )
            == N_ORIGINAL
        ):

            rebuild_original_cache = False

            print()
            print(
                "Loaded original structure cache:"
            )

            print(
                ORIGINAL_STRUCTURE_CACHE
            )

    except Exception:
        rebuild_original_cache = True


if rebuild_original_cache:

    print()
    print(
        "Building original structure cache..."
    )

    cache_rows = []
    failure_rows = []

    t_cache = time.time()


    for base_idx in range(
        N_ORIGINAL
    ):

        try:

            item = original_ds.get_full_item(
                base_idx
            )

            meta = (
                item.get(
                    "meta",
                    {},
                )
                if isinstance(
                    item,
                    dict,
                )
                else {}
            )


            structure = extract_structure_from_item(
                item
            )


            if structure is None:

                # Fallback through dataset item
                try:

                    ds_item = original_ds[
                        base_idx
                    ]

                    structure = (
                        extract_structure_from_item(
                            ds_item
                        )
                    )

                except Exception:
                    structure = None


            if structure is None:
                raise ValueError(
                    "Could not extract pymatgen Structure."
                )


            reduced_formula = (
                structure
                .composition
                .reduced_formula
            )


            cache_rows.append({
                "original_base_idx":
                    int(base_idx),

                "mat_id":
                    meta.get(
                        "mat_id",
                        None,
                    ),

                "formula":
                    meta.get(
                        "formula",
                        reduced_formula,
                    ),

                "file_name":
                    meta.get(
                        "file_name",
                        None,
                    ),

                "reduced_formula":
                    reduced_formula,

                "paired_split":
                    split_by_base_idx.get(
                        base_idx,
                        "unpaired",
                    ),

                "structure":
                    structure,
            })


        except Exception as exc:

            failure_rows.append({
                "original_base_idx":
                    int(base_idx),

                "reason":
                    str(exc),
            })


        if (
            (base_idx + 1) % 1000
            == 0
            or base_idx + 1 == N_ORIGINAL
        ):

            print(
                f"Parsed "
                f"{base_idx + 1}/{N_ORIGINAL}"
            )


    original_struct_df = pd.DataFrame(
        cache_rows
    )

    failure_df = pd.DataFrame(
        failure_rows
    )


    original_struct_df.to_pickle(
        ORIGINAL_STRUCTURE_CACHE
    )

    failure_df.to_csv(
        ORIGINAL_STRUCTURE_FAILURES_CSV,
        index=False,
        encoding="utf-8-sig",
    )


    print()
    print(
        "Original structures cached:",
        len(original_struct_df),
    )

    print(
        "Original structure failures:",
        len(failure_df),
    )

    print(
        "Cache build time:",
        f"{(time.time() - t_cache) / 60:.2f} min",
    )


# ============================================================
# Strict cache completeness check
# ============================================================

if len(
    original_struct_df
) != N_ORIGINAL:

    raise RuntimeError(
        "Original structure cache is incomplete.\n"
        f"Expected {N_ORIGINAL}, "
        f"found {len(original_struct_df)}.\n"
        "Do not perform novelty claims until this is fixed."
    )


if (
    original_struct_df[
        "original_base_idx"
    ].duplicated().any()
):

    raise RuntimeError(
        "Duplicate original_base_idx in original structure cache."
    )


print()
print(
    "Original structure cache completeness: PASS"
)


# ============================================================
# 8. Group original structures by actual reduced formula
#
# Formula is ONLY the prefilter.
# ============================================================

original_by_formula = {
    formula: group.reset_index(
        drop=True
    )

    for formula, group in (
        original_struct_df
        .groupby(
            "reduced_formula",
            sort=False,
        )
    )
}


print(
    "Unique reduced formulas in original DB:",
    len(original_by_formula),
)


# ============================================================
# 9. Prepare primary SLME > 30% candidates
# ============================================================

candidate_rows = []

for _, row in primary_candidates.iterrows():

    mid = str(
        row["material_id"]
    )

    structure = to_pmg_structure(
        external_structure_map[
            mid
        ]
    )


    candidate_rows.append({
        "material_id":
            mid,

        "candidate_reduced_formula":
            structure
            .composition
            .reduced_formula,

        "structure":
            structure,
    })


candidate_structure_df = pd.DataFrame(
    candidate_rows
)


primary_candidates = (
    primary_candidates
    .merge(
        candidate_structure_df[
            [
                "material_id",
                "candidate_reduced_formula",
            ]
        ],
        on="material_id",
        how="left",
        validate="one_to_one",
    )
)


candidate_structure_map = dict(
    zip(
        candidate_structure_df[
            "material_id"
        ],
        candidate_structure_df[
            "structure"
        ],
    )
)


# ============================================================
# 10. StructureMatcher audit
# ============================================================

print()
print("=" * 110)
print("STRUCTUREMATCHER AUDIT")
print("=" * 110)

print(
    "Candidates to audit:",
    len(primary_candidates),
)


audit_records = []
match_records = []
error_records = []

t_match = time.time()


for candidate_i, row in primary_candidates.iterrows():

    mid = str(
        row["material_id"]
    )

    formula = str(
        row["candidate_reduced_formula"]
    )

    candidate_structure = (
        candidate_structure_map[
            mid
        ]
    )


    comparison_group = (
        original_by_formula.get(
            formula,
            None,
        )
    )


    overlap_full = False
    overlap_paired = False
    overlap_train = False

    n_comparisons = 0
    n_errors = 0
    n_matches = 0

    matched_mat_id = None
    matched_formula = None
    matched_file = None
    matched_base_idx = None
    matched_split = None


    # No same reduced formula in original DB:
    # structure novelty is automatically established.
    if comparison_group is not None:

        for _, original_row in (
            comparison_group.iterrows()
        ):

            n_comparisons += 1


            try:

                is_match = MATCHER.fit(
                    candidate_structure,
                    original_row[
                        "structure"
                    ],
                )


            except Exception as exc:

                n_errors += 1

                error_records.append({
                    "SLME_rank":
                        int(
                            row[
                                "SLME_rank"
                            ]
                        ),

                    "material_id":
                        mid,

                    "formula_pretty":
                        row[
                            "formula_pretty"
                        ],

                    "candidate_reduced_formula":
                        formula,

                    "original_base_idx":
                        original_row[
                            "original_base_idx"
                        ],

                    "original_file":
                        original_row[
                            "file_name"
                        ],

                    "error":
                        str(exc),
                })

                continue


            if not is_match:
                continue


            n_matches += 1
            overlap_full = True


            split_value = str(
                original_row[
                    "paired_split"
                ]
            )


            if split_value in {
                "train",
                "val",
                "test",
            }:

                overlap_paired = True


            if split_value == "train":

                overlap_train = True


            match_records.append({
                "SLME_rank":
                    int(
                        row[
                            "SLME_rank"
                        ]
                    ),

                "external_material_id":
                    mid,

                "external_formula":
                    row[
                        "formula_pretty"
                    ],

                "candidate_reduced_formula":
                    formula,

                "predicted_SLME_percent":
                    float(
                        row[
                            "predicted_SLME_percent"
                        ]
                    ),

                "original_base_idx":
                    int(
                        original_row[
                            "original_base_idx"
                        ]
                    ),

                "original_mat_id":
                    original_row[
                        "mat_id"
                    ],

                "original_formula":
                    original_row[
                        "formula"
                    ],

                "original_file":
                    original_row[
                        "file_name"
                    ],

                "original_split":
                    split_value,
            })


            # Prefer a train match as representative provenance.
            if (
                matched_file is None
                or split_value == "train"
            ):

                matched_mat_id = (
                    original_row[
                        "mat_id"
                    ]
                )

                matched_formula = (
                    original_row[
                        "formula"
                    ]
                )

                matched_file = (
                    original_row[
                        "file_name"
                    ]
                )

                matched_base_idx = (
                    original_row[
                        "original_base_idx"
                    ]
                )

                matched_split = (
                    split_value
                )


            # Once a training match is established,
            # all relevant overlap questions are resolved.
            if overlap_train:
                break


    # Novelty can only be declared if no matcher error
    # could have hidden an unmatched comparison.
    audit_complete = (
        n_errors == 0
    )


    new_full_database = (
        audit_complete
        and not overlap_full
    )


    training_leakage_free = (
        audit_complete
        and not overlap_train
    )


    audit_records.append({
        "material_id":
            mid,

        "candidate_reduced_formula":
            formula,

        "formula_exists_in_original_database":
            comparison_group is not None,

        "n_formula_prefilter_comparisons":
            n_comparisons,

        "n_structurematcher_errors":
            n_errors,

        "n_structure_matches":
            n_matches,

        "structure_match_audit_complete":
            audit_complete,

        "structure_match_full_original_database":
            overlap_full,

        "structure_match_paired_optical_database":
            overlap_paired,

        "structure_match_training_split":
            overlap_train,

        "training_leakage_free":
            training_leakage_free,

        "new_relative_to_full_original_database":
            new_full_database,

        "matched_original_base_idx":
            matched_base_idx,

        "matched_original_mat_id":
            matched_mat_id,

        "matched_original_formula":
            matched_formula,

        "matched_original_file":
            matched_file,

        "matched_original_split":
            matched_split,
    })


    if (
        (candidate_i + 1) % 25
        == 0
        or candidate_i + 1
        == len(primary_candidates)
    ):

        print(
            f"Audited "
            f"{candidate_i + 1}/"
            f"{len(primary_candidates)}"
        )


audit_df = pd.DataFrame(
    audit_records
)

matches_df = pd.DataFrame(
    match_records
)

errors_df = pd.DataFrame(
    error_records
)


print()
print(
    "StructureMatcher time:",
    f"{(time.time() - t_match) / 60:.2f} min",
)


# ============================================================
# 11. Merge audit back into FINAL ranking
#
# Candidates <= 30% remain explicitly unaudited rather than
# being incorrectly labelled as novel.
# ============================================================

ranked_final = ranked.merge(
    audit_df,
    on="material_id",
    how="left",
    validate="one_to_one",
)


ranked_final[
    "structure_novelty_audited"
] = (
    ranked_final[
        "structure_match_audit_complete"
    ]
    .fillna(False)
    .astype(bool)
)


ranked_final[
    "eligible_new_for_DFT"
] = (
    (
        ranked_final[
            "predicted_SLME_percent"
        ]
        > SLME_THRESHOLD
    )
    & (
        ranked_final[
            "new_relative_to_full_original_database"
        ]
        .fillna(False)
        .astype(bool)
    )
)


# ============================================================
# 12. FINAL discovery subsets
# ============================================================

gt30_audited = (
    ranked_final[
        ranked_final[
            "predicted_SLME_percent"
        ]
        > 30.0
    ]
    .copy()
)


new_gt30 = (
    ranked_final[
        ranked_final[
            "eligible_new_for_DFT"
        ]
    ]
    .copy()
)


new_gt31 = (
    new_gt30[
        new_gt30[
            "predicted_SLME_percent"
        ]
        > 31.0
    ]
    .copy()
)


new_gt32 = (
    new_gt30[
        new_gt30[
            "predicted_SLME_percent"
        ]
        > 32.0
    ]
    .copy()
)


raw_top50 = (
    ranked_final
    .head(
        TOP_K
    )
    .copy()
)


new_top50 = (
    new_gt30
    .head(
        TOP_K
    )
    .copy()
)


new_top50_formula_diverse = (
    new_gt30
    .drop_duplicates(
        "candidate_reduced_formula",
        keep="first",
    )
    .head(
        TOP_K
    )
    .copy()
)


# ============================================================
# 13. Summary
# ============================================================

n_uncertain = int(
    (
        gt30_audited[
            "structure_match_audit_complete"
        ]
        != True
    ).sum()
)


summary = pd.DataFrame([
    {
        "stage":
            "All FINAL SLME screened Zintl candidates",

        "count":
            len(ranked_final),

        "note":
            "Direct-alpha -> SLME, L=500 nm.",
    },

    {
        "stage":
            "Predicted SLME > 30%",

        "count":
            len(gt30_audited),

        "note":
            "Primary solar-cell discovery pool before novelty filtering.",
    },

    {
        "stage":
            "StructureMatcher-equivalent to training split",

        "count":
            int(
                gt30_audited[
                    "structure_match_training_split"
                ]
                .fillna(False)
                .sum()
            ),

        "note":
            "Training-structure overlap.",
    },

    {
        "stage":
            "StructureMatcher-equivalent to paired optical database",

        "count":
            int(
                gt30_audited[
                    "structure_match_paired_optical_database"
                ]
                .fillna(False)
                .sum()
            ),

        "note":
            "Equivalent to train/val/test optical benchmark.",
    },

    {
        "stage":
            "StructureMatcher-equivalent to full original database",

        "count":
            int(
                gt30_audited[
                    "structure_match_full_original_database"
                ]
                .fillna(False)
                .sum()
            ),

        "note":
            "Equivalent to any of the 10,189 original structures.",
    },

    {
        "stage":
            "Novelty audit uncertain due to matcher error",

        "count":
            n_uncertain,

        "note":
            "These candidates are NOT automatically classified as new.",
    },

    {
        "stage":
            "SLME > 30% and structure-new",

        "count":
            len(new_gt30),

        "note":
            "Primary new-for-DFT pool.",
    },

    {
        "stage":
            "SLME > 31% and structure-new",

        "count":
            len(new_gt31),

        "note":
            "Higher-efficiency new-for-DFT subset.",
    },

    {
        "stage":
            "SLME > 32% and structure-new",

        "count":
            len(new_gt32),

        "note":
            "Highest-efficiency new-for-DFT subset.",
    },

    {
        "stage":
            "Top-50 structure-new by SLME",

        "count":
            len(new_top50),

        "note":
            "Primary ranked shortlist before DFT.",
    },
])


# ============================================================
# 14. Save authoritative outputs
# ============================================================

ALL_FINAL_CSV = (
    OUT_DIR
    / "zintl_all_FINAL_SLME_with_structure_overlap_flags.csv"
)

GT30_AUDIT_CSV = (
    OUT_DIR
    / "zintl_SLME_gt30_structure_overlap_audit.csv"
)

NEW_GT30_CSV = (
    OUT_DIR
    / "zintl_SLME_gt30_new_for_DFT.csv"
)

NEW_GT31_CSV = (
    OUT_DIR
    / "zintl_SLME_gt31_new_for_DFT.csv"
)

NEW_GT32_CSV = (
    OUT_DIR
    / "zintl_SLME_gt32_new_for_DFT.csv"
)

RAW_TOP50_CSV = (
    OUT_DIR
    / "zintl_top50_raw_FINAL_SLME.csv"
)

NEW_TOP50_CSV = (
    OUT_DIR
    / "zintl_top50_structure_new_FINAL_SLME.csv"
)

FORMULA_DIVERSE_CSV = (
    OUT_DIR
    / "zintl_top50_structure_new_formula_diverse_FINAL_SLME.csv"
)

MATCHES_CSV = (
    OUT_DIR
    / "zintl_structurematcher_matches_original_database.csv"
)

ERRORS_CSV = (
    OUT_DIR
    / "zintl_structurematcher_errors.csv"
)

SUMMARY_CSV = (
    OUT_DIR
    / "zintl_FINAL_SLME_structure_novelty_summary.csv"
)


ranked_final.to_csv(
    ALL_FINAL_CSV,
    index=False,
    encoding="utf-8-sig",
)

gt30_audited.to_csv(
    GT30_AUDIT_CSV,
    index=False,
    encoding="utf-8-sig",
)

new_gt30.to_csv(
    NEW_GT30_CSV,
    index=False,
    encoding="utf-8-sig",
)

new_gt31.to_csv(
    NEW_GT31_CSV,
    index=False,
    encoding="utf-8-sig",
)

new_gt32.to_csv(
    NEW_GT32_CSV,
    index=False,
    encoding="utf-8-sig",
)

raw_top50.to_csv(
    RAW_TOP50_CSV,
    index=False,
    encoding="utf-8-sig",
)

new_top50.to_csv(
    NEW_TOP50_CSV,
    index=False,
    encoding="utf-8-sig",
)

new_top50_formula_diverse.to_csv(
    FORMULA_DIVERSE_CSV,
    index=False,
    encoding="utf-8-sig",
)

matches_df.to_csv(
    MATCHES_CSV,
    index=False,
    encoding="utf-8-sig",
)

errors_df.to_csv(
    ERRORS_CSV,
    index=False,
    encoding="utf-8-sig",
)

summary.to_csv(
    SUMMARY_CSV,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 15. FINAL REPORT
# ============================================================

print()
print("=" * 110)
print("FINAL STRUCTURE-NOVELTY AUDIT FINISHED")
print("=" * 110)

display(
    summary
)


print()
print(
    "SLME > 30% candidates:",
    len(gt30_audited),
)

print(
    "Training-structure overlaps:",
    int(
        gt30_audited[
            "structure_match_training_split"
        ]
        .fillna(False)
        .sum()
    ),
)

print(
    "Paired optical DB overlaps:",
    int(
        gt30_audited[
            "structure_match_paired_optical_database"
        ]
        .fillna(False)
        .sum()
    ),
)

print(
    "Full original DB overlaps:",
    int(
        gt30_audited[
            "structure_match_full_original_database"
        ]
        .fillna(False)
        .sum()
    ),
)

print(
    "Matcher-uncertain candidates:",
    n_uncertain,
)

print()
print(
    "SLME > 30% AND structure-new:",
    len(new_gt30),
)

print(
    "SLME > 31% AND structure-new:",
    len(new_gt31),
)

print(
    "SLME > 32% AND structure-new:",
    len(new_gt32),
)


print()
print("Saved:")
print("All ranked + flags       :", ALL_FINAL_CSV)
print("SLME >30 overlap audit   :", GT30_AUDIT_CSV)
print("SLME >30 new for DFT     :", NEW_GT30_CSV)
print("SLME >31 new for DFT     :", NEW_GT31_CSV)
print("SLME >32 new for DFT     :", NEW_GT32_CSV)
print("Raw Top-50               :", RAW_TOP50_CSV)
print("Structure-new Top-50     :", NEW_TOP50_CSV)
print("Formula-diverse Top-50   :", FORMULA_DIVERSE_CSV)
print("Structure matches        :", MATCHES_CSV)
print("Matcher errors           :", ERRORS_CSV)
print("Summary                  :", SUMMARY_CSV)


# ============================================================
# 16. Preview DFT targets
# ============================================================

display_cols = [
    "SLME_rank",
    "material_id",
    "formula_pretty",
    "candidate_reduced_formula",
    "band_gap",
    "energy_above_hull",
    "is_stable",
    "predicted_SLME_percent",
    "structure_match_training_split",
    "structure_match_full_original_database",
    "new_relative_to_full_original_database",
    "n_structure_matches",
    "n_structurematcher_errors",
]


display_cols = [
    c
    for c in display_cols
    if c in new_gt30.columns
]


print()
print(
    "Top structure-new candidates with predicted SLME > 30%:"
)

display(
    new_gt30[
        display_cols
    ].head(
        50
    )
)

FINAL ZINTL STRUCTURE-NOVELTY AUDIT
Primary target       : predicted SLME > 30%
Novelty reference    : full original database
Leakage reference    : original training split

StructureMatcher:
  ltol              : 0.20
  stol              : 0.30
  angle_tol         : 5 deg
  primitive_cell    : True
  scale             : True
  attempt_supercell : True

SLME DISCOVERY POOL
All screened candidates : 1100
SLME > 30% candidates : 355
SLME > 31% candidates  : 282
SLME > 32% candidates  : 205

External validated structures: 1100
External ID alignment       : PASS

FIXED-SPLIT PROVENANCE
Fixed split rows : 9915
Split counts     : {'train': 7931, 'val': 992, 'test': 992}

FULL ORIGINAL DATABASE
Valid original structures: 10189

Loaded original structure cache:
D:\TB3\processed\paired_training\paper_outputs\dft_validation_shortlist\discovery_first_zintl_SLME\original_database_10189_structures.pkl

Original structure cache completeness: PASS
Unique reduced formulas in original DB: 7310

STRUCTU

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_21844\2639310564.py:1290: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_21844\2639310564.py:1308: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_21844\2639310564.py:1438: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('fu

,stage,count,note
0,All FINAL SLME screened Zintl candidates,1100,"Direct-alpha -> SLME, L=500 nm."
1,Predicted SLME > 30%,355,Primary solar-cell discovery pool before novel...
2,StructureMatcher-equivalent to training split,26,Training-structure overlap.
3,StructureMatcher-equivalent to paired optical ...,30,Equivalent to train/val/test optical benchmark.
4,StructureMatcher-equivalent to full original d...,30,"Equivalent to any of the 10,189 original struc..."
5,Novelty audit uncertain due to matcher error,0,These candidates are NOT automatically classif...
6,SLME > 30% and structure-new,325,Primary new-for-DFT pool.
7,SLME > 31% and structure-new,258,Higher-efficiency new-for-DFT subset.
8,SLME > 32% and structure-new,185,Highest-efficiency new-for-DFT subset.
9,Top-50 structure-new by SLME,50,Primary ranked shortlist before DFT.



SLME > 30% candidates: 355
Training-structure overlaps: 26
Paired optical DB overlaps: 30
Full original DB overlaps: 30
Matcher-uncertain candidates: 0

SLME > 30% AND structure-new: 325
SLME > 31% AND structure-new: 258
SLME > 32% AND structure-new: 185

Saved:
All ranked + flags       : D:\TB3\processed\paired_training\paper_outputs\dft_validation_shortlist\discovery_first_zintl_SLME\zintl_all_FINAL_SLME_with_structure_overlap_flags.csv
SLME >30 overlap audit   : D:\TB3\processed\paired_training\paper_outputs\dft_validation_shortlist\discovery_first_zintl_SLME\zintl_SLME_gt30_structure_overlap_audit.csv
SLME >30 new for DFT     : D:\TB3\processed\paired_training\paper_outputs\dft_validation_shortlist\discovery_first_zintl_SLME\zintl_SLME_gt30_new_for_DFT.csv
SLME >31 new for DFT     : D:\TB3\processed\paired_training\paper_outputs\dft_validation_shortlist\discovery_first_zintl_SLME\zintl_SLME_gt31_new_for_DFT.csv
SLME >32 new for DFT     : D:\TB3\processed\paired_training\paper_outp

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_21844\2639310564.py:1690: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_21844\2639310564.py:1701: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_21844\2639310564.py:1712: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('fu

,SLME_rank,material_id,formula_pretty,candidate_reduced_formula,band_gap,energy_above_hull,is_stable,predicted_SLME_percent,structure_match_training_split,structure_match_full_original_database,new_relative_to_full_original_database,n_structure_matches,n_structurematcher_errors
0,1,mp-btzq,NaP5,NaP5,1.354200,0.009960,False,33.753417,False,False,True,0.0,0.0
1,2,mp-bevqy,Rb2Ag(AsSe2)3,Rb2Ag(AsSe2)3,1.349700,0.000000,True,33.727537,False,False,True,0.0,0.0
2,3,mp-cqyqi,KIn(PSe3)2,KIn(PSe3)2,1.351200,0.000479,False,33.690766,False,False,True,0.0,0.0
3,4,mp-bsnx,Ba2B4Se13,Ba2B4Se13,1.333000,0.000000,True,33.674518,False,False,True,0.0,0.0
4,5,mp-cqcrk,KAgPSe3,KAgPSe3,1.354700,0.000000,True,33.668234,False,False,True,0.0,0.0
5,6,mp-cqpjs,Rb2As2Pd,Rb2As2Pd,1.145847,0.000000,True,33.667991,False,False,True,0.0,0.0
6,7,mp-crclc,Ca11GaSb9,Ca11GaSb9,1.170400,0.000000,True,33.666295,False,False,True,0.0,0.0
7,8,mp-bkski,CsAs2Se3,CsAs2Se3,1.389100,0.000000,True,33.661544,False,False,True,0.0,0.0
8,9,mp-ekf,CsSbSe2,CsSbSe2,1.134800,0.000000,True,33.656419,False,False,True,0.0,0.0
9,10,mp-ue,Ba3As14,Ba3As14,1.176300,0.000000,True,33.652101,False,False,True,0.0,0.0


In [11]:
# CELL 9 FINAL
# Export FINAL Top-50 Zintl candidates for DFT
#
# Selection basis:
#   Direct-alpha -> SLME (500 nm)
#   -> predicted SLME > 30%
#   -> StructureMatcher-new relative to full original database
#   -> rank by predicted SLME
#
# Outputs:
#   1. Top-50 manifest CSV
#   2. Material-ID-only CSV / TXT
#   3. 50 CIF files
#   4. 50 JSON files
#   5. One combined JSON containing all structures
#
# No SRP ranking is used here.
# ============================================================

from pathlib import Path
import json
import shutil

import numpy as np
import pandas as pd

from IPython.display import display
from pymatgen.core import Structure


# ============================================================
# 0. Paths
# ============================================================

PROJECT_ROOT = Path(r"D:\TB3")

ROOT = (
    PROJECT_ROOT
    / "processed"
    / "paired_training"
)

DISCOVERY_DIR = (
    ROOT
    / "paper_outputs"
    / "dft_validation_shortlist"
    / "discovery_first_zintl_SLME"
)

TOP50_CSV = (
    DISCOVERY_DIR
    / "zintl_top50_structure_new_FINAL_SLME.csv"
)

STRUCTURE_CACHE = (
    ROOT
    / "paper_outputs"
    / "screening_srp_200nm"
    / "external_pool_cache"
    / "screening_zintl_like_external_fixed_1100_with_structures.pkl"
)

EXPORT_DIR = (
    DISCOVERY_DIR
    / "FINAL_top50_structure_new_SLME_for_DFT"
)

CIF_DIR = (
    EXPORT_DIR
    / "cif"
)

JSON_DIR = (
    EXPORT_DIR
    / "json_per_material"
)

CIF_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

JSON_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 1. Validate inputs
# ============================================================

if not TOP50_CSV.exists():
    raise FileNotFoundError(
        f"FINAL Top-50 CSV not found:\n{TOP50_CSV}"
    )

if not STRUCTURE_CACHE.exists():
    raise FileNotFoundError(
        f"Validated structure cache not found:\n{STRUCTURE_CACHE}"
    )


top50 = pd.read_csv(
    TOP50_CSV
)

struct_df = pd.read_pickle(
    STRUCTURE_CACHE
)


print("=" * 110)
print("FINAL TOP-50 SLME DFT EXPORT")
print("=" * 110)

print("Source shortlist :", TOP50_CSV)
print("Structure cache  :", STRUCTURE_CACHE)
print("Export folder    :", EXPORT_DIR)


# ============================================================
# 2. Strict shortlist checks
# ============================================================

if len(top50) != 50:
    raise RuntimeError(
        f"Expected exactly 50 candidates, found {len(top50)}."
    )


required_cols = [
    "SLME_rank",
    "material_id",
    "formula_pretty",
    "band_gap",
    "energy_above_hull",
    "is_stable",
    "predicted_SLME_percent",
    "new_relative_to_full_original_database",
]

missing = [
    c for c in required_cols
    if c not in top50.columns
]

if missing:
    raise KeyError(
        f"Top-50 CSV is missing columns: {missing}"
    )


top50["material_id"] = (
    top50["material_id"]
    .astype(str)
    .str.strip()
)

top50["predicted_SLME_percent"] = pd.to_numeric(
    top50["predicted_SLME_percent"],
    errors="coerce",
)

top50["band_gap"] = pd.to_numeric(
    top50["band_gap"],
    errors="coerce",
)

top50["energy_above_hull"] = pd.to_numeric(
    top50["energy_above_hull"],
    errors="coerce",
)


if top50["material_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate material_id detected in FINAL Top-50."
    )


if top50["predicted_SLME_percent"].isna().any():
    raise RuntimeError(
        "Missing predicted SLME values in FINAL Top-50."
    )


if not (
    top50["predicted_SLME_percent"] > 30.0
).all():
    bad = top50[
        top50["predicted_SLME_percent"] <= 30.0
    ]

    raise RuntimeError(
        f"{len(bad)} Top-50 candidates do not satisfy SLME > 30%."
    )


# Robust bool conversion
def to_bool_series(series):

    if series.dtype == bool:
        return series

    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(
            [
                "true",
                "1",
                "yes",
            ]
        )
    )


novelty_flag = to_bool_series(
    top50[
        "new_relative_to_full_original_database"
    ]
)

if not novelty_flag.all():

    bad = top50[
        ~novelty_flag
    ]

    raise RuntimeError(
        f"{len(bad)} candidates are not confirmed structure-new."
    )


if (
    "structure_match_audit_complete"
    in top50.columns
):

    audit_complete = to_bool_series(
        top50[
            "structure_match_audit_complete"
        ]
    )

    if not audit_complete.all():
        raise RuntimeError(
            "At least one Top-50 candidate has incomplete "
            "StructureMatcher novelty audit."
        )


if (
    "n_structurematcher_errors"
    in top50.columns
):

    n_errors = pd.to_numeric(
        top50[
            "n_structurematcher_errors"
        ],
        errors="coerce",
    ).fillna(0)

    if (n_errors != 0).any():
        raise RuntimeError(
            "At least one Top-50 candidate has a StructureMatcher error."
        )


top50 = (
    top50
    .sort_values(
        "predicted_SLME_percent",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)

top50[
    "DFT_shortlist_rank"
] = np.arange(
    1,
    51,
)


print()
print("Top-50 validation:")
print("  Number of candidates       :", len(top50))
print(
    "  Minimum predicted SLME     :",
    f"{top50['predicted_SLME_percent'].min():.3f}%"
)
print(
    "  Maximum predicted SLME     :",
    f"{top50['predicted_SLME_percent'].max():.3f}%"
)
print(
    "  Structure-new candidates   :",
    int(novelty_flag.sum())
)


# ============================================================
# 3. Structure cache validation
# ============================================================

if not isinstance(
    struct_df,
    pd.DataFrame,
):
    raise TypeError(
        "Structure cache must be a pandas DataFrame."
    )


for col in [
    "material_id",
    "structure",
]:

    if col not in struct_df.columns:
        raise KeyError(
            f"Structure cache missing column: {col}"
        )


struct_df[
    "material_id"
] = (
    struct_df[
        "material_id"
    ]
    .astype(str)
    .str.strip()
)


if struct_df[
    "material_id"
].duplicated().any():
    raise RuntimeError(
        "Duplicate material_id detected in structure cache."
    )


structure_map = dict(
    zip(
        struct_df[
            "material_id"
        ],
        struct_df[
            "structure"
        ],
    )
)


missing_structures = [
    mid
    for mid in top50[
        "material_id"
    ]
    if mid not in structure_map
]


if missing_structures:

    raise RuntimeError(
        "Missing structures for Top-50 candidates:\n"
        + "\n".join(
            missing_structures
        )
    )


print(
    "  Structures available      :",
    len(top50) - len(missing_structures),
    "/ 50"
)


# ============================================================
# 4. Helpers
# ============================================================

def to_pmg_structure(obj):

    if isinstance(
        obj,
        Structure,
    ):
        return obj

    if isinstance(
        obj,
        dict,
    ):
        return Structure.from_dict(
            obj
        )

    if isinstance(
        obj,
        str,
    ):
        return Structure.from_dict(
            json.loads(
                obj
            )
        )

    raise TypeError(
        f"Unsupported structure type: {type(obj)}"
    )


def safe_filename(text):

    text = str(
        text
    )

    for ch in [
        "\\",
        "/",
        ":",
        "*",
        "?",
        '"',
        "<",
        ">",
        "|",
        " ",
        "(",
        ")",
        "[",
        "]",
        "{",
        "}",
    ]:

        text = text.replace(
            ch,
            "_",
        )

    return text


def safe_bool(value):

    if pd.isna(
        value
    ):
        return None

    if isinstance(
        value,
        (bool, np.bool_),
    ):
        return bool(
            value
        )

    return (
        str(value)
        .strip()
        .lower()
        in {
            "true",
            "1",
            "yes",
        }
    )


# ============================================================
# 5. Export Top-50 structures
# ============================================================

combined_records = []
manifest_rows = []
failed_rows = []


for _, row in top50.iterrows():

    shortlist_rank = int(
        row[
            "DFT_shortlist_rank"
        ]
    )

    slme_rank = int(
        row[
            "SLME_rank"
        ]
    )

    material_id = str(
        row[
            "material_id"
        ]
    ).strip()

    formula = str(
        row[
            "formula_pretty"
        ]
    ).strip()


    try:

        structure = to_pmg_structure(
            structure_map[
                material_id
            ]
        )


        stem = (
            f"rank{shortlist_rank:02d}_"
            f"SLMErank{slme_rank:03d}_"
            f"{safe_filename(material_id)}_"
            f"{safe_filename(formula)}"
        )


        cif_path = (
            CIF_DIR
            / f"{stem}.cif"
        )

        json_path = (
            JSON_DIR
            / f"{stem}.json"
        )


        # ----------------------------------------------------
        # CIF
        # ----------------------------------------------------

        structure.to(
            filename=str(
                cif_path
            ),
            fmt="cif",
        )


        # ----------------------------------------------------
        # Material record
        # ----------------------------------------------------

        record = {
            "DFT_shortlist_rank":
                shortlist_rank,

            "SLME_rank":
                slme_rank,

            "material_id":
                material_id,

            "formula_pretty":
                formula,

            "composition_reduced":
                structure.composition.reduced_formula,

            "elements":
                sorted(
                    [
                        str(el)
                        for el
                        in structure.composition.elements
                    ]
                ),

            "num_elements":
                int(
                    len(
                        structure.composition.elements
                    )
                ),

            "num_sites":
                int(
                    len(
                        structure.sites
                    )
                ),

            "predicted_SLME_percent":
                float(
                    row[
                        "predicted_SLME_percent"
                    ]
                ),

            "band_gap_eV":
                float(
                    row[
                        "band_gap"
                    ]
                ),

            "energy_above_hull_eV_atom":
                float(
                    row[
                        "energy_above_hull"
                    ]
                ),

            "is_stable":
                safe_bool(
                    row[
                        "is_stable"
                    ]
                ),

            "structure_new_relative_to_original_database":
                True,

            "structurematcher_training_overlap":
                (
                    safe_bool(
                        row[
                            "structure_match_training_split"
                        ]
                    )
                    if "structure_match_training_split"
                    in row.index
                    else None
                ),

            "structurematcher_full_database_overlap":
                (
                    safe_bool(
                        row[
                            "structure_match_full_original_database"
                        ]
                    )
                    if "structure_match_full_original_database"
                    in row.index
                    else False
                ),

            "SLME_protocol": {
                "route":
                    "Direct-alpha SSL-init GINE -> SLME",

                "thickness_nm":
                    500.0,

                "temperature_K":
                    300.0,

                "fr":
                    1.0,

                "solar_spectrum":
                    "ASTM G173 AM1.5G",

                "model_seeds":
                    [
                        42,
                        123,
                        2025,
                    ],

                "graph_features":
                    "19 node / 24 edge",

                "primary_discovery_threshold_percent":
                    30.0,
            },

            "lattice": {
                "a":
                    float(
                        structure.lattice.a
                    ),

                "b":
                    float(
                        structure.lattice.b
                    ),

                "c":
                    float(
                        structure.lattice.c
                    ),

                "alpha":
                    float(
                        structure.lattice.alpha
                    ),

                "beta":
                    float(
                        structure.lattice.beta
                    ),

                "gamma":
                    float(
                        structure.lattice.gamma
                    ),

                "volume":
                    float(
                        structure.lattice.volume
                    ),

                "matrix":
                    structure.lattice.matrix.tolist(),
            },

            "cif_file":
                str(
                    cif_path.relative_to(
                        EXPORT_DIR
                    )
                ),

            "structure":
                structure.as_dict(),
        }


        with open(
            json_path,
            "w",
            encoding="utf-8",
        ) as f:

            json.dump(
                record,
                f,
                indent=2,
                ensure_ascii=False,
            )


        combined_records.append(
            record
        )


        manifest_rows.append({
            "DFT_shortlist_rank":
                shortlist_rank,

            "SLME_rank":
                slme_rank,

            "material_id":
                material_id,

            "formula_pretty":
                formula,

            "predicted_SLME_percent":
                float(
                    row[
                        "predicted_SLME_percent"
                    ]
                ),

            "band_gap_eV":
                float(
                    row[
                        "band_gap"
                    ]
                ),

            "energy_above_hull_eV_atom":
                float(
                    row[
                        "energy_above_hull"
                    ]
                ),

            "is_stable":
                safe_bool(
                    row[
                        "is_stable"
                    ]
                ),

            "num_sites":
                int(
                    len(
                        structure.sites
                    )
                ),

            "num_elements":
                int(
                    len(
                        structure.composition.elements
                    )
                ),

            "elements":
                ",".join(
                    sorted(
                        [
                            str(el)
                            for el
                            in structure.composition.elements
                        ]
                    )
                ),

            "structure_new":
                True,

            "cif_file":
                str(
                    cif_path
                ),

            "json_file":
                str(
                    json_path
                ),

            "export_status":
                "OK",

            "reason":
                "",
        })


    except Exception as exc:

        failed_rows.append({
            "DFT_shortlist_rank":
                shortlist_rank,

            "SLME_rank":
                slme_rank,

            "material_id":
                material_id,

            "formula_pretty":
                formula,

            "export_status":
                "FAILED",

            "reason":
                str(
                    exc
                ),
        })


# ============================================================
# 6. Build output tables
# ============================================================

manifest_df = pd.DataFrame(
    manifest_rows
)

failed_df = pd.DataFrame(
    failed_rows
)


if len(
    manifest_df
) != 50:

    raise RuntimeError(
        f"Only {len(manifest_df)}/50 structures were exported."
    )


if len(
    failed_df
) != 0:

    raise RuntimeError(
        f"{len(failed_df)} Top-50 exports failed."
    )


# ============================================================
# 7. Material-ID-only files for DFT collaborator
# ============================================================

ID_ONLY_CSV = (
    EXPORT_DIR
    / "FINAL_top50_material_IDs.csv"
)

ID_ONLY_TXT = (
    EXPORT_DIR
    / "FINAL_top50_material_IDs.txt"
)


id_only_df = (
    manifest_df[
        [
            "DFT_shortlist_rank",
            "material_id",
            "formula_pretty",
            "predicted_SLME_percent",
            "band_gap_eV",
            "energy_above_hull_eV_atom",
            "is_stable",
        ]
    ]
    .copy()
)


id_only_df.to_csv(
    ID_ONLY_CSV,
    index=False,
    encoding="utf-8-sig",
)


with open(
    ID_ONLY_TXT,
    "w",
    encoding="utf-8",
) as f:

    for _, row in id_only_df.iterrows():

        f.write(
            f"{int(row['DFT_shortlist_rank']):02d}\t"
            f"{row['material_id']}\t"
            f"{row['formula_pretty']}\t"
            f"SLME={row['predicted_SLME_percent']:.3f}%\n"
        )


# ============================================================
# 8. Combined JSON + manifest
# ============================================================

COMBINED_JSON = (
    EXPORT_DIR
    / "FINAL_top50_combined_structures.json"
)

MANIFEST_CSV = (
    EXPORT_DIR
    / "FINAL_top50_export_manifest.csv"
)

FAILED_CSV = (
    EXPORT_DIR
    / "FINAL_top50_export_failures.csv"
)


with open(
    COMBINED_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        combined_records,
        f,
        indent=2,
        ensure_ascii=False,
    )


manifest_df.to_csv(
    MANIFEST_CSV,
    index=False,
    encoding="utf-8-sig",
)

failed_df.to_csv(
    FAILED_CSV,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 9. Optional ZIP package for sharing
# ============================================================

ZIP_BASE = (
    DISCOVERY_DIR
    / "FINAL_top50_structure_new_SLME_for_DFT"
)


zip_path = shutil.make_archive(
    str(
        ZIP_BASE
    ),
    "zip",
    root_dir=str(
        EXPORT_DIR
    ),
)


# ============================================================
# 10. Final audit
# ============================================================

print()
print("=" * 110)
print("FINAL TOP-50 DFT EXPORT FINISHED")
print("=" * 110)

print(
    "Candidates exported :",
    len(
        manifest_df
    ),
)

print(
    "Failed exports      :",
    len(
        failed_df
    ),
)

print(
    "Stable candidates   :",
    int(
        manifest_df[
            "is_stable"
        ].sum()
    ),
)

print(
    "SLME range          :",
    f"{manifest_df['predicted_SLME_percent'].min():.3f}%"
    " to "
    f"{manifest_df['predicted_SLME_percent'].max():.3f}%",
)

print(
    "All > 30%           :",
    bool(
        (
            manifest_df[
                "predicted_SLME_percent"
            ]
            > 30.0
        ).all()
    ),
)

print(
    "All structure-new   :",
    True,
)

print()
print("Material IDs:")
print(ID_ONLY_CSV)
print(ID_ONLY_TXT)

print()
print("Structures:")
print(CIF_DIR)
print(JSON_DIR)

print()
print("Combined JSON:")
print(COMBINED_JSON)

print()
print("Manifest:")
print(MANIFEST_CSV)

print()
print("ZIP package:")
print(zip_path)


# ============================================================
# 11. Preview Top-50
# ============================================================

display_cols = [
    "DFT_shortlist_rank",
    "SLME_rank",
    "material_id",
    "formula_pretty",
    "predicted_SLME_percent",
    "band_gap_eV",
    "energy_above_hull_eV_atom",
    "is_stable",
    "num_sites",
]


print()
print("FINAL Top-50 for DFT:")

display(
    manifest_df[
        display_cols
    ]
)

FINAL TOP-50 SLME DFT EXPORT
Source shortlist : D:\TB3\processed\paired_training\paper_outputs\dft_validation_shortlist\discovery_first_zintl_SLME\zintl_top50_structure_new_FINAL_SLME.csv
Structure cache  : D:\TB3\processed\paired_training\paper_outputs\screening_srp_200nm\external_pool_cache\screening_zintl_like_external_fixed_1100_with_structures.pkl
Export folder    : D:\TB3\processed\paired_training\paper_outputs\dft_validation_shortlist\discovery_first_zintl_SLME\FINAL_top50_structure_new_SLME_for_DFT

Top-50 validation:
  Number of candidates       : 50
  Minimum predicted SLME     : 33.387%
  Maximum predicted SLME     : 33.753%
  Structure-new candidates   : 50
  Structures available      : 50 / 50

FINAL TOP-50 DFT EXPORT FINISHED
Candidates exported : 50
Failed exports      : 0
Stable candidates   : 42
SLME range          : 33.387% to 33.753%
All > 30%           : True
All structure-new   : True

Material IDs:
D:\TB3\processed\paired_training\paper_outputs\dft_validation_shor

,DFT_shortlist_rank,SLME_rank,material_id,formula_pretty,predicted_SLME_percent,band_gap_eV,energy_above_hull_eV_atom,is_stable,num_sites
0,1,1,mp-btzq,NaP5,33.753417,1.354200,0.009960,False,24
1,2,2,mp-bevqy,Rb2Ag(AsSe2)3,33.727537,1.349700,0.000000,True,48
2,3,3,mp-cqyqi,KIn(PSe3)2,33.690766,1.351200,0.000479,False,40
3,4,4,mp-bsnx,Ba2B4Se13,33.674518,1.333000,0.000000,True,76
4,5,5,mp-cqcrk,KAgPSe3,33.668234,1.354700,0.000000,True,72
5,6,6,mp-cqpjs,Rb2As2Pd,33.667991,1.145847,0.000000,True,10
6,7,7,mp-crclc,Ca11GaSb9,33.666295,1.170400,0.000000,True,42
7,8,8,mp-bkski,CsAs2Se3,33.661544,1.389100,0.000000,True,48
8,9,9,mp-ekf,CsSbSe2,33.656419,1.134800,0.000000,True,16
9,10,10,mp-ue,Ba3As14,33.652101,1.176300,0.000000,True,34


## Cell 10. Build Database 2 raw AB2C2 C=P/N pool

Database 2 is rebuilt from Materials Project instead of reusing the old combined shortlist.  
Raw pool condition: ternary AB2C2 reduced composition with P2 or N2 and band gap > 0.

In [12]:
# Cell 10. Load Database 2 raw AB2C2 C=P/N snapshot
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# 1. Đường dẫn — giữ tên biến cho các cell tiếp theo
# ------------------------------------------------------------
ROOT = Path(r"D:\TB3\processed\paired_training")

POOL_DIR = ROOT / "paper_outputs" / "external_pools"
DB2_POOL_DIR = POOL_DIR / "database2_ab2c2_C_PN"

DB2_RAW_CSV = (
    DB2_POOL_DIR
    / "pool_ab2c2_C_PN_bandgap_gt0_raw.csv"
)
DB2_RAW_PKL = (
    DB2_POOL_DIR
    / "pool_ab2c2_C_PN_bandgap_gt0_raw_with_structures.pkl"
)


# ------------------------------------------------------------
# 2. Kiểm tra snapshot
# ------------------------------------------------------------
missing_files = [
    path for path in (DB2_RAW_CSV, DB2_RAW_PKL)
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Thiếu file snapshot DB2:\n"
        + "\n".join(str(path) for path in missing_files)
        + "\nCell dừng để tránh tự query lại hoặc thay đổi dataset."
    )

db2_raw_pool = pd.read_pickle(DB2_RAW_PKL).copy()
raw_csv_df = pd.read_csv(DB2_RAW_CSV)

required_columns = [
    "material_id",
    "formula_pretty",
    "band_gap",
    "energy_above_hull",
    "structure",
]

missing_columns = [
    column for column in required_columns
    if column not in db2_raw_pool.columns
]
if missing_columns:
    raise KeyError(f"Structure cache thiếu cột: {missing_columns}")

if db2_raw_pool.empty:
    raise ValueError("Snapshot DB2 không có mẫu.")

if "material_id" not in raw_csv_df.columns:
    raise KeyError("Raw CSV thiếu cột material_id.")


# ------------------------------------------------------------
# 3. Kiểm tra ID và tính nhất quán CSV ↔ structure cache
# ------------------------------------------------------------
for label, frame in (
    ("Structure cache", db2_raw_pool),
    ("Raw CSV", raw_csv_df),
):
    if frame["material_id"].isna().any():
        raise ValueError(f"{label} có material_id bị thiếu.")

    frame["material_id"] = (
        frame["material_id"].astype(str).str.strip()
    )

    if frame["material_id"].eq("").any():
        raise ValueError(f"{label} có material_id rỗng.")

    if frame["material_id"].duplicated().any():
        raise ValueError(f"{label} có material_id bị trùng.")

cache_ids = set(db2_raw_pool["material_id"])
csv_ids = set(raw_csv_df["material_id"])

if cache_ids != csv_ids:
    raise ValueError(
        "Raw CSV và structure cache không có cùng tập material_id.\n"
        f"Chỉ có trong cache: {sorted(cache_ids - csv_ids)[:10]}\n"
        f"Chỉ có trong CSV: {sorted(csv_ids - cache_ids)[:10]}"
    )

if db2_raw_pool["structure"].isna().any():
    raise ValueError("Structure cache có mẫu thiếu structure.")


# ------------------------------------------------------------
# 4. Chuẩn hóa số trong bộ nhớ — không sửa file nguồn
# ------------------------------------------------------------
for column in ("band_gap", "energy_above_hull"):
    db2_raw_pool[column] = pd.to_numeric(
        db2_raw_pool[column],
        errors="coerce",
    )

band_gap = db2_raw_pool["band_gap"].to_numpy(dtype=float)

if not np.isfinite(band_gap).all() or (band_gap <= 0).any():
    raise ValueError(
        "Raw pool có band_gap không hữu hạn hoặc <= 0; "
        "cần kiểm tra snapshot trước khi tiếp tục."
    )

n_missing_ehull = int(
    (~np.isfinite(
        db2_raw_pool["energy_above_hull"].to_numpy(dtype=float)
    )).sum()
)


# ------------------------------------------------------------
# 5. Báo cáo
# ------------------------------------------------------------
print("Chế độ: đọc snapshot DB2 hiện có.")
print("Database 2 raw AB2C2 C=P/N pool:", len(db2_raw_pool))
print("Duplicated material_id:", 0)
print(
    "Duplicated formula_pretty:",
    int(db2_raw_pool["formula_pretty"].duplicated().sum()),
)
print("Số mẫu có energy_above_hull không hữu hạn:", n_missing_ehull)
print("Tập material_id giữa CSV và cache: khớp.")
print("Raw CSV:", DB2_RAW_CSV)
print("Structure cache:", DB2_RAW_PKL)
print("Không có file nào được tạo hoặc ghi đè.")

display(
    db2_raw_pool
    .drop(columns=["structure"], errors="ignore")
    .head(20)
)

Chế độ: đọc snapshot DB2 hiện có.
Database 2 raw AB2C2 C=P/N pool: 285
Duplicated material_id: 0
Duplicated formula_pretty: 135
Số mẫu có energy_above_hull không hữu hạn: 0
Tập material_id giữa CSV và cache: khớp.
Raw CSV: D:\TB3\processed\paired_training\paper_outputs\external_pools\database2_ab2c2_C_PN\pool_ab2c2_C_PN_bandgap_gt0_raw.csv
Structure cache: D:\TB3\processed\paired_training\paper_outputs\external_pools\database2_ab2c2_C_PN\pool_ab2c2_C_PN_bandgap_gt0_raw_with_structures.pkl
Không có file nào được tạo hoặc ghi đè.


,material_id,formula_pretty,band_gap,energy_above_hull,is_stable
0,mp-bgrvq,Ag2CN2,1.0037,0.268040,False
1,mp-crxni,B2CN2,1.0154,0.462611,False
2,mp-tdf,Ba(BeN)2,1.1790,0.000000,True
3,mp-mgl,Ba(CdP)2,0.6823,0.000000,True
4,mp-ckxsa,Ba(GaP)2,1.3570,0.000000,True
5,mp-bpdp,Ba(GeP)2,0.5244,0.000000,True
6,mp-ckyae,Ba(InP)2,0.2509,0.000000,True
7,mp-mgk,Ba(MgP)2,1.1366,0.000000,True
8,mp-cswwy,Ba(ZnN)2,0.5185,0.182978,False
9,mp-cszlu,Ba(ZnN)2,0.5057,0.207507,False


## Cell 11. Filter Database 2 DFT-ready pool

DFT-ready condition: `0.30 <= band_gap <= 4.50 eV` and `energy_above_hull <= 0.10 eV/atom`.

In [13]:
# Cell 11. Filter Database 2 DFT-ready AB2C2 pool

from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

ROOT = Path(r"D:\TB3\processed\paired_training")
DB2_POOL_DIR = ROOT / "paper_outputs" / "external_pools" / "database2_ab2c2_C_PN"

DB2_RAW_PKL = DB2_POOL_DIR / "pool_ab2c2_C_PN_bandgap_gt0_raw_with_structures.pkl"
DB2_DFT_READY_CSV = DB2_POOL_DIR / "pool_ab2c2_C_PN_DFT_ready.csv"
DB2_DFT_READY_PKL = DB2_POOL_DIR / "pool_ab2c2_C_PN_DFT_ready_with_structures.pkl"
DB2_SUMMARY_CSV = DB2_POOL_DIR / "database2_ab2c2_C_PN_pool_summary.csv"

if not DB2_RAW_PKL.exists():
    raise FileNotFoundError(f"Missing Database 2 raw structure cache:\n{DB2_RAW_PKL}")

db2_raw_pool = pd.read_pickle(DB2_RAW_PKL)

db2_raw_pool["band_gap"] = pd.to_numeric(db2_raw_pool["band_gap"], errors="coerce")
db2_raw_pool["energy_above_hull"] = pd.to_numeric(db2_raw_pool["energy_above_hull"], errors="coerce")
db2_raw_pool["is_stable_bool"] = (
    db2_raw_pool["is_stable"]
    .astype(str)
    .str.lower()
    .isin(["true", "1", "yes"])
)

db2_dft_ready_pool = db2_raw_pool[
    (db2_raw_pool["band_gap"] >= 0.30) &
    (db2_raw_pool["band_gap"] <= 4.50) &
    (db2_raw_pool["energy_above_hull"] <= 0.10)
].copy()

db2_dft_ready_pool = db2_dft_ready_pool.sort_values(
    ["is_stable_bool", "energy_above_hull", "band_gap"],
    ascending=[False, True, True],
).reset_index(drop=True)

db2_dft_ready_pool.drop(columns=["structure"], errors="ignore").to_csv(
    DB2_DFT_READY_CSV,
    index=False,
    encoding="utf-8-sig",
)

db2_dft_ready_pool.to_pickle(DB2_DFT_READY_PKL)

def summarize_pool(df, stage, condition, output_file):
    return {
        "database": "Database 2",
        "stage": stage,
        "n_materials": len(df),
        "n_stable": int(df["is_stable_bool"].sum()),
        "n_ehull_le_0.10": int((df["energy_above_hull"] <= 0.10).sum()),
        "n_bg_0.30_4.50": int(((df["band_gap"] >= 0.30) & (df["band_gap"] <= 4.50)).sum()),
        "duplicated_material_id": int(df["material_id"].duplicated().sum()),
        "duplicated_formula_pretty": int(df["formula_pretty"].duplicated().sum()),
        "condition": condition,
        "output_file": str(output_file),
    }

db2_summary = pd.DataFrame([
    summarize_pool(
        db2_raw_pool,
        "Raw AB2C2 C=P/N pool",
        "AB2C2 reduced formula with P2 or N2; band_gap > 0",
        DB2_RAW_PKL,
    ),
    summarize_pool(
        db2_dft_ready_pool,
        "DFT-ready AB2C2 C=P/N pool",
        "0.30 <= band_gap <= 4.50 eV; energy_above_hull <= 0.10 eV/atom",
        DB2_DFT_READY_PKL,
    ),
])

db2_summary.to_csv(DB2_SUMMARY_CSV, index=False, encoding="utf-8-sig")

print("Database 2 raw pool:", len(db2_raw_pool))
print("Database 2 DFT-ready pool:", len(db2_dft_ready_pool))
print("Stable DFT-ready candidates:", int(db2_dft_ready_pool["is_stable_bool"].sum()))
print("Saved DFT-ready CSV:", DB2_DFT_READY_CSV)
print("Saved DFT-ready structure cache:", DB2_DFT_READY_PKL)
print("Saved summary:", DB2_SUMMARY_CSV)

display(db2_summary)
display(db2_dft_ready_pool.drop(columns=["structure"], errors="ignore").head(30))


Database 2 raw pool: 285
Database 2 DFT-ready pool: 72
Stable DFT-ready candidates: 59
Saved DFT-ready CSV: D:\TB3\processed\paired_training\paper_outputs\external_pools\database2_ab2c2_C_PN\pool_ab2c2_C_PN_DFT_ready.csv
Saved DFT-ready structure cache: D:\TB3\processed\paired_training\paper_outputs\external_pools\database2_ab2c2_C_PN\pool_ab2c2_C_PN_DFT_ready_with_structures.pkl
Saved summary: D:\TB3\processed\paired_training\paper_outputs\external_pools\database2_ab2c2_C_PN\database2_ab2c2_C_PN_pool_summary.csv


,database,stage,n_materials,n_stable,n_ehull_le_0.10,n_bg_0.30_4.50,duplicated_material_id,duplicated_formula_pretty,condition,output_file
0,Database 2,Raw AB2C2 C=P/N pool,285,67,83,234,0,135,AB2C2 reduced formula with P2 or N2; band_gap > 0,D:\TB3\processed\paired_training\paper_outputs...
1,Database 2,DFT-ready AB2C2 C=P/N pool,72,59,72,72,0,5,0.30 <= band_gap <= 4.50 eV; energy_above_hull...,D:\TB3\processed\paired_training\paper_outputs...


,material_id,formula_pretty,band_gap,energy_above_hull,is_stable,is_stable_bool
0,mp-bfqo,Eu(PIr)2,0.333900,0.0,True,True
1,mp-cjlfy,Zr2SeN2,0.336800,0.0,True,True
2,mp-whu,Sr(PIr)2,0.394900,0.0,True,True
3,mp-cjlhs,Ca(InP)2,0.408500,0.0,True,True
4,mp-ntz,Ba2ZnN2,0.477100,0.0,True,True
5,mp-qno,Ca(PIr)2,0.494400,0.0,True,True
6,mp-bpdp,Ba(GeP)2,0.524400,0.0,True,True
7,mp-rdn,Zr2SN2,0.560400,0.0,True,True
8,mp-mde,Ba2GeP2,0.568200,0.0,True,True
9,mp-cgorq,Hf2SeN2,0.607100,0.0,True,True


## Cell 12. Screen Database 2 DFT-ready pool with SSL-UE-GINE

This screens the full DFT-ready AB2C2 C=P/N pool before any overlap filtering.

In [14]:
# Cell 12. Screen Database 2 DFT-ready AB2C2 pool
# Direct-alpha SAE-GINE -> MP-gap cleanup -> SLME
#
# Đã tích hợp sửa alignment của Cell 12B.
# Output mới theo timestamp; không ghi đè screening cũ.
# Chưa áp dụng StructureMatcher novelty filtering.
# ============================================================
from pathlib import Path
from datetime import datetime
import sys

import numpy as np
import pandas as pd
import torch

from scipy.special import lambertw
from IPython.display import display
from pvlib import spectrum

PROJECT_ROOT = Path(r"D:\TB3")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils_tb.screening import (
    _tb3_screen_make_graph_dataset,
    _tb3_screen_predict_ensemble,
)


# ------------------------------------------------------------
# 1. Đường dẫn và checkpoint
# ------------------------------------------------------------
ROOT = PROJECT_ROOT / "processed" / "paired_training"

DB2_POOL_DIR = (
    ROOT / "paper_outputs" / "external_pools"
    / "database2_ab2c2_C_PN"
)
DB2_DFT_READY_PKL = (
    DB2_POOL_DIR / "pool_ab2c2_C_PN_DFT_ready_with_structures.pkl"
)

DB2_SCREEN_BASE_DIR = (
    ROOT / "paper_outputs"
    / "screening_SLME_direct_alpha_500nm"
    / "database2_ab2c2_C_PN"
)

model_name = (
    "epsI_alpha_log10_1p_direct_alpha_ssl_init_"
    "enhanced_19_24_d005_wd5e5_3seed"
)
CKPT_DIR = ROOT / model_name
CKPT_PATHS = [
    CKPT_DIR / f"best_optical_{model_name}_seed{seed}.pt"
    for seed in (42, 123, 2025)
]

for path in [DB2_DFT_READY_PKL, *CKPT_PATHS]:
    if not path.is_file():
        raise FileNotFoundError(f"Không tìm thấy:\n{path}")


# ------------------------------------------------------------
# 2. Kiểm tra pool
# ------------------------------------------------------------
db2_pool = pd.read_pickle(DB2_DFT_READY_PKL).copy()

required = [
    "material_id", "formula_pretty", "band_gap",
    "energy_above_hull", "structure",
]
missing = [column for column in required if column not in db2_pool]
if missing:
    raise KeyError(f"DB2 pool thiếu cột: {missing}")

if db2_pool.empty:
    raise ValueError("DB2 DFT-ready pool rỗng.")

if db2_pool["material_id"].isna().any():
    raise ValueError("Có material_id bị thiếu.")

db2_pool["material_id"] = (
    db2_pool["material_id"].astype(str).str.strip()
)
if (
    db2_pool["material_id"].eq("").any()
    or db2_pool["material_id"].duplicated().any()
):
    raise ValueError("material_id rỗng hoặc bị trùng.")

if db2_pool["structure"].isna().any():
    raise ValueError("Có mẫu thiếu structure.")

for column in ("band_gap", "energy_above_hull"):
    db2_pool[column] = pd.to_numeric(
        db2_pool[column], errors="coerce"
    )

pool_gap = db2_pool["band_gap"].to_numpy(dtype=float)
pool_hull = db2_pool["energy_above_hull"].to_numpy(dtype=float)

eligible = (
    np.isfinite(pool_gap)
    & np.isfinite(pool_hull)
    & (pool_gap >= 0.30)
    & (pool_gap <= 4.50)
    & (pool_hull <= 0.10)
)
if not eligible.all():
    raise ValueError(
        "Pool chứa mẫu không đạt điều kiện DFT-ready; kiểm tra Cell 11."
    )


# ------------------------------------------------------------
# 3. Graph và dự đoán ensemble
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 32
energy_ev = np.linspace(0.0, 20.0, 2001, dtype=np.float32)

print("DB2 Direct-alpha → SLME")
print("Device:", device)
print("Số mẫu đầu vào:", len(db2_pool))

graphs, meta_df = _tb3_screen_make_graph_dataset(
    pool_df=db2_pool,
    energy_ev=energy_ev,
    target_key="epsI_alpha_log10_1p",
)
meta_df = meta_df.reset_index(drop=True).copy()

if not graphs or len(graphs) != len(meta_df):
    raise RuntimeError("Graph rỗng hoặc số graph không khớp metadata.")

if meta_df["material_id"].isna().any():
    raise ValueError("Graph metadata có material_id bị thiếu.")

meta_df["material_id"] = (
    meta_df["material_id"].astype(str).str.strip()
)
original_ids = meta_df["material_id"].to_numpy(dtype=str)

if len(set(original_ids)) != len(original_ids):
    raise ValueError("material_id bị trùng trong graph metadata.")

# Dừng nếu graph construction làm mất mẫu; không âm thầm đổi pool.
if (
    len(graphs) != len(db2_pool)
    or set(original_ids) != set(db2_pool["material_id"])
):
    missing_graph_ids = set(db2_pool["material_id"]) - set(original_ids)
    raise RuntimeError(
        "Graph construction không giữ nguyên toàn bộ DB2 pool.\n"
        f"ID thiếu: {sorted(missing_graph_ids)}"
    )

pred_log10_1p_alpha = np.asarray(
    _tb3_screen_predict_ensemble(
        graphs=graphs,
        ckpt_paths=CKPT_PATHS,
        device=device,
        batch_size=batch_size,
    ),
    dtype=np.float64,
)

expected_shape = (len(graphs), len(energy_ev))
if pred_log10_1p_alpha.shape != expected_shape:
    raise RuntimeError(
        f"Prediction shape {pred_log10_1p_alpha.shape}, "
        f"cần {expected_shape}."
    )
if not np.isfinite(pred_log10_1p_alpha).all():
    raise ValueError("Dự đoán log-alpha chứa giá trị không hữu hạn.")

# Ensemble đã được lấy trung bình trong target/log-space.
with np.errstate(over="raise", invalid="raise"):
    alpha_raw_cm1 = np.maximum(
        np.power(10.0, pred_log10_1p_alpha) - 1.0,
        0.0,
    )

if not np.isfinite(alpha_raw_cm1).all():
    raise ValueError("Phổ alpha RAW chứa giá trị không hữu hạn.")


# ------------------------------------------------------------
# 4. MP-gap cleanup — cùng protocol với cell gốc
# ------------------------------------------------------------
gap_ev = pd.to_numeric(
    meta_df["band_gap"], errors="coerce"
).to_numpy(dtype=np.float64)

if not np.isfinite(gap_ev).all():
    raise ValueError("MP band gap trong graph metadata không hữu hạn.")

# Kiểm tra metadata vẫn gắn đúng band gap theo material_id.
expected_gap = (
    db2_pool.set_index("material_id")
    .loc[original_ids, "band_gap"]
    .to_numpy(dtype=float)
)
np.testing.assert_allclose(
    gap_ev, expected_gap, rtol=0, atol=1e-12
)

alpha_clean_cm1 = alpha_raw_cm1.copy()
alpha_clean_cm1[energy_ev[None, :] < gap_ev[:, None]] = 0.0


# ------------------------------------------------------------
# 5. ASTM G173 AM1.5G và hằng số
# ------------------------------------------------------------
Q = 1.602176634e-19
H = 6.62607015e-34
C = 2.99792458e8
KB = 1.380649e-23
HC_EV_NM = 1239.8419843320026

THICKNESS_NM = 500.0
TEMPERATURE_K = 300.0
FR = 1.0

# Hỗ trợ các phiên bản NumPy khác nhau.
integrate = (
    np.trapezoid if hasattr(np, "trapezoid") else np.trapz
)

AM15 = spectrum.get_reference_spectra()
global_column = next(
    column for column in AM15.columns
    if "global" in str(column).lower()
)

lambda_nm = np.asarray(AM15.index, dtype=np.float64)
irradiance = np.asarray(AM15[global_column], dtype=np.float64)

solar_mask = (
    np.isfinite(lambda_nm)
    & np.isfinite(irradiance)
    & (lambda_nm >= 200.0)
    & (lambda_nm <= 2500.0)
    & (irradiance >= 0.0)
)
lambda_nm = lambda_nm[solar_mask]
irradiance = irradiance[solar_mask]

solar_energy_ev = HC_EV_NM / lambda_nm
phi_lambda = irradiance / (solar_energy_ev * Q)
phi_solar_E = phi_lambda * HC_EV_NM / solar_energy_ev**2

order = np.argsort(solar_energy_ev)
phi_solar = np.interp(
    energy_ev,
    solar_energy_ev[order],
    phi_solar_E[order],
    left=0.0,
    right=0.0,
)

P_solar = float(integrate(
    (energy_ev * Q) * phi_solar,
    energy_ev,
))
if not np.isfinite(P_solar) or P_solar <= 0:
    raise ValueError("Psolar không hợp lệ.")


def blackbody_photon_radiance(energy_eV, temperature_K):
    E_J = np.asarray(energy_eV, dtype=np.float64) * Q
    result = np.zeros_like(E_J)
    x = E_J / (KB * temperature_K)
    safe = (E_J > 0.0) & (x < 700.0)

    result[safe] = (
        2.0 * E_J[safe]**2
        / (H**3 * C**2)
        / np.expm1(x[safe])
        * Q
    )
    return result


phi_bb = blackbody_photon_radiance(energy_ev, TEMPERATURE_K)


# ------------------------------------------------------------
# 6. SLME — giữ công thức của cell gốc
# ------------------------------------------------------------
def compute_slme_percent(alpha_cm1):
    alpha_cm1 = np.asarray(alpha_cm1, dtype=np.float64)
    thickness_cm = THICKNESS_NM * 1e-7

    absorptivity = np.clip(
        1.0 - np.exp(-2.0 * alpha_cm1 * thickness_cm),
        0.0,
        1.0,
    )

    jsc = Q * integrate(
        absorptivity * phi_solar[None, :],
        energy_ev,
        axis=1,
    )
    j0 = Q * np.pi / FR * integrate(
        absorptivity * phi_bb[None, :],
        energy_ev,
        axis=1,
    )

    eta = np.full(len(alpha_cm1), np.nan, dtype=np.float64)
    valid = (
        np.isfinite(jsc) & np.isfinite(j0)
        & (jsc > 0.0) & (j0 > 0.0)
    )

    ratio = jsc[valid] / j0[valid]
    vmp_dimensionless = (
        np.real(lambertw(np.e * (1.0 + ratio))) - 1.0
    )
    vmp = KB * TEMPERATURE_K / Q * vmp_dimensionless
    jmp = (
        jsc[valid]
        - j0[valid] * (np.exp(vmp_dimensionless) - 1.0)
    )
    eta[valid] = 100.0 * vmp * jmp / P_solar

    return eta, jsc, j0


slme_percent, Jsc, J0 = compute_slme_percent(alpha_clean_cm1)

finite_slme = np.isfinite(slme_percent)
if not finite_slme.all():
    bad_ids = original_ids[~finite_slme].tolist()
    raise RuntimeError(
        "Có SLME không hữu hạn; dừng trước khi lưu.\n"
        f"material_id: {bad_ids}"
    )

# 35% là ngưỡng kiểm tra của workflow gốc, không phải SQ limit.
if (slme_percent < 0).any() or (slme_percent > 35).any():
    raise RuntimeError("SLME nằm ngoài khoảng sanity check 0–35%.")


# ------------------------------------------------------------
# 7. Bảng xếp hạng
# ------------------------------------------------------------
db2_all_df = meta_df.copy()
db2_all_df["predicted_SLME_percent"] = slme_percent
db2_all_df["MP_gap_cleanup_eV"] = gap_ev
db2_all_df["predicted_alpha_raw_mean_cm1"] = alpha_raw_cm1.mean(axis=1)
db2_all_df["predicted_alpha_clean_mean_cm1"] = alpha_clean_cm1.mean(axis=1)
db2_all_df["Jsc_A_m2"] = Jsc
db2_all_df["J0_A_m2"] = J0

# Giữ cách sắp xếp của cell gốc.
db2_all_df = (
    db2_all_df
    .sort_values("predicted_SLME_percent", ascending=False)
    .reset_index(drop=True)
)
db2_all_df["SLME_rank"] = np.arange(1, len(db2_all_df) + 1)

db2_gt30_df = db2_all_df[
    db2_all_df["predicted_SLME_percent"] > 30.0
].copy()
db2_gt31_df = db2_all_df[
    db2_all_df["predicted_SLME_percent"] > 31.0
].copy()
db2_gt32_df = db2_all_df[
    db2_all_df["predicted_SLME_percent"] > 32.0
].copy()

# Đây chưa phải shortlist cuối sau StructureMatcher.
db2_raw_top50_df = db2_all_df.head(50).copy()


# ------------------------------------------------------------
# 8. FIX: ánh xạ phổ từ graph-order sang ranked-order bằng ID
# ------------------------------------------------------------
ranked_ids = db2_all_df["material_id"].to_numpy(dtype=str)

if len(set(ranked_ids)) != len(ranked_ids):
    raise ValueError("material_id bị trùng trong bảng xếp hạng.")

if set(ranked_ids) != set(original_ids):
    raise RuntimeError("Tập ID của bảng xếp hạng và graph không khớp.")

id_to_original_idx = {
    material_id: index
    for index, material_id in enumerate(original_ids)
}
rank_order = np.array(
    [id_to_original_idx[mid] for mid in ranked_ids],
    dtype=int,
)

alpha_raw_ranked = alpha_raw_cm1[rank_order]
alpha_clean_ranked = alpha_clean_cm1[rank_order]
pred_log_ranked = pred_log10_1p_alpha[rank_order]

np.testing.assert_array_equal(original_ids[rank_order], ranked_ids)
np.testing.assert_allclose(
    alpha_raw_ranked.mean(axis=1),
    db2_all_df["predicted_alpha_raw_mean_cm1"],
    rtol=1e-12, atol=1e-12,
)
np.testing.assert_allclose(
    alpha_clean_ranked.mean(axis=1),
    db2_all_df["predicted_alpha_clean_mean_cm1"],
    rtol=1e-12, atol=1e-12,
)
np.testing.assert_allclose(
    gap_ev[rank_order],
    db2_all_df["MP_gap_cleanup_eV"],
    rtol=0, atol=1e-12,
)
np.testing.assert_allclose(
    np.maximum(10.0**pred_log_ranked - 1.0, 0.0),
    alpha_raw_ranked,
    rtol=1e-12, atol=1e-12,
)

# Tính lại SLME từ chính các phổ đã xếp hạng.
check_slme, _, _ = compute_slme_percent(alpha_clean_ranked)
np.testing.assert_allclose(
    check_slme,
    db2_all_df["predicted_SLME_percent"],
    rtol=0, atol=1e-10,
)


# ------------------------------------------------------------
# 9. Output mới, không ghi đè
# ------------------------------------------------------------
DB2_SCREEN_RUN_DIR = (
    DB2_SCREEN_BASE_DIR
    / f"run_{datetime.now():%Y%m%d_%H%M%S_%f}"
)
DB2_SCREEN_RUN_DIR.mkdir(parents=True, exist_ok=False)
SCREEN_DIR = DB2_SCREEN_RUN_DIR

prefix = "database2_ab2c2_DFT_ready_direct_alpha_SLME"
ALL_CSV = SCREEN_DIR / f"{prefix}_all.csv"
GT30_CSV = SCREEN_DIR / f"{prefix}_gt30.csv"
GT31_CSV = SCREEN_DIR / f"{prefix}_gt31.csv"
GT32_CSV = SCREEN_DIR / f"{prefix}_gt32.csv"
RAW_TOP50_CSV = SCREEN_DIR / f"{prefix}_raw_top50.csv"
SPECTRA_NPZ = SCREEN_DIR / f"{prefix}_spectra.npz"
SUMMARY_CSV = SCREEN_DIR / f"{prefix}_summary.csv"

summary_df = pd.DataFrame([
    {
        "database": "Database 2",
        "stage": stage,
        "count": count,
    }
    for stage, count in [
        ("DFT-ready AB2C2 C=P/N input", len(db2_pool)),
        ("Valid enhanced 19/24 Direct-alpha graphs", len(graphs)),
        ("All Direct-alpha SLME screened", len(db2_all_df)),
        ("Predicted SLME > 30%", len(db2_gt30_df)),
        ("Predicted SLME > 31%", len(db2_gt31_df)),
        ("Predicted SLME > 32%", len(db2_gt32_df)),
        ("Raw top-50 before structure-novelty audit", len(db2_raw_top50_df)),
    ]
])

for frame, path in [
    (db2_all_df, ALL_CSV),
    (db2_gt30_df, GT30_CSV),
    (db2_gt31_df, GT31_CSV),
    (db2_gt32_df, GT32_CSV),
    (db2_raw_top50_df, RAW_TOP50_CSV),
    (summary_df, SUMMARY_CSV),
]:
    frame.to_csv(path, index=False, encoding="utf-8-sig")

np.savez_compressed(
    SPECTRA_NPZ,
    material_id=ranked_ids,
    energy_eV=energy_ev,
    predicted_log10_1p_alpha=pred_log_ranked,
    predicted_alpha_raw_cm1=alpha_raw_ranked,
    predicted_alpha_clean_cm1=alpha_clean_ranked,
    MP_gap_cleanup_eV=db2_all_df["MP_gap_cleanup_eV"].to_numpy(float),
    predicted_SLME_percent=db2_all_df["predicted_SLME_percent"].to_numpy(float),
    SLME_rank=db2_all_df["SLME_rank"].to_numpy(int),
)


# ------------------------------------------------------------
# 10. Đọc lại file và kiểm tra CSV ↔ NPZ
# ------------------------------------------------------------
saved_table = pd.read_csv(ALL_CSV)

with np.load(SPECTRA_NPZ, allow_pickle=False) as saved:
    np.testing.assert_array_equal(
        saved["material_id"],
        saved_table["material_id"].astype(str).to_numpy(),
    )
    np.testing.assert_array_equal(
        saved["SLME_rank"],
        saved_table["SLME_rank"].to_numpy(int),
    )
    np.testing.assert_array_equal(
        saved["predicted_alpha_raw_cm1"], alpha_raw_ranked
    )
    np.testing.assert_array_equal(
        saved["predicted_alpha_clean_cm1"], alpha_clean_ranked
    )
    np.testing.assert_allclose(
        saved["predicted_SLME_percent"],
        saved_table["predicted_SLME_percent"].to_numpy(float),
        rtol=0, atol=1e-10,
    )


# ------------------------------------------------------------
# 11. Báo cáo
# ------------------------------------------------------------
print("\nDB2 CSV ↔ NPZ alignment: PASS")
print(f"Psolar: {P_solar:.6f} W/m²")
print(
    f"SLME: {slme_percent.min():.6f}"
    f"–{slme_percent.max():.6f}%"
)
print("Số cặp ID–phổ đã kiểm tra:", len(ranked_ids))
print("Thư mục kết quả mới:", DB2_SCREEN_RUN_DIR)
print("Bảng screening:", ALL_CSV)
print("Phổ đã ghép đúng ID:", SPECTRA_NPZ)
print("Không cần chạy Cell 12B cũ.")

display(summary_df)

display_columns = [
    "SLME_rank", "material_id", "formula_pretty",
    "band_gap", "energy_above_hull", "is_stable",
    "predicted_SLME_percent",
]
display(
    db2_raw_top50_df[
        [c for c in display_columns if c in db2_raw_top50_df.columns]
    ]
)

DB2 Direct-alpha → SLME
Device: cuda
Số mẫu đầu vào: 72


D:\TB3\utils_tb\features.py:932: UserWarning: No data available for atomic_radius_calculated for Ce
  atomic_radius_calculated, _ = clip_scale(getattr(element, "atomic_radius_calculated", np.nan), 3.5)
D:\TB3\utils_tb\features.py:932: UserWarning: No data available for atomic_radius_calculated for Th
  atomic_radius_calculated, _ = clip_scale(getattr(element, "atomic_radius_calculated", np.nan), 3.5)


Valid graphs for epsI_alpha_log10_1p: 72 / 72
Failed structures for epsI_alpha_log10_1p: 0
Predicting with: D:\TB3\processed\paired_training\epsI_alpha_log10_1p_direct_alpha_ssl_init_enhanced_19_24_d005_wd5e5_3seed\best_optical_epsI_alpha_log10_1p_direct_alpha_ssl_init_enhanced_19_24_d005_wd5e5_3seed_seed42.pt
Predicting with: D:\TB3\processed\paired_training\epsI_alpha_log10_1p_direct_alpha_ssl_init_enhanced_19_24_d005_wd5e5_3seed\best_optical_epsI_alpha_log10_1p_direct_alpha_ssl_init_enhanced_19_24_d005_wd5e5_3seed_seed123.pt
Predicting with: D:\TB3\processed\paired_training\epsI_alpha_log10_1p_direct_alpha_ssl_init_enhanced_19_24_d005_wd5e5_3seed\best_optical_epsI_alpha_log10_1p_direct_alpha_ssl_init_enhanced_19_24_d005_wd5e5_3seed_seed2025.pt

DB2 CSV ↔ NPZ alignment: PASS
Psolar: 994.139911 W/m²
SLME: 0.002096–33.666809%
Số cặp ID–phổ đã kiểm tra: 72
Thư mục kết quả mới: D:\TB3\processed\paired_training\paper_outputs\screening_SLME_direct_alpha_500nm\database2_ab2c2_C_PN\run_20260

,database,stage,count
0,Database 2,DFT-ready AB2C2 C=P/N input,72
1,Database 2,Valid enhanced 19/24 Direct-alpha graphs,72
2,Database 2,All Direct-alpha SLME screened,72
3,Database 2,Predicted SLME > 30%,17
4,Database 2,Predicted SLME > 31%,13
5,Database 2,Predicted SLME > 32%,12
6,Database 2,Raw top-50 before structure-novelty audit,50


,SLME_rank,material_id,formula_pretty,band_gap,energy_above_hull,is_stable,predicted_SLME_percent
0,1,mp-lmu,Th2SeN2,1.357700,0.000000,True,33.666809
1,2,mp-ckxsa,Ba(GaP)2,1.357000,0.000000,True,33.607407
2,3,mp-lcr,K2P2Pd,1.204200,0.000000,True,33.163376
3,4,mp-cjkam,Ca2CdP2,1.292700,0.000000,True,33.160749
4,5,mp-lmv,Th2SN2,1.424000,0.000000,True,33.067132
5,6,mp-cqppq,Rb2P2Pd,1.265319,0.000000,True,32.942919
6,7,mp-wuz,K2GaP2,1.229200,0.000000,True,32.795662
7,8,mp-csyvr,Ca2MgN2,1.331700,0.056386,False,32.708407
8,9,mp-crnwz,Mg2ZnP2,1.475500,0.000000,True,32.439985
9,10,mp-met,K2SiP2,1.208700,0.000000,True,32.416114


## Cell 13. Discovery-first Top-50 Database 2 and overlap check

Raw top-50 is kept first. Overlap with the original fixed train/val/test split is checked only after ranking.

In [16]:
# Cell 13. Database 2 structure-novelty audit
#
# Workflow:
#   Direct-alpha -> SLME -> StructureMatcher novelty audit
#
# Primary shortlist:
#   predicted SLME > 30%
#   AND no structure match in the full original database
#   AND no unresolved StructureMatcher error
#
# Reference hierarchy:
#   1. Training split
#   2. Validation/test remainder
#   3. Full-original database entries outside the paired split
#
# This cell reads the screening output from the new Cell 12.
# Existing screening and audit files are not overwritten.
# ============================================================

from pathlib import Path
from datetime import datetime
from collections import defaultdict
import sys
import json

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

from pymatgen.core import Structure
from pymatgen.analysis.structure_matcher import StructureMatcher


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------
PROJECT_ROOT = Path(r"D:\TB3")
ROOT = PROJECT_ROOT / "processed" / "paired_training"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils_tb.data import CrystalOpticalDataset

if "DB2_SCREEN_RUN_DIR" not in globals():
    raise RuntimeError(
        "DB2_SCREEN_RUN_DIR is not defined. Run the updated Cell 12 "
        "first, or explicitly set DB2_SCREEN_RUN_DIR to the exact "
        "screening run directory to audit."
    )

SCREEN_DIR = Path(DB2_SCREEN_RUN_DIR)
SCREEN_ALL_CSV = (
    SCREEN_DIR
    / "database2_ab2c2_DFT_ready_direct_alpha_SLME_all.csv"
)
SCREEN_SPECTRA_NPZ = (
    SCREEN_DIR
    / "database2_ab2c2_DFT_ready_direct_alpha_SLME_spectra.npz"
)

DB2_POOL_DIR = (
    ROOT / "paper_outputs" / "external_pools"
    / "database2_ab2c2_C_PN"
)
DB2_DFT_READY_PKL = (
    DB2_POOL_DIR
    / "pool_ab2c2_C_PN_DFT_ready_with_structures.pkl"
)

FIXED_SPLIT_CSV = (
    PROJECT_ROOT / "processed" / "fixed_splits"
    / "fixed_split_paired_epsR_epsI_seed2025_v1.csv"
)
DATA_ZIP = PROJECT_ROOT / "database_300.zip"

AUDIT_BASE_DIR = (
    ROOT / "paper_outputs" / "dft_validation_shortlist"
    / "discovery_first_ab2c2_database2_SLME"
)

for path in (
    SCREEN_ALL_CSV,
    SCREEN_SPECTRA_NPZ,
    DB2_DFT_READY_PKL,
    FIXED_SPLIT_CSV,
    DATA_ZIP,
):
    if not path.is_file():
        raise FileNotFoundError(f"Required input not found:\n{path}")


# ------------------------------------------------------------
# 2. Load the selected screening run and structure cache
# ------------------------------------------------------------
# Always read the selected CSV instead of a potentially stale
# db2_all_df variable remaining in the notebook kernel.
ranked_df = pd.read_csv(SCREEN_ALL_CSV)
pool_df = pd.read_pickle(DB2_DFT_READY_PKL).copy()
split_df = pd.read_csv(FIXED_SPLIT_CSV)

required_screening = [
    "material_id",
    "formula_pretty",
    "predicted_SLME_percent",
    "MP_gap_cleanup_eV",
    "SLME_rank",
]
missing = [c for c in required_screening if c not in ranked_df]
if missing:
    raise KeyError(f"Screening CSV is missing columns: {missing}")

required_pool = ["material_id", "structure", "band_gap"]
missing = [c for c in required_pool if c not in pool_df]
if missing:
    raise KeyError(f"Structure cache is missing columns: {missing}")


def normalize_unique_ids(frame, label):
    if frame["material_id"].isna().any():
        raise ValueError(f"{label}: missing material_id.")

    frame["material_id"] = (
        frame["material_id"].astype(str).str.strip()
    )
    if frame["material_id"].eq("").any():
        raise ValueError(f"{label}: empty material_id.")

    if frame["material_id"].duplicated().any():
        raise ValueError(f"{label}: duplicated material_id.")


normalize_unique_ids(ranked_df, "Screening CSV")
normalize_unique_ids(pool_df, "Structure cache")

if ranked_df.empty:
    raise ValueError("The screening table is empty.")

if set(ranked_df["material_id"]) != set(pool_df["material_id"]):
    raise ValueError(
        "Screening CSV and structure cache contain different ID sets."
    )

for column in ("predicted_SLME_percent", "MP_gap_cleanup_eV"):
    ranked_df[column] = pd.to_numeric(
        ranked_df[column], errors="raise"
    )
    if not np.isfinite(ranked_df[column].to_numpy(float)).all():
        raise ValueError(f"Non-finite values in {column}.")

if ranked_df["formula_pretty"].isna().any():
    raise ValueError("Missing formula_pretty in the screening CSV.")

saved_ranks = pd.to_numeric(
    ranked_df["SLME_rank"], errors="raise"
).to_numpy(float)

if not np.array_equal(
    np.sort(saved_ranks),
    np.arange(1, len(ranked_df) + 1),
):
    raise ValueError("SLME_rank must contain each integer from 1 to N.")

# Preserve Cell 12 ranking, including the order of tied scores.
ranked_df["SLME_rank"] = saved_ranks.astype(int)
ranked_df = (
    ranked_df.sort_values("SLME_rank")
    .reset_index(drop=True)
)

if not ranked_df["predicted_SLME_percent"].is_monotonic_decreasing:
    raise ValueError("SLME values are inconsistent with SLME_rank.")

pool_gap = pd.to_numeric(
    pool_df.set_index("material_id")
    .loc[ranked_df["material_id"], "band_gap"],
    errors="raise",
).to_numpy(float)

np.testing.assert_allclose(
    ranked_df["MP_gap_cleanup_eV"].to_numpy(float),
    pool_gap,
    rtol=0,
    atol=1e-12,
    err_msg="Screening gaps do not match the selected structure cache.",
)

# Verify that CSV and NPZ belong to the same ranked screening run.
with np.load(SCREEN_SPECTRA_NPZ, allow_pickle=False) as spectra:
    np.testing.assert_array_equal(
        spectra["material_id"].astype(str),
        ranked_df["material_id"].to_numpy(dtype=str),
    )
    np.testing.assert_array_equal(
        spectra["SLME_rank"],
        ranked_df["SLME_rank"].to_numpy(int),
    )
    np.testing.assert_allclose(
        spectra["predicted_SLME_percent"],
        ranked_df["predicted_SLME_percent"].to_numpy(float),
        rtol=0,
        atol=1e-10,
    )
    np.testing.assert_allclose(
        spectra["MP_gap_cleanup_eV"],
        pool_gap,
        rtol=0,
        atol=1e-12,
    )

print("Screening source:", SCREEN_ALL_CSV)
print("Screening CSV/NPZ consistency: PASS")
print("Candidates to audit:", len(ranked_df))


# ------------------------------------------------------------
# 3. Load the original database explicitly
# ------------------------------------------------------------
# Do not reuse a dataset or get_structure function from globals.
# This prevents accidental dependence on another notebook run.
print("\nLoading the full original database:", DATA_ZIP)

base_dataset = CrystalOpticalDataset(
    zip_path=DATA_ZIP,
    require_any_target=True,
    require_full_spectrum=False,
    verbose=False,
)

N_FULL = len(base_dataset)
if N_FULL != 10189:
    raise RuntimeError(
        "Original database size mismatch: "
        f"expected 10189 valid structures, found {N_FULL}."
    )


def to_structure(value):
    if isinstance(value, Structure):
        return value
    if isinstance(value, dict):
        return Structure.from_dict(value)
    if isinstance(value, str):
        return Structure.from_dict(json.loads(value))
    raise TypeError(f"Unsupported structure type: {type(value)}")


# Build metadata directly from the loaded dataset.
# In utils_tb.data, identifiers are stored in sample["meta"].
original_structures = {}
full_meta_rows = []

for base_idx in tqdm(
    range(N_FULL),
    desc="Reading original structures",
):
    item = base_dataset.get_full_item(base_idx)
    structure = to_structure(item["structure"])
    metadata = item.get("meta", {})

    original_structures[base_idx] = structure
    full_meta_rows.append({
        "base_idx": base_idx,
        "mat_id": metadata.get("mat_id"),
        "file_name": metadata.get("file_name"),
        "reduced_formula": structure.composition.reduced_formula,
    })

full_meta_df = pd.DataFrame(full_meta_rows)
full_meta_lookup = full_meta_df.set_index("base_idx")


# ------------------------------------------------------------
# 4. Validate fixed-split provenance
# ------------------------------------------------------------
required_split = ["base_idx", "split", "mat_id", "file_name"]
missing = [c for c in required_split if c not in split_df]
if missing:
    raise KeyError(f"Fixed split is missing columns: {missing}")

base_indices = pd.to_numeric(
    split_df["base_idx"], errors="raise"
).to_numpy(float)

if (
    not np.isfinite(base_indices).all()
    or not np.equal(base_indices, np.floor(base_indices)).all()
):
    raise ValueError("Fixed-split base_idx contains invalid indices.")

split_df["base_idx"] = base_indices.astype(int)

if split_df["base_idx"].duplicated().any():
    raise ValueError("Fixed split contains duplicated base_idx.")

if not split_df["base_idx"].between(0, N_FULL - 1).all():
    raise ValueError("Fixed-split base_idx is outside the database.")

split_df["split"] = (
    split_df["split"].astype(str).str.strip().str.lower()
)

expected_split_counts = {
    "train": 7931,
    "val": 992,
    "test": 992,
}
actual_split_counts = split_df["split"].value_counts().to_dict()

if actual_split_counts != expected_split_counts:
    raise ValueError(
        "Unexpected fixed-split counts.\n"
        f"Expected: {expected_split_counts}\n"
        f"Found: {actual_split_counts}"
    )

# Check the actual identity behind every base_idx, not only counts.
for column in ("mat_id", "file_name"):
    expected = full_meta_lookup.loc[
        split_df["base_idx"], column
    ].reset_index(drop=True)
    observed = split_df[column].reset_index(drop=True)

    if expected.isna().any() or observed.isna().any():
        raise ValueError(f"Missing {column} during split validation.")

    if not np.array_equal(
        expected.astype(str).str.strip().to_numpy(),
        observed.astype(str).str.strip().to_numpy(),
    ):
        raise ValueError(
            f"Fixed-split {column} does not match database base_idx."
        )

# Use formulas derived from the actual reference structures.
split_df["reduced_formula"] = (
    split_df["base_idx"]
    .map(full_meta_lookup["reduced_formula"])
)

paired_base_indices = set(split_df["base_idx"])
full_meta_df["in_paired_optical_db"] = (
    full_meta_df["base_idx"].isin(paired_base_indices)
)
split_lookup = split_df.set_index("base_idx")

print("Original database / fixed-split identity check: PASS")
print("Full original structures:", N_FULL)
print("Paired optical structures:", len(split_df))
print("Full-only reference structures:", N_FULL - len(split_df))


# ------------------------------------------------------------
# 5. Candidate structures
# ------------------------------------------------------------
candidate_structure_map = {}
candidate_formula_map = {}

for _, row in pool_df.iterrows():
    mid = row["material_id"]
    try:
        structure = to_structure(row["structure"])
    except Exception as exc:
        raise RuntimeError(
            f"Cannot read candidate structure for {mid}."
        ) from exc

    candidate_structure_map[mid] = structure
    candidate_formula_map[mid] = structure.composition.reduced_formula

ranked_df["structure_reduced_formula"] = (
    ranked_df["material_id"].map(candidate_formula_map)
)


# ------------------------------------------------------------
# 6. Reference groups and matcher settings
# ------------------------------------------------------------
train_df = split_df[split_df["split"].eq("train")].copy()
nontrain_paired_df = split_df[
    split_df["split"].isin(["val", "test"])
].copy()
full_extra_df = full_meta_df[
    ~full_meta_df["in_paired_optical_db"]
].copy()


def build_formula_groups(frame):
    groups = defaultdict(list)
    for row in frame.itertuples(index=False):
        groups[row.reduced_formula].append(int(row.base_idx))
    return groups


train_groups = build_formula_groups(train_df)
nontrain_groups = build_formula_groups(nontrain_paired_df)
full_extra_groups = build_formula_groups(full_extra_df)

MATCHER_CONFIG = {
    "ltol": 0.20,
    "stol": 0.30,
    "angle_tol": 5.0,
    "primitive_cell": True,
    "scale": True,
    "attempt_supercell": True,
}
matcher = StructureMatcher(**MATCHER_CONFIG)

match_records = []
match_errors = []


def reference_info(base_idx):
    info = full_meta_lookup.loc[base_idx]
    reference_split = (
        split_lookup.loc[base_idx, "split"]
        if base_idx in split_lookup.index
        else "full_only_extra"
    )
    return {
        "reference_base_idx": int(base_idx),
        "reference_mat_id": info["mat_id"],
        "reference_formula": info["reduced_formula"],
        "reference_split": reference_split,
    }


def match_against_group(
    query_mid,
    query_structure,
    query_formula,
    reference_indices,
    reference_set_name,
):
    for ref_idx in reference_indices:
        try:
            is_match = matcher.fit(
                original_structures[ref_idx],
                query_structure,
            )
        except Exception as exc:
            match_errors.append({
                "query_material_id": query_mid,
                "reference_base_idx": int(ref_idx),
                "reference_set": reference_set_name,
                "error": str(exc),
            })
            continue

        if is_match:
            record = {
                "query_material_id": query_mid,
                "query_formula": query_formula,
                "reference_set": reference_set_name,
            }
            record.update(reference_info(ref_idx))
            match_records.append(record)
            return True, int(ref_idx)

    return False, None


# ------------------------------------------------------------
# 7. Audit every screened candidate
# ------------------------------------------------------------
audit_rows = []

for row in tqdm(
    ranked_df.itertuples(index=False),
    total=len(ranked_df),
    desc="Database 2 StructureMatcher",
):
    mid = row.material_id
    query_structure = candidate_structure_map[mid]
    query_formula = candidate_formula_map[mid]

    matched_train, ref_idx = match_against_group(
        mid, query_structure, query_formula,
        train_groups.get(query_formula, []),
        "train",
    )

    matched_paired = matched_train
    matched_full = matched_train
    final_reference_set = "train" if matched_train else None
    final_ref_idx = ref_idx if matched_train else None

    if not matched_paired:
        matched_nontrain, ref_idx = match_against_group(
            mid, query_structure, query_formula,
            nontrain_groups.get(query_formula, []),
            "paired_val_test",
        )
        if matched_nontrain:
            matched_paired = True
            matched_full = True
            final_reference_set = "paired_val_test"
            final_ref_idx = ref_idx

    if not matched_full:
        matched_extra, ref_idx = match_against_group(
            mid, query_structure, query_formula,
            full_extra_groups.get(query_formula, []),
            "full_only_extra",
        )
        if matched_extra:
            matched_full = True
            final_reference_set = "full_only_extra"
            final_ref_idx = ref_idx

    audit_rows.append({
        "material_id": mid,
        "structure_match_train": bool(matched_train),
        "structure_match_paired_optical_db": bool(matched_paired),
        "structure_match_full_original_db": bool(matched_full),
        "matched_reference_set": final_reference_set,
        "matched_reference_base_idx": final_ref_idx,
    })

audit_df = pd.DataFrame(audit_rows)
ranked_df = ranked_df.merge(
    audit_df,
    on="material_id",
    how="left",
    validate="one_to_one",
    sort=False,
).sort_values("SLME_rank").reset_index(drop=True)


# ------------------------------------------------------------
# 8. Conservative handling of matching errors
# ------------------------------------------------------------
matcher_error_ids = {
    record["query_material_id"] for record in match_errors
}

ranked_df["structure_matcher_had_error"] = (
    ranked_df["material_id"].isin(matcher_error_ids)
)

ranked_df["structure_novelty_uncertain"] = (
    ~ranked_df["structure_match_full_original_db"]
    & ranked_df["structure_matcher_had_error"]
)

# A failed comparison must not be interpreted as evidence of novelty.
ranked_df["structure_new_vs_full_original_db"] = (
    ~ranked_df["structure_match_full_original_db"]
    & ~ranked_df["structure_matcher_had_error"]
)

ranked_df["structure_novelty_status"] = np.select(
    [
        ranked_df["structure_match_full_original_db"],
        ranked_df["structure_novelty_uncertain"],
    ],
    [
        "matched_full_original_db",
        "uncertain_matcher_error",
    ],
    default="no_match_found_under_protocol",
)

# ID overlap is descriptive only. Structural matching determines
# the novelty classification.
known_full_ids = set(
    full_meta_df["mat_id"].dropna().astype(str).str.strip()
)
ranked_df["material_id_in_full_original_db"] = (
    ranked_df["material_id"].isin(known_full_ids)
)


# ------------------------------------------------------------
# 9. Final subsets
# ------------------------------------------------------------
gt30 = ranked_df[
    ranked_df["predicted_SLME_percent"] > 30.0
].copy()
gt31 = ranked_df[
    ranked_df["predicted_SLME_percent"] > 31.0
].copy()
gt32 = ranked_df[
    ranked_df["predicted_SLME_percent"] > 32.0
].copy()

gt30_new = gt30[
    gt30["structure_new_vs_full_original_db"]
].copy()
gt31_new = gt31[
    gt31["structure_new_vs_full_original_db"]
].copy()
gt32_new = gt32[
    gt32["structure_new_vs_full_original_db"]
].copy()

raw_top50 = ranked_df.head(50).copy()

# Secondary list: it may contain SLME <= 30%.
# It is not the primary DFT shortlist.
top50_structure_new = ranked_df[
    ranked_df["structure_new_vs_full_original_db"]
].head(50).copy()

uncertain_df = ranked_df[
    ranked_df["structure_novelty_uncertain"]
].copy()

# Define columns explicitly so empty audit files still have headers.
matches_df = pd.DataFrame(
    match_records,
    columns=[
        "query_material_id",
        "query_formula",
        "reference_set",
        "reference_base_idx",
        "reference_mat_id",
        "reference_formula",
        "reference_split",
    ],
)
errors_df = pd.DataFrame(
    match_errors,
    columns=[
        "query_material_id",
        "reference_base_idx",
        "reference_set",
        "error",
    ],
)


# ------------------------------------------------------------
# 10. Summary
# ------------------------------------------------------------
summary_values = [
    ("all_Database2_DFT_ready", len(ranked_df)),
    ("SLME_gt30", len(gt30)),
    ("SLME_gt30_match_train", int(gt30["structure_match_train"].sum())),
    (
        "SLME_gt30_match_paired_optical_db",
        int(gt30["structure_match_paired_optical_db"].sum()),
    ),
    (
        "SLME_gt30_match_full_original_db",
        int(gt30["structure_match_full_original_db"].sum()),
    ),
    ("SLME_gt30_structure_new_for_DFT", len(gt30_new)),
    ("SLME_gt31_structure_new_for_DFT", len(gt31_new)),
    ("SLME_gt32_structure_new_for_DFT", len(gt32_new)),
    ("raw_top50_FINAL_SLME", len(raw_top50)),
    ("structure_new_top50_secondary", len(top50_structure_new)),
    ("StructureMatcher_errors", len(errors_df)),
    ("all_structure_novelty_uncertain", len(uncertain_df)),
    (
        "SLME_gt30_structure_novelty_uncertain",
        int(gt30["structure_novelty_uncertain"].sum()),
    ),
]
summary_df = pd.DataFrame(summary_values, columns=["item", "count"])


# ------------------------------------------------------------
# 11. Save to a new audit directory
# ------------------------------------------------------------
DB2_DISCOVERY_RUN_DIR = (
    AUDIT_BASE_DIR
    / f"audit_{datetime.now():%Y%m%d_%H%M%S_%f}"
)
OUT_DIR = DB2_DISCOVERY_RUN_DIR
OUT_DIR.mkdir(parents=True, exist_ok=False)

ALL_FINAL_CSV = OUT_DIR / "ab2c2_all_FINAL_SLME_with_structure_overlap_flags.csv"
GT30_AUDIT_CSV = OUT_DIR / "ab2c2_SLME_gt30_structure_overlap_audit.csv"
GT30_NEW_CSV = OUT_DIR / "ab2c2_SLME_gt30_new_for_DFT.csv"
GT31_NEW_CSV = OUT_DIR / "ab2c2_SLME_gt31_new_for_DFT.csv"
GT32_NEW_CSV = OUT_DIR / "ab2c2_SLME_gt32_new_for_DFT.csv"
RAW_TOP50_CSV = OUT_DIR / "ab2c2_raw_top50_FINAL_SLME.csv"
TOP50_NEW_SECONDARY_CSV = (
    OUT_DIR / "ab2c2_top50_structure_new_FINAL_SLME_secondary.csv"
)
MATCHES_CSV = OUT_DIR / "ab2c2_structure_matcher_matches.csv"
ERRORS_CSV = OUT_DIR / "ab2c2_structure_matcher_errors.csv"
UNCERTAIN_CSV = OUT_DIR / "ab2c2_structure_novelty_uncertain.csv"
SUMMARY_CSV = OUT_DIR / "ab2c2_FINAL_SLME_structure_novelty_summary.csv"

for frame, path in [
    (ranked_df, ALL_FINAL_CSV),
    (gt30, GT30_AUDIT_CSV),
    (gt30_new, GT30_NEW_CSV),
    (gt31_new, GT31_NEW_CSV),
    (gt32_new, GT32_NEW_CSV),
    (raw_top50, RAW_TOP50_CSV),
    (top50_structure_new, TOP50_NEW_SECONDARY_CSV),
    (matches_df, MATCHES_CSV),
    (errors_df, ERRORS_CSV),
    (uncertain_df, UNCERTAIN_CSV),
    (summary_df, SUMMARY_CSV),
]:
    frame.to_csv(path, index=False, encoding="utf-8-sig")

provenance = {
    "created_at": datetime.now().astimezone().isoformat(),
    "screening_csv": str(SCREEN_ALL_CSV.resolve()),
    "screening_npz": str(SCREEN_SPECTRA_NPZ.resolve()),
    "structure_pool": str(DB2_DFT_READY_PKL.resolve()),
    "original_database_zip": str(DATA_ZIP.resolve()),
    "fixed_split_csv": str(FIXED_SPLIT_CSV.resolve()),
    "matcher_config": MATCHER_CONFIG,
    "n_screened": len(ranked_df),
    "n_full_original": N_FULL,
    "n_paired": len(split_df),
    "split_counts": {
        key: int(value) for key, value in actual_split_counts.items()
    },
    "n_matcher_errors": len(errors_df),
    "n_uncertain_candidates": len(uncertain_df),
    "primary_shortlist_rule": (
        "predicted_SLME_percent > 30; no structure match in the "
        "full original database; no unresolved matcher error"
    ),
    "novelty_scope": (
        "No equivalent structure found in the selected reference "
        "database under the stated StructureMatcher protocol. "
        "This does not establish novelty against all published materials."
    ),
}

(OUT_DIR / "audit_provenance.json").write_text(
    json.dumps(provenance, indent=2, ensure_ascii=False),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 12. Report
# ------------------------------------------------------------
print("\nDATABASE 2 — STRUCTURE NOVELTY AUDIT COMPLETE")
print("Screening source:", SCREEN_ALL_CSV)
print("Audit output:", OUT_DIR)
print("Primary DFT shortlist:", GT30_NEW_CSV)
print("Matcher errors:", len(errors_df))
print("Uncertain candidates excluded from new-material lists:", len(uncertain_df))
print("Existing audit files were not overwritten.")

display(summary_df)

display_columns = [
    "SLME_rank",
    "material_id",
    "formula_pretty",
    "band_gap",
    "energy_above_hull",
    "is_stable",
    "predicted_SLME_percent",
    "structure_match_train",
    "structure_match_paired_optical_db",
    "structure_match_full_original_db",
    "structure_new_vs_full_original_db",
    "structure_novelty_status",
    "matched_reference_set",
]
display_columns = [
    column for column in display_columns if column in ranked_df
]

print("\nPRIMARY SHORTLIST: SLME > 30% AND STRUCTURE-NEW")
display(gt30_new[display_columns])

print("\nSLME > 30% STRUCTURE OVERLAP AUDIT")
display(gt30[display_columns])

if not uncertain_df.empty:
    print("\nUNRESOLVED CANDIDATES — NOT CLASSIFIED AS STRUCTURE-NEW")
    display(uncertain_df[display_columns])

print("\nFor the next export cell, use:")
print("DB2_DISCOVERY_DIR = Path(DB2_DISCOVERY_RUN_DIR)")

Screening source: D:\TB3\processed\paired_training\paper_outputs\screening_SLME_direct_alpha_500nm\database2_ab2c2_C_PN\run_20260922_202059_742351\database2_ab2c2_DFT_ready_direct_alpha_SLME_all.csv
Screening CSV/NPZ consistency: PASS
Candidates to audit: 72

Loading the full original database: D:\TB3\database_300.zip


Reading original structures:   0%|          | 0/10189 [00:00<?, ?it/s]

C:\Users\ADMIN\AppData\Local\Programs\Python\Python310\lib\site-packages\pymatgen\core\composition.py:1366: UserWarning: No Pauling electronegativity for Ne. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  syms: list[str] = sorted(sym_amt, key=lambda x: [get_el_sp(x).X, x])
C:\Users\ADMIN\AppData\Local\Programs\Python\Python310\lib\site-packages\pymatgen\core\composition.py:1366: UserWarning: No Pauling electronegativity for Ar. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  syms: list[str] = sorted(sym_amt, key=lambda x: [get_el_sp(x).X, x])
C:\Users\ADMIN\AppData\Local\Programs\Python\Python310\lib\site-packages\pymatgen\core\composition.py:1366: UserWarning: No Pauling electronegativity for He. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  syms: list[str] = sorted(

Original database / fixed-split identity check: PASS
Full original structures: 10189
Paired optical structures: 9915
Full-only reference structures: 274


Database 2 StructureMatcher:   0%|          | 0/72 [00:00<?, ?it/s]

...


PRIMARY SHORTLIST: SLME > 30% AND STRUCTURE-NEW


,SLME_rank,material_id,formula_pretty,band_gap,energy_above_hull,is_stable,predicted_SLME_percent,structure_match_train,structure_match_paired_optical_db,structure_match_full_original_db,structure_new_vs_full_original_db,structure_novelty_status,matched_reference_set
0,1,mp-lmu,Th2SeN2,1.357700,0.000000,True,33.666809,False,False,False,True,no_match_found_under_protocol,None
1,2,mp-ckxsa,Ba(GaP)2,1.357000,0.000000,True,33.607407,False,False,False,True,no_match_found_under_protocol,None
2,3,mp-lcr,K2P2Pd,1.204200,0.000000,True,33.163376,False,False,False,True,no_match_found_under_protocol,None
3,4,mp-cjkam,Ca2CdP2,1.292700,0.000000,True,33.160749,False,False,False,True,no_match_found_under_protocol,None
4,5,mp-lmv,Th2SN2,1.424000,0.000000,True,33.067132,False,False,False,True,no_match_found_under_protocol,None
5,6,mp-cqppq,Rb2P2Pd,1.265319,0.000000,True,32.942919,False,False,False,True,no_match_found_under_protocol,None
6,7,mp-wuz,K2GaP2,1.229200,0.000000,True,32.795662,False,False,False,True,no_match_found_under_protocol,None
7,8,mp-csyvr,Ca2MgN2,1.331700,0.056386,False,32.708407,False,False,False,True,no_match_found_under_protocol,None
8,9,mp-crnwz,Mg2ZnP2,1.475500,0.000000,True,32.439985,False,False,False,True,no_match_found_under_protocol,None
9,10,mp-met,K2SiP2,1.208700,0.000000,True,32.416114,False,False,False,True,no_match_found_under_protocol,None



SLME > 30% STRUCTURE OVERLAP AUDIT


,SLME_rank,material_id,formula_pretty,band_gap,energy_above_hull,is_stable,predicted_SLME_percent,structure_match_train,structure_match_paired_optical_db,structure_match_full_original_db,structure_new_vs_full_original_db,structure_novelty_status,matched_reference_set
0,1,mp-lmu,Th2SeN2,1.357700,0.000000,True,33.666809,False,False,False,True,no_match_found_under_protocol,None
1,2,mp-ckxsa,Ba(GaP)2,1.357000,0.000000,True,33.607407,False,False,False,True,no_match_found_under_protocol,None
2,3,mp-lcr,K2P2Pd,1.204200,0.000000,True,33.163376,False,False,False,True,no_match_found_under_protocol,None
3,4,mp-cjkam,Ca2CdP2,1.292700,0.000000,True,33.160749,False,False,False,True,no_match_found_under_protocol,None
4,5,mp-lmv,Th2SN2,1.424000,0.000000,True,33.067132,False,False,False,True,no_match_found_under_protocol,None
5,6,mp-cqppq,Rb2P2Pd,1.265319,0.000000,True,32.942919,False,False,False,True,no_match_found_under_protocol,None
6,7,mp-wuz,K2GaP2,1.229200,0.000000,True,32.795662,False,False,False,True,no_match_found_under_protocol,None
7,8,mp-csyvr,Ca2MgN2,1.331700,0.056386,False,32.708407,False,False,False,True,no_match_found_under_protocol,None
8,9,mp-crnwz,Mg2ZnP2,1.475500,0.000000,True,32.439985,False,False,False,True,no_match_found_under_protocol,None
9,10,mp-met,K2SiP2,1.208700,0.000000,True,32.416114,False,False,False,True,no_match_found_under_protocol,None



For the next export cell, use:
DB2_DISCOVERY_DIR = Path(DB2_DISCOVERY_RUN_DIR)


## Cell 14. Export Database 2 Top-50 lists to JSON and CIF

In [18]:
# Cell 14. Export FINAL Database 2 DFT shortlist
#
# Selection:
#   predicted SLME > 30%
#   AND no structure match in the full original database
#   AND no unresolved StructureMatcher error
#
# Inputs: the audit directory produced by the updated Cell 13.
# Outputs: CIF, JSON, manifest, shortlist, provenance, and ZIP.
# Existing packages are not overwritten.
# ============================================================

from pathlib import Path
from datetime import datetime
import json
import re
import zipfile

import numpy as np
import pandas as pd

from IPython.display import display
from pymatgen.core import Structure


# ------------------------------------------------------------
# 1. Input paths
# ------------------------------------------------------------
if "DB2_DISCOVERY_RUN_DIR" not in globals():
    raise RuntimeError(
        "DB2_DISCOVERY_RUN_DIR is not defined. Run the updated "
        "Cell 13 first, or explicitly set it to the exact audit "
        "directory to export."
    )

DB2_DISCOVERY_DIR = Path(DB2_DISCOVERY_RUN_DIR)

DB2_FINAL_CSV = (
    DB2_DISCOVERY_DIR / "ab2c2_SLME_gt30_new_for_DFT.csv"
)
DB2_AUDIT_CSV = (
    DB2_DISCOVERY_DIR
    / "ab2c2_all_FINAL_SLME_with_structure_overlap_flags.csv"
)
AUDIT_PROVENANCE_JSON = DB2_DISCOVERY_DIR / "audit_provenance.json"

for path in (
    DB2_FINAL_CSV,
    DB2_AUDIT_CSV,
    AUDIT_PROVENANCE_JSON,
):
    if not path.is_file():
        raise FileNotFoundError(f"Required audit input not found:\n{path}")

audit_provenance = json.loads(
    AUDIT_PROVENANCE_JSON.read_text(encoding="utf-8")
)

# Read the exact structure-cache path recorded by Cell 13.
DB2_DFT_READY_PKL = Path(audit_provenance["structure_pool"])

if not DB2_DFT_READY_PKL.is_file():
    raise FileNotFoundError(
        f"Recorded structure cache not found:\n{DB2_DFT_READY_PKL}"
    )

final_df = pd.read_csv(DB2_FINAL_CSV)
audit_df = pd.read_csv(DB2_AUDIT_CSV)
struct_df = pd.read_pickle(DB2_DFT_READY_PKL).copy()

# Frozen manuscript snapshot check; this does not select candidates.
EXPECTED_FINAL_COUNT = 16


# ------------------------------------------------------------
# 2. Validation helpers
# ------------------------------------------------------------
BOOL_COLUMNS = [
    "structure_match_train",
    "structure_match_paired_optical_db",
    "structure_match_full_original_db",
    "structure_new_vs_full_original_db",
    "structure_matcher_had_error",
    "structure_novelty_uncertain",
]


def normalize_ids(frame, label):
    if "material_id" not in frame:
        raise KeyError(f"{label}: missing material_id column.")

    if frame["material_id"].isna().any():
        raise ValueError(f"{label}: missing material_id values.")

    frame["material_id"] = (
        frame["material_id"].astype(str).str.strip()
    )

    if frame["material_id"].eq("").any():
        raise ValueError(f"{label}: empty material_id values.")

    if frame["material_id"].duplicated().any():
        raise ValueError(f"{label}: duplicated material_id values.")


def parse_boolean_column(series, label):
    parsed = (
        series.astype(str).str.strip().str.lower()
        .map({"true": True, "false": False, "1": True, "0": False})
    )
    if parsed.isna().any():
        raise ValueError(f"Invalid or missing boolean values in {label}.")
    return parsed.astype(bool)


def validate_audit_table(frame, label):
    required = [
        "material_id",
        "formula_pretty",
        "structure_reduced_formula",
        "SLME_rank",
        "predicted_SLME_percent",
        *BOOL_COLUMNS,
    ]
    missing = [column for column in required if column not in frame]
    if missing:
        raise KeyError(f"{label}: missing columns {missing}")

    normalize_ids(frame, label)

    for column in ("formula_pretty", "structure_reduced_formula"):
        if frame[column].isna().any():
            raise ValueError(f"{label}: missing {column}.")
        frame[column] = frame[column].astype(str).str.strip()
        if frame[column].eq("").any():
            raise ValueError(f"{label}: empty {column}.")

    frame["predicted_SLME_percent"] = pd.to_numeric(
        frame["predicted_SLME_percent"], errors="raise"
    )
    if not np.isfinite(
        frame["predicted_SLME_percent"].to_numpy(float)
    ).all():
        raise ValueError(f"{label}: non-finite SLME values.")

    ranks = pd.to_numeric(
        frame["SLME_rank"], errors="raise"
    ).to_numpy(float)

    if (
        not np.isfinite(ranks).all()
        or (ranks < 1).any()
        or not np.equal(ranks, np.floor(ranks)).all()
        or len(np.unique(ranks)) != len(ranks)
    ):
        raise ValueError(f"{label}: invalid or duplicated SLME ranks.")

    frame["SLME_rank"] = ranks.astype(int)

    for column in BOOL_COLUMNS:
        frame[column] = parse_boolean_column(
            frame[column], f"{label}.{column}"
        )


def primary_selection_mask(frame):
    return (
        (frame["predicted_SLME_percent"] > 30.0)
        & frame["structure_new_vs_full_original_db"]
        & ~frame["structure_match_train"]
        & ~frame["structure_match_paired_optical_db"]
        & ~frame["structure_match_full_original_db"]
        & ~frame["structure_matcher_had_error"]
        & ~frame["structure_novelty_uncertain"]
    )


def to_structure(value):
    if isinstance(value, Structure):
        return value
    if isinstance(value, dict):
        return Structure.from_dict(value)
    if isinstance(value, str):
        return Structure.from_dict(json.loads(value))
    raise TypeError(f"Unsupported structure type: {type(value)}")


def safe_name(value):
    name = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value))
    return name.strip("._") or "unnamed"


def clean_scalar(value):
    if value is None or pd.isna(value):
        return None
    # bool must be checked before int.
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return int(value)
    if isinstance(value, (float, np.floating)):
        return float(value) if np.isfinite(value) else None
    return str(value)


def write_json_new(path, value):
    # Exclusive creation prevents accidental overwriting.
    with path.open("x", encoding="utf-8") as handle:
        json.dump(
            value,
            handle,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )


# ------------------------------------------------------------
# 3. Validate the shortlist against the complete audit
# ------------------------------------------------------------
validate_audit_table(final_df, "Final shortlist")
validate_audit_table(audit_df, "Full audit")
normalize_ids(struct_df, "Structure cache")

if final_df.empty:
    raise ValueError("The primary shortlist is empty.")

if len(final_df) != EXPECTED_FINAL_COUNT:
    raise RuntimeError(
        "Frozen manuscript snapshot count mismatch.\n"
        f"Expected: {EXPECTED_FINAL_COUNT}\n"
        f"Found: {len(final_df)}\n"
        "Inspect the audit before changing EXPECTED_FINAL_COUNT."
    )

if not primary_selection_mask(final_df).all():
    raise ValueError(
        "The shortlist contains an ineligible or unresolved candidate."
    )

expected_final = audit_df.loc[
    primary_selection_mask(audit_df)
].copy()

if set(final_df["material_id"]) != set(expected_final["material_id"]):
    raise ValueError(
        "The shortlist does not equal the eligible subset of the full audit."
    )

final_df = final_df.sort_values("SLME_rank").reset_index(drop=True)
expected_final = (
    expected_final.set_index("material_id")
    .loc[final_df["material_id"]]
)

np.testing.assert_array_equal(
    final_df["SLME_rank"].to_numpy(),
    expected_final["SLME_rank"].to_numpy(),
)
np.testing.assert_allclose(
    final_df["predicted_SLME_percent"].to_numpy(float),
    expected_final["predicted_SLME_percent"].to_numpy(float),
    rtol=0,
    atol=1e-10,
)

for column in (
    "formula_pretty",
    "structure_reduced_formula",
    *BOOL_COLUMNS,
):
    np.testing.assert_array_equal(
        final_df[column].to_numpy(),
        expected_final[column].to_numpy(),
        err_msg=f"Shortlist/full-audit mismatch in {column}.",
    )

if not final_df["predicted_SLME_percent"].is_monotonic_decreasing:
    raise ValueError("Shortlist ranking is inconsistent with SLME.")

if "structure" not in struct_df:
    raise KeyError("Structure cache is missing the structure column.")

cache_by_id = struct_df.set_index("material_id")

if not set(final_df["material_id"]).issubset(cache_by_id.index):
    raise ValueError("Some shortlist IDs are missing from the structure cache.")


# ------------------------------------------------------------
# 4. Prepare and validate structures before creating output
# ------------------------------------------------------------
prepared_structures = {}

for row in final_df.itertuples(index=False):
    structure = to_structure(
        cache_by_id.loc[row.material_id, "structure"]
    )

    if structure.composition.reduced_formula != row.structure_reduced_formula:
        raise ValueError(
            f"Structure/audit formula mismatch for {row.material_id}."
        )

    if (
        len(structure) == 0
        or not np.isfinite(structure.lattice.matrix).all()
        or not np.isfinite(structure.frac_coords).all()
        or structure.volume <= 0
    ):
        raise ValueError(f"Invalid structure for {row.material_id}.")

    prepared_structures[row.material_id] = structure

print("Primary shortlist:", DB2_FINAL_CSV)
print("Candidate count:", len(final_df))
print("Shortlist / full-audit consistency: PASS")


# ------------------------------------------------------------

# 5. Create a new package directory with a short Windows path
# ------------------------------------------------------------
run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

DB2_EXPORT_RUN_DIR = (
    Path(r"D:\TB3\DFT_exports")
    / f"DB2_{run_stamp}"
)

EXPORT_DIR = DB2_EXPORT_RUN_DIR
EXPORT_DIR.mkdir(parents=True, exist_ok=False)

CIF_DIR = EXPORT_DIR / "cif"
JSON_DIR = EXPORT_DIR / "json_per_material"

CIF_DIR.mkdir()
JSON_DIR.mkdir()

combined_records = []
manifest_rows = []

print("Export directory:", EXPORT_DIR)


# ------------------------------------------------------------
# 6. Export CIF and per-material JSON
# ------------------------------------------------------------
for index, row in final_df.iterrows():
    shortlist_rank = index + 1
    mid = row["material_id"]
    formula = row["formula_pretty"]
    slme_rank = int(row["SLME_rank"])
    eta = float(row["predicted_SLME_percent"])
    structure = prepared_structures[mid]

    stem = (
        f"shortlist{shortlist_rank:02d}"
        f"_SLMErank{slme_rank:03d}"
        f"_{safe_name(mid)}_{safe_name(formula)}"
    )
    cif_path = CIF_DIR / f"{stem}.cif"
    json_path = JSON_DIR / f"{stem}.json"

    if cif_path.exists() or json_path.exists():
        raise FileExistsError(f"Unexpected output-name collision: {stem}")

    # Preserve the selected structure; do not standardize or relax it.
    structure.to(filename=str(cif_path), fmt="cif")

    record = {
        "database": "Database 2",
        "database_name": "AB2C2-type, C = P or N",
        "selection_rule": (
            "predicted SLME > 30%; no full-original structure match; "
            "no unresolved matcher error"
        ),
        "shortlist_rank": shortlist_rank,
        "SLME_rank": slme_rank,
        "material_id": mid,
        "formula_pretty": formula,
        "composition_reduced": structure.composition.reduced_formula,
        "elements": sorted(el.symbol for el in structure.composition.elements),
        "num_sites": len(structure),
        "band_gap_eV": clean_scalar(row.get("band_gap")),
        "energy_above_hull_eV_atom": clean_scalar(row.get("energy_above_hull")),
        "is_stable": clean_scalar(row.get("is_stable")),
        "predicted_SLME_percent": eta,
        "thickness_nm": 500.0,
        "temperature_K": 300.0,
        "fr": 1.0,
        "gap_cleanup_source": "Materials Project band_gap",
        "cif_file": cif_path.relative_to(EXPORT_DIR).as_posix(),
        "structure": structure.as_dict(),
    }
    for column in BOOL_COLUMNS:
        record[column] = bool(row[column])

    write_json_new(json_path, record)

    # Validate the written JSON structure.
    loaded_record = json.loads(json_path.read_text(encoding="utf-8"))
    loaded_structure = Structure.from_dict(loaded_record["structure"])

    if loaded_record["material_id"] != mid:
        raise RuntimeError(f"JSON material ID mismatch: {mid}")

    np.testing.assert_allclose(
        loaded_structure.lattice.matrix,
        structure.lattice.matrix,
        rtol=0,
        atol=1e-10,
    )
    np.testing.assert_allclose(
        loaded_structure.frac_coords,
        structure.frac_coords,
        rtol=0,
        atol=1e-10,
    )

    # Check that the exported CIF can be parsed.
    cif_structure = Structure.from_file(str(cif_path))
    if not cif_structure.composition.element_composition.reduced_composition.almost_equals(
        structure.composition.element_composition.reduced_composition
    ):
        raise RuntimeError(f"CIF composition mismatch: {mid}")

    combined_records.append(record)
    manifest_rows.append({
        "shortlist_rank": shortlist_rank,
        "SLME_rank": slme_rank,
        "material_id": mid,
        "formula_pretty": formula,
        "band_gap_eV": record["band_gap_eV"],
        "energy_above_hull_eV_atom": record["energy_above_hull_eV_atom"],
        "is_stable": record["is_stable"],
        "predicted_SLME_percent": eta,
        "structure_new_vs_full_original_db": True,
        "structure_novelty_uncertain": False,
        "num_sites": len(structure),
        "num_elements": len(structure.composition.elements),
        "elements": ",".join(record["elements"]),
        # Portable paths relative to the package root.
        "cif_file": cif_path.relative_to(EXPORT_DIR).as_posix(),
        "json_file": json_path.relative_to(EXPORT_DIR).as_posix(),
        "export_status": "OK",
        "reason": "",
    })


# ------------------------------------------------------------
# 7. Save package metadata
# ------------------------------------------------------------
prefix = "database2_FINAL_SLME_gt30_structure_new"

COMBINED_JSON = EXPORT_DIR / f"{prefix}_combined.json"
MANIFEST_CSV = EXPORT_DIR / f"{prefix}_manifest.csv"
ID_CSV = EXPORT_DIR / f"{prefix}_material_ids.csv"
ID_TXT = EXPORT_DIR / f"{prefix}_material_ids.txt"
SUMMARY_CSV = EXPORT_DIR / f"{prefix}_export_summary.csv"

manifest_df = pd.DataFrame(manifest_rows)
id_df = manifest_df[[
    "shortlist_rank",
    "SLME_rank",
    "material_id",
    "formula_pretty",
    "predicted_SLME_percent",
]].copy()

write_json_new(COMBINED_JSON, combined_records)

manifest_df.to_csv(MANIFEST_CSV, index=False, encoding="utf-8-sig")
id_df.to_csv(ID_CSV, index=False, encoding="utf-8-sig")
final_df.to_csv(
    EXPORT_DIR / "ab2c2_SLME_gt30_new_for_DFT.csv",
    index=False,
    encoding="utf-8-sig",
)

with ID_TXT.open("x", encoding="utf-8") as handle:
    handle.write("\n".join(id_df["material_id"]) + "\n")

n_expected = len(final_df)
n_cif = len(list(CIF_DIR.glob("*.cif")))
n_json = len(list(JSON_DIR.glob("*.json")))

if not (
    len(manifest_df) == len(combined_records)
    == n_cif == n_json == n_expected
):
    raise RuntimeError("Exported file counts do not match the shortlist.")

if set(manifest_df["material_id"]) != set(final_df["material_id"]):
    raise RuntimeError("Exported IDs do not match the shortlist.")

summary_df = pd.DataFrame([{
    "database": "Database 2",
    "selection": "SLME >30%; structure-new; no unresolved matcher error",
    "input_candidates": n_expected,
    "exported_candidates": len(manifest_df),
    "failed_candidates": 0,
    "minimum_SLME_percent": final_df["predicted_SLME_percent"].min(),
    "maximum_SLME_percent": final_df["predicted_SLME_percent"].max(),
    "CIF_count": n_cif,
    "JSON_count": n_json,
}])
summary_df.to_csv(SUMMARY_CSV, index=False, encoding="utf-8-sig")

write_json_new(
    EXPORT_DIR / "source_audit_provenance.json",
    audit_provenance,
)
write_json_new(
    EXPORT_DIR / "export_provenance.json",
    {
        "created_at": datetime.now().astimezone().isoformat(),
        "source_shortlist": str(DB2_FINAL_CSV.resolve()),
        "source_full_audit": str(DB2_AUDIT_CSV.resolve()),
        "source_structure_cache": str(DB2_DFT_READY_PKL.resolve()),
        "n_exported": n_expected,
        "manifest_paths": "Relative to the package root",
        "structure_processing": "No relaxation or standardization applied",
        "novelty_scope": (
            "No match against the selected original database under "
            "the recorded StructureMatcher protocol."
        ),
    },
)

readme = f"""DATABASE 2 — DFT STRUCTURE PACKAGE

Number of candidates: {n_expected}

Selection:
- Predicted SLME > 30%.
- No structure match in the full original reference database.
- No unresolved StructureMatcher error.

Screening protocol:
- Direct-alpha prediction.
- MP band-gap cleanup.
- Thickness: 500 nm.
- Temperature: 300 K.
- Radiative fraction: 1.

Ranks:
- SLME_rank is the original full-pool screening rank.
- shortlist_rank is the position in this export package.

Files:
- cif/: one CIF per candidate.
- json_per_material/: metadata and structure per candidate.
- Manifest paths are relative to this package root.
- The selected shortlist and audit provenance are included.

These are candidate structures and screening metadata.
They are not completed DFT results or a full DFT calculation setup.

Structure-new is relative to the selected reference database
and the recorded matching protocol, not all published materials.
"""
with (EXPORT_DIR / "README.txt").open("x", encoding="utf-8") as handle:
    handle.write(readme)


# ------------------------------------------------------------
# 8. Create and validate ZIP
# ------------------------------------------------------------
ZIP_PATH = EXPORT_DIR.with_suffix(".zip")

# Exclusive ZIP creation prevents overwriting an existing archive.
with zipfile.ZipFile(
    ZIP_PATH,
    mode="x",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in sorted(EXPORT_DIR.rglob("*")):
        if path.is_file():
            archive.write(
                path,
                arcname=path.relative_to(EXPORT_DIR).as_posix(),
            )

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise RuntimeError(f"ZIP integrity check failed: {bad_member}")

    names = archive.namelist()
    zip_cifs = [
        name for name in names
        if name.startswith("cif/") and name.endswith(".cif")
    ]
    zip_jsons = [
        name for name in names
        if name.startswith("json_per_material/") and name.endswith(".json")
    ]

    if len(zip_cifs) != n_expected or len(zip_jsons) != n_expected:
        raise RuntimeError("ZIP structure-file counts do not match.")

    archived_records = json.loads(
        archive.read(COMBINED_JSON.name).decode("utf-8")
    )
    archived_ids = [record["material_id"] for record in archived_records]

    if (
        len(archived_ids) != n_expected
        or len(set(archived_ids)) != n_expected
        or set(archived_ids) != set(final_df["material_id"])
    ):
        raise RuntimeError("ZIP material IDs do not match the shortlist.")


# ------------------------------------------------------------
# 9. Final report
# ------------------------------------------------------------
DB2_EXPORT_ZIP = ZIP_PATH

print("\nDATABASE 2 DFT PACKAGE — PASS")
print("Source shortlist:", DB2_FINAL_CSV)
print("Candidates:", n_expected)
print("CIF files:", n_cif)
print("Per-material JSON files:", n_json)
print("Package directory:", EXPORT_DIR)
print("Manifest:", MANIFEST_CSV)
print("ZIP:", DB2_EXPORT_ZIP)
print("ZIP integrity and material-ID checks: PASS")
print("Existing packages were not overwritten.")

display(id_df)
display(summary_df)

Primary shortlist: D:\TB3\processed\paired_training\paper_outputs\dft_validation_shortlist\discovery_first_ab2c2_database2_SLME\audit_20260922_202740_195831\ab2c2_SLME_gt30_new_for_DFT.csv
Candidate count: 16
Shortlist / full-audit consistency: PASS
Export directory: D:\TB3\DFT_exports\DB2_20260922_203139_064450


C:\Users\ADMIN\AppData\Local\Programs\Python\Python310\lib\site-packages\pymatgen\core\structure.py:3109: UserWarning: Issues encountered while parsing CIF: 8 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]



DATABASE 2 DFT PACKAGE — PASS
Source shortlist: D:\TB3\processed\paired_training\paper_outputs\dft_validation_shortlist\discovery_first_ab2c2_database2_SLME\audit_20260922_202740_195831\ab2c2_SLME_gt30_new_for_DFT.csv
Candidates: 16
CIF files: 16
Per-material JSON files: 16
Package directory: D:\TB3\DFT_exports\DB2_20260922_203139_064450
Manifest: D:\TB3\DFT_exports\DB2_20260922_203139_064450\database2_FINAL_SLME_gt30_structure_new_manifest.csv
ZIP: D:\TB3\DFT_exports\DB2_20260922_203139_064450.zip
ZIP integrity and material-ID checks: PASS
Existing packages were not overwritten.


,shortlist_rank,SLME_rank,material_id,formula_pretty,predicted_SLME_percent
0,1,1,mp-lmu,Th2SeN2,33.666809
1,2,2,mp-ckxsa,Ba(GaP)2,33.607407
2,3,3,mp-lcr,K2P2Pd,33.163376
3,4,4,mp-cjkam,Ca2CdP2,33.160749
4,5,5,mp-lmv,Th2SN2,33.067132
5,6,6,mp-cqppq,Rb2P2Pd,32.942919
6,7,7,mp-wuz,K2GaP2,32.795662
7,8,8,mp-csyvr,Ca2MgN2,32.708407
8,9,9,mp-crnwz,Mg2ZnP2,32.439985
9,10,10,mp-met,K2SiP2,32.416114


,database,selection,input_candidates,exported_candidates,failed_candidates,minimum_SLME_percent,maximum_SLME_percent,CIF_count,JSON_count
0,Database 2,SLME >30%; structure-new; no unresolved matche...,16,16,0,30.1555,33.666809,16,16


## Cell 15. Combine Database 1 and Database 2 final discovery outputs

In [19]:
# Cell 15. Combine FINAL Database 1 + Database 2 portfolios
#
# DB1: Top-50 structure-new candidates ranked by SLME.
# DB2: Structure-new candidates with predicted SLME > 30%.
#
# Frozen manuscript expectations:
#   DB1 = 50
#   DB2 = 16
#   Shared candidate = mp-ckxsa
#   Combined unique portfolio = 65
#
# Source shortlists are preserved.
# Existing combined outputs are not overwritten.
# ============================================================

from pathlib import Path
from datetime import datetime
from itertools import combinations
import json

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

from pymatgen.core import Structure
from pymatgen.analysis.structure_matcher import StructureMatcher


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------
ROOT = Path(r"D:\TB3\processed\paired_training")
DFT_DIR = ROOT / "paper_outputs" / "dft_validation_shortlist"

DB1_DIR = DFT_DIR / "discovery_first_zintl_SLME"
DB1_FINAL_CSV = (
    DB1_DIR / "zintl_top50_structure_new_FINAL_SLME.csv"
)

# Historical directory name: used only as a structure cache.
# No SRP ranking is read from this directory.
DB1_STRUCTURE_PKL = (
    ROOT / "paper_outputs" / "screening_srp_200nm"
    / "external_pool_cache"
    / "screening_zintl_like_external_fixed_1100_with_structures.pkl"
)

if "DB2_DISCOVERY_RUN_DIR" not in globals():
    raise RuntimeError(
        "DB2_DISCOVERY_RUN_DIR is not defined. Run the updated "
        "Cell 13 first, or explicitly select the exact audit directory."
    )

DB2_DIR = Path(DB2_DISCOVERY_RUN_DIR)
DB2_FINAL_CSV = DB2_DIR / "ab2c2_SLME_gt30_new_for_DFT.csv"
DB2_PROVENANCE_JSON = DB2_DIR / "audit_provenance.json"

for path in (
    DB1_FINAL_CSV,
    DB1_STRUCTURE_PKL,
    DB2_FINAL_CSV,
    DB2_PROVENANCE_JSON,
):
    if not path.is_file():
        raise FileNotFoundError(f"Required input not found:\n{path}")

db2_provenance = json.loads(
    DB2_PROVENANCE_JSON.read_text(encoding="utf-8")
)
DB2_STRUCTURE_PKL = Path(db2_provenance["structure_pool"])

if not DB2_STRUCTURE_PKL.is_file():
    raise FileNotFoundError(
        f"Recorded DB2 structure cache not found:\n{DB2_STRUCTURE_PKL}"
    )

EXPECTED_DB1 = 50
EXPECTED_DB2 = 16
EXPECTED_SOURCE_ROWS = 66
EXPECTED_FINAL_UNIQUE = 65
EXPECTED_SHARED_IDS = {"mp-ckxsa"}
SLME_CROSS_POOL_TOL_PP = 1e-5

db1 = pd.read_csv(DB1_FINAL_CSV)
db2 = pd.read_csv(DB2_FINAL_CSV)


# ------------------------------------------------------------
# 2. Validation helpers
# ------------------------------------------------------------
def normalize_ids(frame, label):
    if "material_id" not in frame:
        raise KeyError(f"{label}: missing material_id.")

    if frame["material_id"].isna().any():
        raise ValueError(f"{label}: missing material_id values.")

    frame["material_id"] = (
        frame["material_id"].astype(str).str.strip()
    )

    if frame["material_id"].eq("").any():
        raise ValueError(f"{label}: empty material_id values.")

    if frame["material_id"].duplicated().any():
        raise ValueError(f"{label}: duplicated material_id values.")


def require_boolean(frame, column, expected, label):
    if column not in frame:
        raise KeyError(f"{label}: missing required flag {column}.")

    values = (
        frame[column].astype(str).str.strip().str.lower()
        .map({"true": True, "false": False, "1": True, "0": False})
    )

    if values.isna().any():
        raise ValueError(f"{label}: invalid boolean values in {column}.")

    frame[column] = values.astype(bool)

    if not frame[column].eq(expected).all():
        bad_ids = frame.loc[
            ~frame[column].eq(expected), "material_id"
        ].tolist()
        raise ValueError(
            f"{label}: {column} must be {expected}; invalid IDs: {bad_ids}"
        )


def prepare_shortlist(frame, label, expected_count, database_name):
    frame = frame.copy()

    required = [
        "material_id",
        "formula_pretty",
        "SLME_rank",
        "predicted_SLME_percent",
    ]
    missing = [column for column in required if column not in frame]
    if missing:
        raise KeyError(f"{label}: missing columns {missing}")

    normalize_ids(frame, label)

    if len(frame) != expected_count:
        raise ValueError(
            f"{label}: expected {expected_count} candidates, "
            f"found {len(frame)}. Inspect the source audit."
        )

    if frame["formula_pretty"].isna().any():
        raise ValueError(f"{label}: missing formula_pretty.")

    frame["formula_pretty"] = (
        frame["formula_pretty"].astype(str).str.strip()
    )
    if frame["formula_pretty"].eq("").any():
        raise ValueError(f"{label}: empty formula_pretty.")

    scores = pd.to_numeric(
        frame["predicted_SLME_percent"], errors="raise"
    ).to_numpy(float)

    if not np.isfinite(scores).all() or not (scores > 30.0).all():
        raise ValueError(f"{label}: all SLME values must be finite and >30%.")

    frame["predicted_SLME_percent"] = scores

    ranks = pd.to_numeric(
        frame["SLME_rank"], errors="raise"
    ).to_numpy(float)

    if (
        not np.isfinite(ranks).all()
        or (ranks < 1).any()
        or not np.equal(ranks, np.floor(ranks)).all()
        or len(np.unique(ranks)) != len(ranks)
    ):
        raise ValueError(f"{label}: invalid or duplicated SLME_rank.")

    frame["SLME_rank"] = ranks.astype(int)
    frame = frame.sort_values("SLME_rank").reset_index(drop=True)

    if not frame["predicted_SLME_percent"].is_monotonic_decreasing:
        raise ValueError(f"{label}: source ranking is inconsistent.")

    frame["source_SLME_rank"] = frame["SLME_rank"]
    frame["source_shortlist_rank"] = np.arange(1, len(frame) + 1)
    frame["database"] = label
    frame["database_name"] = database_name

    return frame


db1 = prepare_shortlist(
    db1, "Database 1", EXPECTED_DB1, "Zintl-like"
)
db2 = prepare_shortlist(
    db2, "Database 2", EXPECTED_DB2, "AB2C2-type, C=P/N"
)

# DB1 and DB2 use different novelty-column names.
require_boolean(
    db1, "new_relative_to_full_original_database", True, "Database 1"
)
require_boolean(
    db1, "structure_match_full_original_database", False, "Database 1"
)
require_boolean(
    db1, "structure_match_audit_complete", True, "Database 1"
)
require_boolean(
    db1, "eligible_new_for_DFT", True, "Database 1"
)

db1_error_count = pd.to_numeric(
    db1["n_structurematcher_errors"], errors="raise"
).to_numpy(float)
if not np.isfinite(db1_error_count).all() or (db1_error_count != 0).any():
    raise ValueError("Database 1 shortlist contains unresolved matcher errors.")

require_boolean(
    db2, "structure_new_vs_full_original_db", True, "Database 2"
)
for column in (
    "structure_match_train",
    "structure_match_paired_optical_db",
    "structure_match_full_original_db",
    "structure_matcher_had_error",
    "structure_novelty_uncertain",
):
    require_boolean(db2, column, False, "Database 2")

# Add common, explicitly validated flags to both source tables.
for frame in (db1, db2):
    frame["validated_structure_new"] = True
    frame["validated_no_unresolved_matcher_error"] = True

print("DB1 source:", DB1_FINAL_CSV)
print("DB2 source:", DB2_FINAL_CSV)
print("Individual shortlist validation: PASS")


# ------------------------------------------------------------
# 3. Load only the selected structures from each cache
# ------------------------------------------------------------
def to_structure(value):
    if isinstance(value, Structure):
        return value
    if isinstance(value, dict):
        return Structure.from_dict(value)
    if isinstance(value, str):
        return Structure.from_dict(json.loads(value))
    raise TypeError(f"Unsupported structure type: {type(value)}")


def load_selected_structures(path, shortlist, label):
    cache = pd.read_pickle(path).copy()
    normalize_ids(cache, f"{label} structure cache")

    if "structure" not in cache:
        raise KeyError(f"{label}: structure cache lacks structure.")

    cache = cache.set_index("material_id")
    missing = set(shortlist["material_id"]) - set(cache.index)
    if missing:
        raise ValueError(f"{label}: missing structures for {sorted(missing)}")

    selected = {}
    for mid in shortlist["material_id"]:
        structure = to_structure(cache.loc[mid, "structure"])
        if (
            len(structure) == 0
            or structure.volume <= 0
            or not np.isfinite(structure.lattice.matrix).all()
            or not np.isfinite(structure.frac_coords).all()
        ):
            raise ValueError(f"{label}: invalid structure for {mid}.")
        selected[mid] = structure

    return selected


db1_structure_map = load_selected_structures(
    DB1_STRUCTURE_PKL, db1, "Database 1"
)
db2_structure_map = load_selected_structures(
    DB2_STRUCTURE_PKL, db2, "Database 2"
)

# Formula prefilters are derived from actual structures.
for frame, structure_map, original_formula_column in (
    (db1, db1_structure_map, "candidate_reduced_formula"),
    (db2, db2_structure_map, "structure_reduced_formula"),
):
    frame["reduced_formula"] = frame["material_id"].map(
        {
            mid: structure.composition.reduced_formula
            for mid, structure in structure_map.items()
        }
    )

    if original_formula_column in frame:
        if not np.array_equal(
            frame[original_formula_column].astype(str).to_numpy(),
            frame["reduced_formula"].to_numpy(),
        ):
            raise ValueError(
                f"Cached structure formulas differ from {original_formula_column}."
            )


# ------------------------------------------------------------
# 4. Structural comparisons
# ------------------------------------------------------------
MATCHER_CONFIG = {
    "ltol": 0.20,
    "stol": 0.30,
    "angle_tol": 5.0,
    "primitive_cell": True,
    "scale": True,
    "attempt_supercell": True,
}
matcher = StructureMatcher(**MATCHER_CONFIG)

cross_material_ids = sorted(
    set(db1["material_id"]) & set(db2["material_id"])
)
cross_formulas = sorted(
    set(db1["reduced_formula"]) & set(db2["reduced_formula"])
)

cross_rows = []
error_rows = []
within_rows = []

# Check within-pool equivalence too, so "unique structures" is
# not inferred solely from unique material IDs.
for label, frame, structure_map in (
    ("Database 1", db1, db1_structure_map),
    ("Database 2", db2, db2_structure_map),
):
    for formula, group in frame.groupby("reduced_formula", sort=False):
        for mid1, mid2 in combinations(group["material_id"], 2):
            try:
                matched = matcher.fit(
                    structure_map[mid1], structure_map[mid2]
                )
            except Exception as exc:
                error_rows.append({
                    "scope": "within_pool",
                    "database1": label,
                    "database2": label,
                    "database1_material_id": mid1,
                    "database2_material_id": mid2,
                    "reduced_formula": formula,
                    "error": str(exc),
                })
                continue

            within_rows.append({
                "database": label,
                "material_id_1": mid1,
                "material_id_2": mid2,
                "reduced_formula": formula,
                "structure_match": bool(matched),
            })

db2_groups = {
    formula: group
    for formula, group in db2.groupby("reduced_formula", sort=False)
}

for row1 in tqdm(
    db1.itertuples(index=False),
    total=len(db1),
    desc="DB1 vs DB2 StructureMatcher",
):
    candidates = db2_groups.get(row1.reduced_formula)
    if candidates is None:
        continue

    for row2 in candidates.itertuples(index=False):
        try:
            matched = matcher.fit(
                db1_structure_map[row1.material_id],
                db2_structure_map[row2.material_id],
            )
        except Exception as exc:
            error_rows.append({
                "scope": "cross_pool",
                "database1": "Database 1",
                "database2": "Database 2",
                "database1_material_id": row1.material_id,
                "database2_material_id": row2.material_id,
                "reduced_formula": row1.reduced_formula,
                "error": str(exc),
            })
            continue

        cross_rows.append({
            "database1_material_id": row1.material_id,
            "database1_formula": row1.formula_pretty,
            "database1_SLME_percent": row1.predicted_SLME_percent,
            "database2_material_id": row2.material_id,
            "database2_formula": row2.formula_pretty,
            "database2_SLME_percent": row2.predicted_SLME_percent,
            "reduced_formula": row1.reduced_formula,
            "same_material_id": row1.material_id == row2.material_id,
            "structure_match": bool(matched),
        })

cross_formula_df = pd.DataFrame(
    cross_rows,
    columns=[
        "database1_material_id", "database1_formula",
        "database1_SLME_percent", "database2_material_id",
        "database2_formula", "database2_SLME_percent",
        "reduced_formula", "same_material_id", "structure_match",
    ],
)
cross_structure_matches_df = cross_formula_df.loc[
    cross_formula_df["structure_match"].eq(True)
].copy()

within_audit_df = pd.DataFrame(
    within_rows,
    columns=[
        "database", "material_id_1", "material_id_2",
        "reduced_formula", "structure_match",
    ],
)
within_matches_df = within_audit_df.loc[
    within_audit_df["structure_match"].eq(True)
].copy()

all_errors_df = pd.DataFrame(
    error_rows,
    columns=[
        "scope", "database1", "database2",
        "database1_material_id", "database2_material_id",
        "reduced_formula", "error",
    ],
)
cross_structure_errors_df = all_errors_df.loc[
    all_errors_df["scope"].eq("cross_pool")
].copy()


# ------------------------------------------------------------
# 5. Save comparison diagnostics to a new directory
# ------------------------------------------------------------
COMBINED_RUN_DIR = (
    DFT_DIR / "FINAL_combined_database1_database2_SLME"
    / f"run_{datetime.now():%Y%m%d_%H%M%S_%f}"
)
FINAL_DIR = COMBINED_RUN_DIR
FINAL_DIR.mkdir(parents=True, exist_ok=False)

CROSS_FORMULA_CSV = FINAL_DIR / "FINAL_cross_database_formula_overlap.csv"
CROSS_STRUCTURE_CSV = FINAL_DIR / "FINAL_cross_database_structure_matches.csv"
CROSS_STRUCTURE_ERROR_CSV = (
    FINAL_DIR / "FINAL_cross_database_structure_match_errors.csv"
)

for frame, path in (
    (cross_formula_df, CROSS_FORMULA_CSV),
    (cross_structure_matches_df, CROSS_STRUCTURE_CSV),
    (cross_structure_errors_df, CROSS_STRUCTURE_ERROR_CSV),
    (all_errors_df, FINAL_DIR / "all_structure_matcher_errors.csv"),
    (within_audit_df, FINAL_DIR / "within_pool_structure_comparisons.csv"),
):
    frame.to_csv(path, index=False, encoding="utf-8-sig")

# No final combined portfolio is written if the audit is unresolved.
if not all_errors_df.empty:
    display(all_errors_df)
    raise RuntimeError(
        f"StructureMatcher errors detected. Diagnostics saved to:\n{FINAL_DIR}"
    )

if not within_matches_df.empty:
    display(within_matches_df)
    raise RuntimeError(
        "Within-pool structural duplicates were found. "
        "Inspect them before claiming a unique-structure portfolio."
    )

if set(cross_material_ids) != EXPECTED_SHARED_IDS:
    raise RuntimeError(
        f"Unexpected shared IDs: {cross_material_ids}. "
        "Inspect the source snapshots."
    )

matched_pairs = set(zip(
    cross_structure_matches_df["database1_material_id"],
    cross_structure_matches_df["database2_material_id"],
))
if matched_pairs != {("mp-ckxsa", "mp-ckxsa")}:
    raise RuntimeError(
        f"Unexpected cross-pool structural matches: {matched_pairs}. "
        "No automatic deduplication was applied."
    )


# ------------------------------------------------------------
# 6. Preserve source memberships and verify shared SLME
# ------------------------------------------------------------
combined_source = pd.concat([db1, db2], ignore_index=True, sort=False)

if len(combined_source) != EXPECTED_SOURCE_ROWS:
    raise RuntimeError("Unexpected number of source membership rows.")

shared_scores = combined_source.loc[
    combined_source["material_id"].eq("mp-ckxsa"),
    "predicted_SLME_percent",
].to_numpy(float)

if len(shared_scores) != 2:
    raise RuntimeError("Expected exactly two source rows for mp-ckxsa.")

duplicate_slme_difference = float(np.ptp(shared_scores))
if duplicate_slme_difference > SLME_CROSS_POOL_TOL_PP:
    raise RuntimeError(
        "Shared-candidate SLME mismatch.\n"
        f"Absolute difference: {duplicate_slme_difference:.12e} pp\n"
        f"Tolerance: {SLME_CROSS_POOL_TOL_PP:.1e} pp"
    )

membership_rows = []

for mid, group in combined_source.groupby("material_id", sort=False):
    source_names = sorted(group["database"].unique())
    membership = {
        "material_id": mid,
        "source_pool_count": len(source_names),
        "source_pools": ";".join(source_names),
        "source_database_names": ";".join(
            sorted(group["database_name"].unique())
        ),
        "cross_pool_candidate": len(source_names) > 1,
    }

    for number, label in ((1, "Database 1"), (2, "Database 2")):
        source = group[group["database"].eq(label)]
        present = not source.empty
        membership[f"database{number}_shortlist_rank"] = (
            int(source.iloc[0]["source_shortlist_rank"]) if present else np.nan
        )
        membership[f"database{number}_SLME_rank"] = (
            int(source.iloc[0]["source_SLME_rank"]) if present else np.nan
        )
        membership[f"database{number}_SLME_percent"] = (
            float(source.iloc[0]["predicted_SLME_percent"])
            if present else np.nan
        )

    membership["cross_pool_SLME_abs_diff_pp"] = (
        abs(
            membership["database1_SLME_percent"]
            - membership["database2_SLME_percent"]
        )
        if len(source_names) > 1 else np.nan
    )
    membership_rows.append(membership)

membership_df = pd.DataFrame(membership_rows)


# ------------------------------------------------------------
# 7. Build the unique union
# ------------------------------------------------------------
# ID deduplication is permitted here only after the structural
# audit confirms that the sole duplicate is the shared mp-ckxsa.
# Preserve the original representative-selection rule:
# higher SLME first, then Database 1 when scores are identical.
combined_unique = (
    combined_source
    .sort_values(
        ["predicted_SLME_percent", "database"],
        ascending=[False, True],
        kind="mergesort",
    )
    .drop_duplicates("material_id", keep="first")
    .reset_index(drop=True)
)

membership_columns = [
    column for column in membership_df if column != "material_id"
]
combined_unique = (
    combined_unique
    .drop(columns=membership_columns, errors="ignore")
    .merge(
        membership_df,
        on="material_id",
        how="left",
        validate="one_to_one",
    )
    .sort_values("predicted_SLME_percent", ascending=False, kind="mergesort")
    .reset_index(drop=True)
)

if (
    len(combined_unique) != EXPECTED_FINAL_UNIQUE
    or combined_unique["material_id"].duplicated().any()
):
    raise RuntimeError("Final unique portfolio count/ID check failed.")

if set(combined_unique["material_id"]) != (
    set(db1["material_id"]) | set(db2["material_id"])
):
    raise RuntimeError("The final union does not preserve all source IDs.")

combined_unique["global_SLME_rank"] = np.arange(
    1, len(combined_unique) + 1
)
combined_unique["combined_portfolio_size"] = len(combined_unique)
combined_unique["combined_selection_rule"] = (
    "Union of DB1 final Top-50 and DB2 structure-new SLME >30% "
    "shortlists; the structurally verified shared candidate is retained once."
)

db2_unique_additions = int(
    membership_df["source_pools"].eq("Database 2").sum()
)

summary_values = [
    ("Database1_source_shortlist", len(db1)),
    ("Database2_source_shortlist", len(db2)),
    ("Source_rows_before_cross_pool_dedup", len(combined_source)),
    ("Exact_cross_pool_material_overlap", len(cross_material_ids)),
    ("Cross_pool_reduced_formula_overlap", len(cross_formulas)),
    ("Cross_pool_StructureMatcher_overlap", len(cross_structure_matches_df)),
    ("Cross_pool_StructureMatcher_errors", len(cross_structure_errors_df)),
    ("Within_pool_StructureMatcher_overlap", len(within_matches_df)),
    ("Database2_unique_additions_beyond_DB1", db2_unique_additions),
    ("FINAL_unique_DFT_portfolio", len(combined_unique)),
]
for threshold in (30, 31, 32):
    summary_values.append((
        f"FINAL_SLME_gt{threshold}",
        int((combined_unique["predicted_SLME_percent"] > threshold).sum()),
    ))

summary_df = pd.DataFrame(summary_values, columns=["item", "count"])


# ------------------------------------------------------------
# 8. Save combined outputs and provenance
# ------------------------------------------------------------
COMBINED_CSV = (
    FINAL_DIR / "FINAL_combined_65_unique_database1_database2_SLME.csv"
)
SUMMARY_CSV = FINAL_DIR / "FINAL_combined_65_unique_summary.csv"
MEMBERSHIP_CSV = FINAL_DIR / "FINAL_cross_pool_membership.csv"
SOURCE_66_CSV = (
    FINAL_DIR / "FINAL_source_66_membership_rows_before_dedup.csv"
)

for frame, path in (
    (combined_unique, COMBINED_CSV),
    (summary_df, SUMMARY_CSV),
    (membership_df, MEMBERSHIP_CSV),
    (combined_source, SOURCE_66_CSV),
):
    frame.to_csv(path, index=False, encoding="utf-8-sig")

provenance = {
    "created_at": datetime.now().astimezone().isoformat(),
    "database1_shortlist": str(DB1_FINAL_CSV.resolve()),
    "database2_shortlist": str(DB2_FINAL_CSV.resolve()),
    "database1_structure_cache": str(DB1_STRUCTURE_PKL.resolve()),
    "database2_structure_cache": str(DB2_STRUCTURE_PKL.resolve()),
    "database2_audit_provenance": db2_provenance,
    "matcher_config": MATCHER_CONFIG,
    "n_database1": len(db1),
    "n_database2": len(db2),
    "n_source_memberships": len(combined_source),
    "n_unique_candidates": len(combined_unique),
    "shared_material_ids": cross_material_ids,
    "shared_SLME_abs_difference_pp": duplicate_slme_difference,
    "shared_SLME_tolerance_pp": SLME_CROSS_POOL_TOL_PP,
    "representative_selection": (
        "Higher predicted SLME; Database 1 breaks exact score ties. "
        "Both source scores and ranks remain in the membership table."
    ),
    "scope": (
        "Candidate portfolio, not a count of completed DFT validations. "
        "Structural equivalence is defined by the recorded matcher protocol."
    ),
}
with (FINAL_DIR / "combined_provenance.json").open(
    "x", encoding="utf-8"
) as handle:
    json.dump(provenance, handle, indent=2, ensure_ascii=False, allow_nan=False)


# ------------------------------------------------------------
# 9. Report
# ------------------------------------------------------------
print("\nFINAL DB1 + DB2 PORTFOLIO — PASS")
print("Database 1:", len(db1))
print("Database 2:", len(db2))
print("Source memberships:", len(combined_source))
print("Shared candidate:", cross_material_ids)
print("Shared SLME difference:", duplicate_slme_difference, "pp")
print("DB2 unique additions:", db2_unique_additions)
print("Unique portfolio:", len(combined_unique))
print("Combined output directory:", FINAL_DIR)
print("Combined CSV:", COMBINED_CSV)
print("Existing combined outputs were not overwritten.")

display(summary_df)

print("\nSHARED CANDIDATE AND SOURCE PROVENANCE")
display(
    combined_unique.loc[
        combined_unique["cross_pool_candidate"],
        [
            "global_SLME_rank", "material_id", "formula_pretty",
            "source_pools",
            "database1_SLME_rank", "database2_SLME_rank",
            "database1_SLME_percent", "database2_SLME_percent",
            "cross_pool_SLME_abs_diff_pp",
        ],
    ]
)

print("\nFINAL GLOBAL SLME RANKING")
display_columns = [
    "global_SLME_rank",
    "source_pools",
    "material_id",
    "formula_pretty",
    "predicted_SLME_percent",
    "database1_shortlist_rank",
    "database2_shortlist_rank",
]
display(combined_unique[display_columns])

DB1 source: D:\TB3\processed\paired_training\paper_outputs\dft_validation_shortlist\discovery_first_zintl_SLME\zintl_top50_structure_new_FINAL_SLME.csv
DB2 source: D:\TB3\processed\paired_training\paper_outputs\dft_validation_shortlist\discovery_first_ab2c2_database2_SLME\audit_20260922_202740_195831\ab2c2_SLME_gt30_new_for_DFT.csv
Individual shortlist validation: PASS


DB1 vs DB2 StructureMatcher:   0%|          | 0/50 [00:00<?, ?it/s]


FINAL DB1 + DB2 PORTFOLIO — PASS
Database 1: 50
Database 2: 16
Source memberships: 66
Shared candidate: ['mp-ckxsa']
Shared SLME difference: 1.6496812804689398e-08 pp
DB2 unique additions: 15
Unique portfolio: 65
Combined output directory: D:\TB3\processed\paired_training\paper_outputs\dft_validation_shortlist\FINAL_combined_database1_database2_SLME\run_20260922_203316_145557
Combined CSV: D:\TB3\processed\paired_training\paper_outputs\dft_validation_shortlist\FINAL_combined_database1_database2_SLME\run_20260922_203316_145557\FINAL_combined_65_unique_database1_database2_SLME.csv
Existing combined outputs were not overwritten.


,item,count
0,Database1_source_shortlist,50
1,Database2_source_shortlist,16
2,Source_rows_before_cross_pool_dedup,66
3,Exact_cross_pool_material_overlap,1
4,Cross_pool_reduced_formula_overlap,1
5,Cross_pool_StructureMatcher_overlap,1
6,Cross_pool_StructureMatcher_errors,0
7,Within_pool_StructureMatcher_overlap,0
8,Database2_unique_additions_beyond_DB1,15
9,FINAL_unique_DFT_portfolio,65



SHARED CANDIDATE AND SOURCE PROVENANCE


,global_SLME_rank,material_id,formula_pretty,source_pools,database1_SLME_rank,database2_SLME_rank,database1_SLME_percent,database2_SLME_percent,cross_pool_SLME_abs_diff_pp
17,18,mp-ckxsa,Ba(GaP)2,Database 1;Database 2,17.0,2.0,33.607407,33.607407,1.649681e-08



FINAL GLOBAL SLME RANKING


,global_SLME_rank,source_pools,material_id,formula_pretty,predicted_SLME_percent,database1_shortlist_rank,database2_shortlist_rank
0,1,Database 1,mp-btzq,NaP5,33.753417,1.0,NaN
1,2,Database 1,mp-bevqy,Rb2Ag(AsSe2)3,33.727537,2.0,NaN
2,3,Database 1,mp-cqyqi,KIn(PSe3)2,33.690766,3.0,NaN
3,4,Database 1,mp-bsnx,Ba2B4Se13,33.674518,4.0,NaN
4,5,Database 1,mp-cqcrk,KAgPSe3,33.668234,5.0,NaN
...,...,...,...,...,...,...,...
60,61,Database 2,mp-mgk,Ba(MgP)2,32.012468,NaN,12.0
61,62,Database 2,mp-cqplq,Cs2P2Pd,31.678174,NaN,13.0
62,63,Database 2,mp-cgplr,Na2GeN2,30.568927,NaN,14.0
63,64,Database 2,mp-ldj,Th2TeN2,30.265500,NaN,15.0


In [22]:
# Cell 16A. Package the existing FINAL Database 1 Top-50 export
#
# This cell:
# - Uses the explicitly selected DB1 export directory.
# - Validates the shortlist, manifest, and per-material JSON.
# - Requires exactly 50 CIF files and 50 per-material JSON files.
# - Creates a new ZIP without deleting or overwriting old files.
#
# It does not rerun screening, export new structures,
# modify source files, or send the package to anyone.
# ============================================================

from pathlib import Path
from datetime import datetime
import json
import zipfile

import numpy as np
import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# 1. Explicit source paths
# ------------------------------------------------------------
ROOT = Path(r"D:\TB3\processed\paired_training")

DB1_DIR = (
    ROOT / "paper_outputs" / "dft_validation_shortlist"
    / "discovery_first_zintl_SLME"
)

FINAL_CSV = DB1_DIR / "zintl_top50_structure_new_FINAL_SLME.csv"

EXPORT_SOURCE_DIR = (
    DB1_DIR / "SEND_TO_DFT_DB1_FINAL_Top50_SLME"
)
SOURCE_MANIFEST = (
    EXPORT_SOURCE_DIR / "DB1_FINAL_Top50_SLME_manifest.csv"
)
SOURCE_CIF_DIR = EXPORT_SOURCE_DIR / "CIF"
SOURCE_JSON_DIR = EXPORT_SOURCE_DIR / "JSON"

EXPECTED_COUNT = 50
# Absolute numerical tolerance in percentage points.
# IDs, ranks, and formulas are still checked exactly.
SLME_TOL_PP = 1e-6

for path in (FINAL_CSV, SOURCE_MANIFEST):
    if not path.is_file():
        raise FileNotFoundError(f"Required input not found:\n{path}")

for path in (SOURCE_CIF_DIR, SOURCE_JSON_DIR):
    if not path.is_dir():
        raise FileNotFoundError(f"Required structure directory not found:\n{path}")

final_df = pd.read_csv(FINAL_CSV)
manifest_df = pd.read_csv(SOURCE_MANIFEST)


# ------------------------------------------------------------
# 2. Validation helpers
# ------------------------------------------------------------
def validate_ids(frame, label):
    if "material_id" not in frame:
        raise KeyError(f"{label}: missing material_id.")

    if frame["material_id"].isna().any():
        raise ValueError(f"{label}: missing material_id values.")

    frame["material_id"] = (
        frame["material_id"].astype(str).str.strip()
    )

    if frame["material_id"].eq("").any():
        raise ValueError(f"{label}: empty material_id values.")

    if frame["material_id"].duplicated().any():
        raise ValueError(f"{label}: duplicated material_id values.")

    if len(frame) != EXPECTED_COUNT:
        raise ValueError(
            f"{label}: expected {EXPECTED_COUNT} rows, found {len(frame)}."
        )


def require_boolean(frame, column, expected):
    if column not in frame:
        raise KeyError(f"Shortlist is missing {column}.")

    values = (
        frame[column].astype(str).str.strip().str.lower()
        .map({"true": True, "false": False, "1": True, "0": False})
    )

    if values.isna().any() or not values.eq(expected).all():
        raise ValueError(f"Shortlist requires {column} = {expected}.")


def resolve_member(directory, filename):
    if pd.isna(filename):
        raise ValueError("Missing structure filename in manifest.")

    filename = str(filename).strip()
    path = (directory / filename).resolve()

    # The existing DB1 manifest uses filenames directly within
    # the CIF and JSON directories.
    if path.parent != directory.resolve():
        raise ValueError(f"Unexpected manifest path: {filename}")

    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f"Missing or empty structure file:\n{path}")

    return path


# ------------------------------------------------------------
# 3. Validate shortlist and manifest
# ------------------------------------------------------------
validate_ids(final_df, "Final shortlist")
validate_ids(manifest_df, "Export manifest")

required_final = [
    "SLME_rank",
    "formula_pretty",
    "predicted_SLME_percent",
    "n_structurematcher_errors",
]
required_manifest = [
    "SLME_rank",
    "formula_pretty",
    "predicted_SLME_percent",
    "CIF_file",
    "JSON_file",
    "export_status",
]

for frame, columns, label in (
    (final_df, required_final, "Final shortlist"),
    (manifest_df, required_manifest, "Export manifest"),
):
    missing = [column for column in columns if column not in frame]
    if missing:
        raise KeyError(f"{label}: missing columns {missing}")

require_boolean(
    final_df, "new_relative_to_full_original_database", True
)
require_boolean(
    final_df, "structure_match_full_original_database", False
)
require_boolean(
    final_df, "structure_match_audit_complete", True
)
require_boolean(final_df, "eligible_new_for_DFT", True)

errors = pd.to_numeric(
    final_df["n_structurematcher_errors"], errors="raise"
).to_numpy(float)

if not np.isfinite(errors).all() or (errors != 0).any():
    raise ValueError("The shortlist contains unresolved matcher errors.")

if not manifest_df["export_status"].eq("OK").all():
    raise ValueError("The source manifest contains unsuccessful exports.")

if set(final_df["material_id"]) != set(manifest_df["material_id"]):
    raise ValueError("Manifest IDs do not match the final Top-50 shortlist.")

# Reorder the manifest explicitly by the source shortlist.
manifest_df = (
    manifest_df.set_index("material_id")
    .loc[final_df["material_id"]]
    .reset_index()
)

for frame in (final_df, manifest_df):
    frame["predicted_SLME_percent"] = pd.to_numeric(
        frame["predicted_SLME_percent"], errors="raise"
    )
    scores = frame["predicted_SLME_percent"].to_numpy(float)

    if not np.isfinite(scores).all() or not (scores > 30.0).all():
        raise ValueError("All shortlisted SLME values must be finite and >30%.")

    ranks = pd.to_numeric(
        frame["SLME_rank"], errors="raise"
    ).to_numpy(float)

    if (
        not np.isfinite(ranks).all()
        or (ranks < 1).any()
        or not np.equal(ranks, np.floor(ranks)).all()
        or len(np.unique(ranks)) != len(ranks)
    ):
        raise ValueError("Invalid or duplicated SLME ranks.")

    frame["SLME_rank"] = ranks.astype(int)

np.testing.assert_array_equal(
    final_df["SLME_rank"].to_numpy(),
    manifest_df["SLME_rank"].to_numpy(),
)
np.testing.assert_allclose(
    final_df["predicted_SLME_percent"].to_numpy(float),
    manifest_df["predicted_SLME_percent"].to_numpy(float),
    rtol=0,
    atol=SLME_TOL_PP,
)
np.testing.assert_array_equal(
    final_df["formula_pretty"].astype(str).to_numpy(),
    manifest_df["formula_pretty"].astype(str).to_numpy(),
)


# ------------------------------------------------------------
# 4. Validate the exact structure files to include
# ------------------------------------------------------------
structure_members = []
portable_rows = []

for _, row in manifest_df.iterrows():
    mid = row["material_id"]

    cif_path = resolve_member(SOURCE_CIF_DIR, row["CIF_file"])
    json_path = resolve_member(SOURCE_JSON_DIR, row["JSON_file"])

    if cif_path.suffix.lower() != ".cif":
        raise ValueError(f"Unexpected CIF extension for {mid}.")
    if json_path.suffix.lower() != ".json":
        raise ValueError(f"Unexpected JSON extension for {mid}.")

    record = json.loads(json_path.read_text(encoding="utf-8"))

    if str(record.get("material_id", "")).strip() != mid:
        raise ValueError(f"JSON material_id mismatch: {mid}")

    if not isinstance(record.get("structure"), dict):
        raise ValueError(f"Missing structure dictionary in JSON: {mid}")

    if record.get("SLME_rank") != int(row["SLME_rank"]):
        raise ValueError(f"JSON SLME_rank mismatch: {mid}")

    json_eta = float(record["predicted_SLME_percent"])
    if (
        not np.isfinite(json_eta)
        or abs(json_eta - float(row["predicted_SLME_percent"])) > SLME_TOL_PP
    ):
        raise ValueError(f"JSON SLME mismatch: {mid}")

    cif_member = f"CIF/{cif_path.name}"
    json_member = f"JSON/{json_path.name}"

    structure_members.extend([
        (cif_path, cif_member),
        (json_path, json_member),
    ])

    portable_row = row.to_dict()
    portable_row["CIF_file"] = cif_member
    portable_row["JSON_file"] = json_member
    portable_rows.append(portable_row)

member_names = [name for _, name in structure_members]
if len(set(member_names)) != 2 * EXPECTED_COUNT:
    raise ValueError("Duplicate structure filenames in the selected package.")

if len(list(SOURCE_CIF_DIR.glob("*.cif"))) != EXPECTED_COUNT:
    raise ValueError("Source CIF directory does not contain exactly 50 CIFs.")

if len(list(SOURCE_JSON_DIR.glob("*.json"))) != EXPECTED_COUNT:
    raise ValueError("Source JSON directory does not contain exactly 50 JSONs.")

portable_manifest = pd.DataFrame(portable_rows)

print("Source shortlist:", FINAL_CSV)
print("Source package:", EXPORT_SOURCE_DIR)
print("Shortlist / manifest / JSON validation: PASS")


# ------------------------------------------------------------
# 5. Audit reports and package metadata
# ------------------------------------------------------------
report_candidates = [
    DB1_DIR / "zintl_FINAL_SLME_structure_novelty_summary.csv",
    DB1_DIR / "zintl_structure_matcher_matches.csv",
    DB1_DIR / "zintl_structure_matcher_errors.csv",
]

report_files = [path for path in report_candidates if path.is_file()]
missing_reports = [
    path.name for path in report_candidates if not path.is_file()
]

README_TEXT = f"""DATABASE 1 — ZINTL-LIKE TOP-50 STRUCTURE PACKAGE

Candidates: {EXPECTED_COUNT}

Selection:
- Final DB1 Top-50 ranked by predicted SLME.
- Predicted SLME > 30%.
- Structure-new relative to the original reference database.
- Source shortlist audit completed without matcher errors.

Screening:
- Direct-alpha prediction using the SSL-initialized GINE ensemble.
- Materials Project band-gap cleanup.
- AM1.5G, 500 nm, 300 K, radiative fraction 1.

Package:
- CIF/: 50 existing CIF files.
- JSON/: 50 existing per-material JSON files.
- package_manifest.csv: paths relative to the ZIP root.
- reports/: selected shortlist and available audit reports.

No structures were relaxed, standardized, or regenerated here.
This cell packages existing files and validates their IDs and metadata.
It does not revalidate CIF geometry or DFT convergence.

These are screening-selected candidate structures.
The package does not claim that all 50 candidates have completed
external DFT spectral validation.

Missing optional audit reports:
{", ".join(missing_reports) if missing_reports else "None"}
"""

provenance = {
    "created_at": datetime.now().astimezone().isoformat(),
    "source_shortlist": str(FINAL_CSV.resolve()),
    "source_export_directory": str(EXPORT_SOURCE_DIR.resolve()),
    "source_manifest": str(SOURCE_MANIFEST.resolve()),
    "candidate_count": EXPECTED_COUNT,
    "SLME_comparison_tolerance_pp": SLME_TOL_PP,
    "included_audit_reports": [str(path.resolve()) for path in report_files],
    "missing_optional_audit_reports": missing_reports,
    "verification_scope": [
        "Shortlist selection flags",
        "Manifest IDs, ranks, formulas, and SLME",
        "Per-material JSON IDs, ranks, SLME, and structure-field presence",
        "CIF/JSON existence and counts",
        "ZIP integrity and archived material IDs",
    ],
}


# ------------------------------------------------------------
# 6. Create a new ZIP at a short Windows path
# ------------------------------------------------------------
SEND_DIR = Path(r"D:\TB3\DFT_exports")
SEND_DIR.mkdir(parents=True, exist_ok=True)

run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
DB1_PACKAGE_ZIP = SEND_DIR / f"DB1_Top50_{run_stamp}.zip"
ZIP_PATH = DB1_PACKAGE_ZIP

with zipfile.ZipFile(
    ZIP_PATH,
    mode="x",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    archive.writestr("README.txt", README_TEXT)
    archive.writestr(
        "package_provenance.json",
        json.dumps(provenance, indent=2, ensure_ascii=False),
    )
    archive.writestr(
        "package_manifest.csv",
        portable_manifest.to_csv(index=False).encode("utf-8-sig"),
    )
    archive.writestr(
        "material_ids.txt",
        "\n".join(final_df["material_id"]) + "\n",
    )

    archive.write(
        FINAL_CSV,
        arcname=f"reports/{FINAL_CSV.name}",
    )

    for path in report_files:
        archive.write(path, arcname=f"reports/{path.name}")

    for path, member_name in structure_members:
        archive.write(path, arcname=member_name)


# ------------------------------------------------------------
# 7. Verify the archive
# ------------------------------------------------------------
with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    corrupt_member = archive.testzip()
    if corrupt_member is not None:
        raise RuntimeError(f"ZIP integrity failure: {corrupt_member}")

    names = archive.namelist()
    if len(names) != len(set(names)):
        raise RuntimeError("The ZIP contains duplicated member names.")

    cif_members = [
        name for name in names
        if name.startswith("CIF/") and name.endswith(".cif")
    ]
    json_members = [
        name for name in names
        if name.startswith("JSON/") and name.endswith(".json")
    ]

    if (
        len(cif_members) != EXPECTED_COUNT
        or len(json_members) != EXPECTED_COUNT
    ):
        raise RuntimeError("ZIP CIF/JSON counts do not match the shortlist.")

    archived_ids = [
        str(json.loads(
            archive.read(name).decode("utf-8")
        )["material_id"]).strip()
        for name in json_members
    ]

    if (
        len(set(archived_ids)) != EXPECTED_COUNT
        or set(archived_ids) != set(final_df["material_id"])
    ):
        raise RuntimeError("Archived material IDs do not match the shortlist.")


# ------------------------------------------------------------
# 8. Report
# ------------------------------------------------------------
print("\nDATABASE 1 TOP-50 PACKAGE — PASS")
print("Candidates:", EXPECTED_COUNT)
print("CIF files:", len(cif_members))
print("Per-material JSON files:", len(json_members))
print("ZIP integrity and material-ID checks: PASS")
print("ZIP package:", DB1_PACKAGE_ZIP)
print("Missing optional audit reports:", missing_reports or "None")
print("Source files and existing packages were not modified.")
print("No package was sent automatically.")

display(
    portable_manifest[[
        "DFT_shortlist_rank",
        "SLME_rank",
        "material_id",
        "formula_pretty",
        "predicted_SLME_percent",
        "export_status",
    ]]
)

Source shortlist: D:\TB3\processed\paired_training\paper_outputs\dft_validation_shortlist\discovery_first_zintl_SLME\zintl_top50_structure_new_FINAL_SLME.csv
Source package: D:\TB3\processed\paired_training\paper_outputs\dft_validation_shortlist\discovery_first_zintl_SLME\SEND_TO_DFT_DB1_FINAL_Top50_SLME
Shortlist / manifest / JSON validation: PASS

DATABASE 1 TOP-50 PACKAGE — PASS
Candidates: 50
CIF files: 50
Per-material JSON files: 50
ZIP integrity and material-ID checks: PASS
ZIP package: D:\TB3\DFT_exports\DB1_Top50_20260922_204506_357338.zip
Missing optional audit reports: ['zintl_structure_matcher_matches.csv', 'zintl_structure_matcher_errors.csv']
Source files and existing packages were not modified.
No package was sent automatically.


,DFT_shortlist_rank,SLME_rank,material_id,formula_pretty,predicted_SLME_percent,export_status
0,1,1,mp-btzq,NaP5,33.753417,OK
1,2,2,mp-bevqy,Rb2Ag(AsSe2)3,33.727537,OK
2,3,3,mp-cqyqi,KIn(PSe3)2,33.690766,OK
3,4,4,mp-bsnx,Ba2B4Se13,33.674518,OK
4,5,5,mp-cqcrk,KAgPSe3,33.668234,OK
5,6,6,mp-cqpjs,Rb2As2Pd,33.667991,OK
6,7,7,mp-crclc,Ca11GaSb9,33.666295,OK
7,8,8,mp-bkski,CsAs2Se3,33.661544,OK
8,9,9,mp-ekf,CsSbSe2,33.656419,OK
9,10,10,mp-ue,Ba3As14,33.652101,OK


In [25]:
# ============================================================
# FINAL — Detect completed DFT automatically
#         and package ONLY the remaining DB1 Top-50 candidates
#
# Assumption:
# Cell 16A has already validated the FINAL Top-50 package.
#
# Workflow:
# FINAL Top-50
# -> scan D:\DFT_TEST for valid epsilon .dat files
# -> match by material_id
# -> determine completed DFT automatically
# -> package only remaining CIF + JSON
# ============================================================

from pathlib import Path
import io
import re
import shutil
import zipfile

import numpy as np
import pandas as pd
from IPython.display import display


# ============================================================
# 1. PATHS
# ============================================================

ROOT = Path(r"D:\TB3\processed\paired_training")

DB1_DIR = (
    ROOT
    / "paper_outputs"
    / "dft_validation_shortlist"
    / "discovery_first_zintl_SLME"
)

SOURCE_DIR = (
    DB1_DIR
    / "SEND_TO_DFT_DB1_FINAL_Top50_SLME"
)

SOURCE_MANIFEST = (
    SOURCE_DIR
    / "DB1_FINAL_Top50_SLME_manifest.csv"
)

SOURCE_CIF_DIR = SOURCE_DIR / "CIF"
SOURCE_JSON_DIR = SOURCE_DIR / "JSON"

DFT_ROOT = Path(r"D:\DFT_TEST")


# ============================================================
# 2. LOAD FINAL TOP-50
# ============================================================

manifest = pd.read_csv(SOURCE_MANIFEST)

manifest["material_id"] = (
    manifest["material_id"]
    .astype(str)
    .str.strip()
    .str.lower()
)

assert len(manifest) == 50
assert manifest["material_id"].nunique() == 50

TOP50_IDS = set(
    manifest["material_id"]
)


# ============================================================
# 3. HELPERS
# ============================================================

def get_material_id(filename):
    """
    Examples:
      epsilon_rank001_mp_btzq_NaP5.dat
          -> mp-btzq

      epsilon_rank..._mp-ekf_....dat
          -> mp-ekf
    """
    text = (
        Path(filename)
        .name
        .lower()
        .replace("_", "-")
    )

    match = re.search(
        r"mp-[a-z0-9]+",
        text,
    )

    return (
        match.group(0)
        if match
        else None
    )


def valid_epsilon_dat(raw_bytes):
    """
    Require a numerical dielectric-spectrum table
    with at least:
        E,
        eps1_x, eps1_y, eps1_z,
        eps2_x, eps2_y, eps2_z
    """
    try:
        arr = np.loadtxt(
            io.BytesIO(raw_bytes)
        )

        arr = np.asarray(
            arr,
            dtype=float,
        )

        return (
            arr.ndim == 2
            and arr.shape[0] > 10
            and arr.shape[1] >= 7
            and np.isfinite(
                arr[:, :7]
            ).all()
        )

    except Exception:
        return False


# ============================================================
# 4. SCAN ALL DFT RESULTS
# ============================================================

completed_sources = {}
invalid_files = []


def register_result(
    material_id,
    source,
):
    if material_id not in TOP50_IDS:
        return

    completed_sources.setdefault(
        material_id,
        set(),
    ).add(
        str(source)
    )


# ------------------------------------------------------------
# 4A. Loose .dat files
# ------------------------------------------------------------

for dat_path in DFT_ROOT.rglob("*.dat"):

    material_id = get_material_id(
        dat_path.name
    )

    if material_id not in TOP50_IDS:
        continue

    try:
        raw = dat_path.read_bytes()

        if valid_epsilon_dat(raw):

            register_result(
                material_id,
                dat_path,
            )

        else:

            invalid_files.append(
                str(dat_path)
            )

    except Exception:

        invalid_files.append(
            str(dat_path)
        )


# ------------------------------------------------------------
# 4B. .dat files inside ZIP archives
# ------------------------------------------------------------

for zip_path in DFT_ROOT.rglob("*.zip"):

    try:

        with zipfile.ZipFile(
            zip_path,
            "r",
        ) as zf:

            for member in zf.namelist():

                if not member.lower().endswith(
                    ".dat"
                ):
                    continue

                material_id = get_material_id(
                    member
                )

                if material_id not in TOP50_IDS:
                    continue

                raw = zf.read(
                    member
                )

                if valid_epsilon_dat(raw):

                    register_result(
                        material_id,
                        f"{zip_path}::{member}",
                    )

                else:

                    invalid_files.append(
                        f"{zip_path}::{member}"
                    )

    except zipfile.BadZipFile:

        print(
            "WARNING — bad ZIP:",
            zip_path,
        )


# ============================================================
# 5. COMPLETED vs REMAINING
# ============================================================

COMPLETED_DFT_IDS = set(
    completed_sources
)

REMAINING_DFT_IDS = (
    TOP50_IDS
    - COMPLETED_DFT_IDS
)


completed = (
    manifest[
        manifest["material_id"].isin(
            COMPLETED_DFT_IDS
        )
    ]
    .copy()
    .sort_values(
        "DFT_shortlist_rank"
        if "DFT_shortlist_rank" in manifest.columns
        else "SLME_rank"
    )
    .reset_index(drop=True)
)


remaining = (
    manifest[
        manifest["material_id"].isin(
            REMAINING_DFT_IDS
        )
    ]
    .copy()
    .sort_values(
        "DFT_shortlist_rank"
        if "DFT_shortlist_rank" in manifest.columns
        else "SLME_rank"
    )
    .reset_index(drop=True)
)


remaining["DFT_run_rank"] = (
    np.arange(
        len(remaining)
    )
    + 1
)


assert (
    len(completed)
    + len(remaining)
    == 50
)


# ============================================================
# 6. ADD DFT SOURCE INFORMATION
# ============================================================

completed["DFT_source_count"] = (
    completed["material_id"]
    .map(
        lambda mid:
        len(
            completed_sources[mid]
        )
    )
)

completed["DFT_sources"] = (
    completed["material_id"]
    .map(
        lambda mid:
        " | ".join(
            sorted(
                completed_sources[mid]
            )
        )
    )
)


# ============================================================
# 7. OUTPUT
# ============================================================

N_COMPLETED = len(
    completed
)

N_REMAINING = len(
    remaining
)


AUDIT_DIR = (
    DB1_DIR
    / "DFT_completion_audit"
)

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


COMPLETED_CSV = (
    AUDIT_DIR
    / "DB1_Top50_completed_DFT_detected.csv"
)

REMAINING_CSV = (
    AUDIT_DIR
    / "DB1_Top50_remaining_DFT_candidates.csv"
)


completed.to_csv(
    COMPLETED_CSV,
    index=False,
    encoding="utf-8-sig",
)

remaining.to_csv(
    REMAINING_CSV,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 8. CREATE PACKAGE ONLY IF SOMETHING REMAINS
# ============================================================

if N_REMAINING > 0:

    OUT_DIR = (
        DB1_DIR
        / f"SEND_TO_DFT_DB1_FINAL_{N_REMAINING}_REMAINING"
    )

    OUT_CIF_DIR = (
        OUT_DIR
        / "CIF"
    )

    OUT_JSON_DIR = (
        OUT_DIR
        / "JSON"
    )


    if OUT_DIR.exists():

        shutil.rmtree(
            OUT_DIR
        )


    OUT_CIF_DIR.mkdir(
        parents=True
    )

    OUT_JSON_DIR.mkdir(
        parents=True
    )


    # --------------------------------------------------------
    # Copy remaining structures
    # --------------------------------------------------------

    for _, row in remaining.iterrows():

        src_cif = (
            SOURCE_CIF_DIR
            / str(
                row["CIF_file"]
            )
        )

        src_json = (
            SOURCE_JSON_DIR
            / str(
                row["JSON_file"]
            )
        )


        if not src_cif.is_file():

            raise FileNotFoundError(
                src_cif
            )


        if not src_json.is_file():

            raise FileNotFoundError(
                src_json
            )


        shutil.copy2(
            src_cif,
            OUT_CIF_DIR
            / src_cif.name,
        )

        shutil.copy2(
            src_json,
            OUT_JSON_DIR
            / src_json.name,
        )


    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------

    remaining.to_csv(
        OUT_DIR
        / "remaining_DFT_manifest.csv",
        index=False,
        encoding="utf-8-sig",
    )


    completed.to_csv(
        OUT_DIR
        / "already_completed_DFT.csv",
        index=False,
        encoding="utf-8-sig",
    )


    (
        OUT_DIR
        / "remaining_material_ids.txt"
    ).write_text(
        "\n".join(
            remaining[
                "material_id"
            ]
        )
        + "\n",
        encoding="utf-8",
    )


    # --------------------------------------------------------
    # README
    # --------------------------------------------------------

    (
        OUT_DIR
        / "README.txt"
    ).write_text(
        f"""FINAL DB1 REMAINING DFT PACKAGE

Original FINAL shortlist : 50
Completed DFT detected    : {N_COMPLETED}
Remaining for DFT         : {N_REMAINING}

Completed status was determined automatically
from valid dielectric-spectrum .dat files under:

{DFT_ROOT}

No material ID was manually hard-coded as completed.

Original DFT_shortlist_rank / SLME_rank are preserved.
DFT_run_rank = 1..{N_REMAINING} is only the execution
order for the remaining calculations.
""",
        encoding="utf-8",
    )


    # --------------------------------------------------------
    # Sanity checks
    # --------------------------------------------------------

    cifs = list(
        OUT_CIF_DIR.glob(
            "*.cif"
        )
    )

    jsons = list(
        OUT_JSON_DIR.glob(
            "*.json"
        )
    )


    assert (
        len(cifs)
        == N_REMAINING
    )

    assert (
        len(jsons)
        == N_REMAINING
    )


    # --------------------------------------------------------
    # ZIP
    # --------------------------------------------------------

    ZIP_BASE = (
        DB1_DIR
        / f"SEND_TO_DFT_DB1_FINAL_{N_REMAINING}_REMAINING"
    )


    ZIP_PATH = Path(
        shutil.make_archive(
            str(ZIP_BASE),
            "zip",
            root_dir=str(
                OUT_DIR
            ),
        )
    )

else:

    OUT_DIR = None
    ZIP_PATH = None


# ============================================================
# 9. FINAL REPORT
# ============================================================

print()
print("=" * 90)
print("DB1 TOP-50 DFT COMPLETION AUDIT")
print("=" * 90)

print(
    "Original Top-50     :",
    50,
)

print(
    "Completed DFT found :",
    N_COMPLETED,
)

print(
    "Remaining for DFT   :",
    N_REMAINING,
)

print(
    "Invalid .dat ignored:",
    len(
        invalid_files
    ),
)


print("\nCompleted DFT ranks:")

rank_col = (
    "DFT_shortlist_rank"
    if "DFT_shortlist_rank"
    in completed.columns
    else "SLME_rank"
)

print(
    completed[
        rank_col
    ]
    .tolist()
)


print("\nRemaining ranks:")

print(
    remaining[
        rank_col
    ]
    .tolist()
)


print(
    "\nRemaining material IDs:"
)

for _, row in remaining.iterrows():

    print(
        f"Rank "
        f"{row[rank_col]:>2} | "
        f"{row['material_id']} | "
        f"{row.get('formula_pretty', '')}"
    )


print(
    "\nCompleted audit CSV:"
)

print(
    COMPLETED_CSV
)


print(
    "\nRemaining audit CSV:"
)

print(
    REMAINING_CSV
)


if ZIP_PATH is not None:

    print(
        "\nZIP TO SEND:"
    )

    print(
        ZIP_PATH
    )

else:

    print(
        "\nAll 50 candidates already have valid DFT results."
    )


# ============================================================
# 10. TABLE PREVIEW
# ============================================================

preview_cols = [
    "DFT_run_rank",
    "DFT_shortlist_rank",
    "SLME_rank",
    "material_id",
    "formula_pretty",
    "predicted_SLME_percent",
]

preview_cols = [
    c
    for c in preview_cols
    if c in remaining.columns
]

display(
    remaining[
        preview_cols
    ]
)


DB1 TOP-50 DFT COMPLETION AUDIT
Original Top-50     : 50
Completed DFT found : 39
Remaining for DFT   : 11
Invalid .dat ignored: 0

Completed DFT ranks:
[1, 2, 3, 4, 6, 7, 8, 10, 11, 13, 14, 15, 17, 19, 20, 21, 22, 23, 24, 25, 26, 28, 29, 30, 31, 32, 35, 36, 37, 38, 39, 40, 42, 43, 44, 45, 46, 49, 50]

Remaining ranks:
[5, 9, 12, 16, 18, 27, 33, 34, 41, 47, 48]

Remaining material IDs:
Rank  5 | mp-cqcrk | KAgPSe3
Rank  9 | mp-ekf | CsSbSe2
Rank 12 | mp-bevsd | Rb2Th(PSe3)3
Rank 16 | mp-bmsnb | Li3As7
Rank 18 | mp-bkomx | Cs4Sb14S23
Rank 27 | mp-bpuc | Ba4Sb4Se11
Rank 33 | mp-bgnjt | K2Sm2As2Se9
Rank 34 | mp-cqctd | Ba2Ge(Sb2S5)2
Rank 41 | mp-bxhaf | K3Bi(AsSe2)6
Rank 47 | mp-bmceh | Rb3Ce(PSe4)2
Rank 48 | mp-bgkgb | Rb2HgPSe5

Completed audit CSV:
D:\TB3\processed\paired_training\paper_outputs\dft_validation_shortlist\discovery_first_zintl_SLME\DFT_completion_audit\DB1_Top50_completed_DFT_detected.csv

Remaining audit CSV:
D:\TB3\processed\paired_training\paper_outputs\dft_validation

,DFT_run_rank,DFT_shortlist_rank,SLME_rank,material_id,formula_pretty,predicted_SLME_percent
0,1,5,5,mp-cqcrk,KAgPSe3,33.668234
1,2,9,9,mp-ekf,CsSbSe2,33.656419
2,3,12,12,mp-bevsd,Rb2Th(PSe3)3,33.640780
3,4,16,16,mp-bmsnb,Li3As7,33.617348
4,5,18,18,mp-bkomx,Cs4Sb14S23,33.605215
5,6,27,27,mp-bpuc,Ba4Sb4Se11,33.573279
6,7,33,35,mp-bgnjt,K2Sm2As2Se9,33.510448
7,8,34,36,mp-cqctd,Ba2Ge(Sb2S5)2,33.509687
8,9,41,43,mp-bxhaf,K3Bi(AsSe2)6,33.415582
9,10,47,49,mp-bmceh,Rb3Ce(PSe4)2,33.392054
